# Current trimmed workflow — Dubai Real Estate v7

This trimmed copy keeps the current workflow only and removes the older duplicate v5 sections that appear after the latest validation/table cells.

## Use this copy for
1. no-rescrape Twitter/X MiniLM setup,
2. loading/cleaning locked Twitter/X files,
3. loading Reddit contextual files,
4. merging Twitter/X + Reddit,
5. refined domain/listing/contact filtering,
6. low-information cleanup,
7. author-aware duplicate/repost cleanup,
8. validation sample creation,
9. paper-ready table creation.

## Do not use this copy for
- live Twitter/X scraping,
- old generic v5 reruns,
- final SMDI + DLD research layer.

SMDI + DLD should be built as the next notebook or next section after annotation/validation is complete.
---


# Dubai Real Estate One-Notebook Pipeline v7  
## NO-RESCRAPE MiniLM filtering + v5 reviewer fixes

This version is for the safer workflow:

1. load the already-saved Twitter/X strict keyword raw file,
2. load the already-saved expanded/semantic candidate raw file,
3. apply MiniLM/SBERT semantic relevance filtering to the expanded candidate stream,
4. keep MiniLM outputs in `twitter_minilm_semantic_collection/` without overwriting old raw files,
5. force the later v5 pipeline to load the MiniLM-kept semantic stream for final processing.

Do **not** use this notebook to rescrape Twitter/X. It does not need `accounts_private.csv` or active twscrape accounts.


In [ ]:
# ============================================================
# 00_PRECONFIG_UNIFIED_MODE
# NO-RESCRAPE MODE: use existing Twitter raw files, then apply MiniLM
# ============================================================

# Main mode:
# - True means run the MiniLM stage before v5.
# - In this NO-RESCRAPE version, MiniLM uses already-saved raw Twitter files from Drive.
RUN_UNIFIED_TWITTER_MINILM_COLLECTION = True

# IMPORTANT: no live Twitter/X scraping in this version.
# This avoids twscrape account/rate-limit issues and keeps the old collection intact.
RUN_TWITTER_COLLECTION = False
COLLECT_STRICT_STREAM = False
COLLECT_BROAD_STREAM = False

# Still apply MiniLM to the existing broad/semantic candidate stream.
RUN_MINILM_FILTER = True
USE_EXISTING_V5_TWITTER_FILES_FOR_MINILM = True

# Do NOT overwrite old v5 inputs. v5 later loads the MiniLM-kept semantic file directly.
EXPORT_TO_V5_INPUT_PATHS = False
ALLOW_OVERWRITE_V5_INPUT_PATHS = False

print("NO-RESCRAPE MiniLM mode loaded.")
print("RUN_TWITTER_COLLECTION:", RUN_TWITTER_COLLECTION)
print("RUN_MINILM_FILTER:", RUN_MINILM_FILTER)
print("USE_EXISTING_V5_TWITTER_FILES_FOR_MINILM:", USE_EXISTING_V5_TWITTER_FILES_FOR_MINILM)
print("EXPORT_TO_V5_INPUT_PATHS:", EXPORT_TO_V5_INPUT_PATHS)


In [ ]:
# ============================================================
# 00A_INSTALL_IMPORTS_FOR_MINILM_AND_TWSCRAPE
# ============================================================

!pip -q install "twscrape[curl]" sentence-transformers pandas pyarrow tqdm nest_asyncio scipy scikit-learn

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os
import re
import json
import math
import time
import shutil
import asyncio
import warnings
from pathlib import Path
from datetime import datetime, timezone
from getpass import getpass

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import nest_asyncio
nest_asyncio.apply()

from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import (
    confusion_matrix,
    precision_recall_fscore_support,
    accuracy_score,
    cohen_kappa_score,
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 180)
pd.set_option("display.width", 250)

try:
    from IPython.display import display
except Exception:
    display = print


In [ ]:
# ============================================================
# 00B_MINILM_TWITTER_COLLECTION_CONFIG
# ============================================================

BASE = "/content/drive/MyDrive/Dubai_Real_Estate_Data"

MINILM_OUT_DIR = f"{BASE}/twitter_minilm_semantic_collection"
os.makedirs(MINILM_OUT_DIR, exist_ok=True)

# Existing v5 expected Twitter raw input paths
V5_REGEX_RAW_PATH = f"{BASE}/raw_data_files/master_raw.parquet"
V5_SEMANTIC_RAW_PATH = f"{BASE}/raw_data_files_semantic_filtering_x/master_raw_semantic_run.parquet"

# Private twscrape account files. Do not upload/share these.
# Use the same DB name as the v5 section so the MiniLM collector does not create a separate empty DB.
TWSCRAPE_DIR = f"{BASE}/twscrape"
TWS_DB_PATH = f"{TWSCRAPE_DIR}/accounts.db"
TWS_ACCOUNT_CSV = f"{TWSCRAPE_DIR}/accounts_private.csv"
os.makedirs(TWSCRAPE_DIR, exist_ok=True)

# Collection dates. Twitter/X 'until' is usually exclusive.
START_DATE = "2026-01-01"
END_DATE = "2026-05-01"   # include Jan-Apr if until is exclusive

# Collection limits. Start moderate to avoid rate-limit pain.
MAX_TWEETS_PER_QUERY_STRICT = 1200
MAX_TWEETS_PER_QUERY_BROAD = 1500

# MiniLM semantic filtering
MINILM_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
MINILM_RELEVANCE_THRESHOLD = 0.50
MINILM_BATCH_SIZE = 128

# Validation
VALIDATION_CONFIDENCE_Z = 1.96
VALIDATION_MARGIN_ERROR = 0.05
VALIDATION_EXPECTED_P = 0.5
MIN_VALIDATION_SAMPLE = 400
MAX_VALIDATION_SAMPLE = 600
SEMANTIC_ONLY_SAMPLE_SIZE = 250

# Near-duplicate audit
RUN_NEAR_DUPLICATE_AUDIT = True
NEAR_DUP_THRESHOLD = 0.90

print("MINILM_OUT_DIR:", MINILM_OUT_DIR)
print("TWS_DB_PATH:", TWS_DB_PATH)
print("TWS_ACCOUNT_CSV:", TWS_ACCOUNT_CSV)
print("V5 regex path:", V5_REGEX_RAW_PATH)
print("V5 semantic path:", V5_SEMANTIC_RAW_PATH)


In [ ]:
# ============================================================
# 00D_MINILM_TWITTER_QUERY_STREAMS
# ============================================================

TWITTER_QUERY_EXCLUSION_CUES = (
    '-crypto -giveaway -nft -job -hiring '
    '-biggboss -bb19 -#biggboss19 -#bb19 '
    '-#tanyamittal -#farrhanabhatt -#gauravkhanna'
)

# Strict stream = high-precision baseline
TWITTER_STRICT_REGEX_STREAMS = {
    "macro_investment": (
        '("dubai property" OR "dubai real estate" OR "offplan dubai" OR '
        '"off-plan dubai" OR "dubai market" OR "buy property dubai" OR '
        '"buying property in dubai" OR "buy apartment dubai" OR '
        '"buy villa dubai" OR "dubai apartment" OR '
        '"dubai villa" OR "property investment dubai" OR '
        '"dubai home" OR "owning property in dubai")'
    ),
    "rental_painpoints": (
        '("dubai rent" OR "dubai rental" OR "dubai landlord" OR "dubai tenant" OR '
        '"rent increase dubai" OR "dubai rent increase" OR "eviction dubai" OR '
        '"dubai eviction" OR "service charge dubai" OR "ejari" OR '
        '"rera rent" OR "dubai lease")'
    ),
    "developer_activity": (
        '(emaar OR damac OR nakheel OR sobha OR danube OR binghatti OR azizi OR '
        'aldar OR omniyat) (dubai OR uae OR rera)'
    ),
    "neighborhood_retail": (
        '(("Dubai Marina" OR "JVC" OR "Business Bay" OR '
        '"Palm Jumeirah" OR "Dubai Hills" OR '
        '"Downtown Dubai" OR "JLT" OR '
        '"Dubai Creek Harbour" OR "Arabian Ranches" OR '
        '"Motor City" OR "Dubai Sports City" OR '
        '"Al Furjan" OR "Town Square" OR '
        '"Dubai Silicon Oasis" OR "Mirdif" OR '
        '"Discovery Gardens" OR "The Greens" OR '
        '"The Springs" OR "The Meadows" OR '
        '"DAMAC Hills" OR "Dubai South" OR '
        '"International City") '
        'AND '
        '(property OR apartment OR villa OR rent OR rental OR buying OR investment))'
    ),
}

# Broad candidate stream = high-recall candidates for MiniLM filtering.
# Still constrained to Dubai/UAE to avoid exploding noise.
TWITTER_BROAD_CANDIDATE_STREAMS = {
    "rental_broad": (
        '(rent OR rental OR tenant OR landlord OR lease OR renewal OR eviction OR '
        '"rent increase" OR "rental dispute" OR "security deposit" OR ejari OR "service charge") '
        'AND (dubai OR uae)'
    ),
    "investment_broad": (
        '("real estate" OR property OR apartment OR villa OR offplan OR "off-plan" OR '
        'mortgage OR investor OR investment OR yield OR roi OR handover OR "payment plan" OR resale) '
        'AND (dubai OR uae)'
    ),
    "developer_broad": (
        '(emaar OR damac OR nakheel OR sobha OR danube OR binghatti OR azizi OR aldar OR omniyat OR '
        'developer OR handover OR launch OR "project launch" OR "payment plan") '
        'AND (dubai OR uae OR rera)'
    ),
    "community_broad_core": (
        '(("Dubai Marina" OR JVC OR "Jumeirah Village Circle" OR "Business Bay" OR '
        '"Downtown Dubai" OR "Palm Jumeirah" OR JLT OR "Dubai Hills" OR "Dubai Creek Harbour") '
        'AND (rent OR property OR apartment OR villa OR buying OR investment OR landlord OR tenant OR "service charge"))'
    ),
    "community_broad_emerging": (
        '(("Dubai South" OR "Dubai Silicon Oasis" OR "Motor City" OR "Al Furjan" OR '
        '"Town Square" OR "Damac Hills" OR Mirdif OR "International City" OR "Discovery Gardens") '
        'AND (rent OR property OR apartment OR villa OR buying OR investment OR landlord OR tenant OR "service charge"))'
    ),
    "market_stress_broad": (
        '(bubble OR crash OR overpriced OR unaffordable OR "rent hike" OR "rent hikes" OR '
        '"market slowdown" OR "property prices" OR "housing prices") '
        'AND (dubai OR uae)'
    ),
}

def add_date_and_filters(query: str) -> str:
    return f'({query}) lang:en since:{START_DATE} until:{END_DATE} {TWITTER_QUERY_EXCLUSION_CUES}'.strip()

print("Strict queries:")
for k, q in TWITTER_STRICT_REGEX_STREAMS.items():
    print("\n", k, "=>", add_date_and_filters(q))

print("\nBroad candidate queries:")
for k, q in TWITTER_BROAD_CANDIDATE_STREAMS.items():
    print("\n", k, "=>", add_date_and_filters(q))


In [ ]:
# ============================================================
# 00E_TWSCRAPE_ACCOUNT_SETUP_AND_DIAGNOSTICS
# ============================================================
# This cell makes the MiniLM collection use the private account file you saved in:
#   <BASE>/twscrape/accounts_private.csv
#
# IMPORTANT:
# - Your CSV can contain a `cookies` column like: auth_token=...; ct0=...
# - Or separate `auth_token` and `ct0` columns.
# - Secrets are never printed.
#
# This version imports cookie accounts through the twscrape CLI instead of the
# Python pool method. That avoids the `OperationalError: no such table: accounts`
# issue that can happen when a half-created SQLite DB is present.

from twscrape import API, gather
import subprocess
import shlex
import sqlite3
import json


RESET_TWSCRAPE_DB_IF_BROKEN = True  # safe when the DB has no accounts / no accounts table


def _mask_secret_value(x, keep=4):
    x = "" if pd.isna(x) else str(x)
    if not x:
        return ""
    return x[:keep] + "..." + x[-keep:] if len(x) > keep * 2 else "***"


def _account_name_from_row(row, idx):
    for col in ["account_name", "label", "username", "user", "login"]:
        if col in row and pd.notna(row.get(col)) and str(row.get(col)).strip():
            return str(row.get(col)).strip()
    return f"account_{idx+1}"


def _cookie_from_row(row):
    """Build a cookie string from one row of accounts_private.csv without printing secrets."""
    # Full cookie header/string already provided.
    if "cookies" in row and pd.notna(row.get("cookies")) and str(row.get("cookies")).strip():
        return str(row.get("cookies")).strip()

    # Common safer CSV layout: auth_token and ct0 in separate columns.
    auth_token = row.get("auth_token", "")
    ct0 = row.get("ct0", "")
    if pd.notna(auth_token) and pd.notna(ct0) and str(auth_token).strip() and str(ct0).strip():
        return f"auth_token={str(auth_token).strip()}; ct0={str(ct0).strip()}"

    return ""


def _normalise_account_columns(df):
    """Normalize CSV/Excel headers so Auth Token, auth_token, etc. all work."""
    df = df.copy()
    df.columns = [
        str(c).strip().lower().replace(" ", "_").replace("-", "_")
        for c in df.columns
    ]
    return df


def read_accounts_private_table(path):
    """Read accounts_private.csv even if Drive/Excel saved it with a non-UTF8 encoding."""
    with open(path, "rb") as f:
        raw = f.read(4096)

    if raw.startswith(b"SQLite format 3"):
        raise ValueError(
            "accounts_private.csv appears to be a SQLite DB, not a CSV. "
            "Make sure TWS_ACCOUNT_CSV points to accounts_private.csv, and TWS_DB_PATH points to accounts.db."
        )

    # Excel .xlsx files are zip files and start with PK. Sometimes users rename xlsx -> csv.
    if raw.startswith(b"PK"):
        print("accounts_private.csv looks like an Excel .xlsx file, so reading it with read_excel().")
        df = pd.read_excel(path)
        return _normalise_account_columns(df)

    encodings_to_try = ["utf-8-sig", "utf-8", "utf-16", "utf-16le", "utf-16be", "cp1252", "latin1"]
    last_error = None
    for enc in encodings_to_try:
        try:
            # sep=None lets pandas sniff comma/tab/semicolon delimiters.
            df = pd.read_csv(path, encoding=enc, sep=None, engine="python")
            df = _normalise_account_columns(df)
            if len(df.columns) > 0:
                print(f"Read accounts_private.csv using encoding={enc}. Columns:", list(df.columns))
                return df
        except Exception as e:
            last_error = e

    raise RuntimeError(f"Could not decode/read the account file using common encodings. Last error: {repr(last_error)}")


def _db_has_accounts_table(db_path):
    if not os.path.exists(db_path):
        return False
    try:
        con = sqlite3.connect(db_path)
        cur = con.cursor()
        cur.execute("SELECT name FROM sqlite_master WHERE type='table' AND name='accounts'")
        ok = cur.fetchone() is not None
        con.close()
        return ok
    except Exception:
        return False


def _run_twscrape_cli(args, secret_values=None, check=False):
    """Run twscrape CLI while hiding cookie/token values from output."""
    secret_values = [s for s in (secret_values or []) if s]
    cmd = ["twscrape", "--db", TWS_DB_PATH] + list(args)
    result = subprocess.run(cmd, capture_output=True, text=True)
    out = (result.stdout or "") + (result.stderr or "")
    for secret in secret_values:
        out = out.replace(secret, "***COOKIE_HIDDEN***")
    if out.strip():
        print(out.strip())
    if check and result.returncode != 0:
        raise RuntimeError(f"twscrape command failed with code {result.returncode}: twscrape --db <DB> {' '.join(args[:2])} ...")
    return result


def _format_accounts_status(accounts):
    rows = []
    for a in accounts or []:
        if isinstance(a, dict):
            get = a.get
        else:
            get = lambda k, default=None: getattr(a, k, default)
        rows.append({
            "username": get("username", get("login", "")),
            "logged_in": get("logged_in", get("loggedIn", None)),
            "active": get("active", None),
            "error_msg": get("error_msg", get("errorMsg", None)),
        })
    return pd.DataFrame(rows)


async def get_twscrape_accounts_df():
    api = API(TWS_DB_PATH)
    try:
        accounts = await api.pool.get_all()
        return _format_accounts_status(accounts)
    except Exception as e:
        print("Could not read accounts through Python API:", repr(e))
        return pd.DataFrame()


def _has_active_logged_in_account(accounts_df):
    if accounts_df is None or len(accounts_df) == 0:
        return False
    logged = accounts_df.get("logged_in", pd.Series(False, index=accounts_df.index)).astype(str).str.lower().isin(["true", "1", "yes"])
    active = accounts_df.get("active", pd.Series(False, index=accounts_df.index)).astype(str).str.lower().isin(["true", "1", "yes"])
    return bool((active | (logged & active)).any())


async def import_accounts_private_csv_if_needed(force=False):
    """Import accounts_private.csv into TWS_DB_PATH if the DB has no active accounts."""
    print("Using twscrape DB:", TWS_DB_PATH)
    print("Looking for account CSV:", TWS_ACCOUNT_CSV)

    # If a broken zero-table DB was created earlier, remove it before CLI import.
    if RESET_TWSCRAPE_DB_IF_BROKEN and os.path.exists(TWS_DB_PATH) and not _db_has_accounts_table(TWS_DB_PATH):
        os.remove(TWS_DB_PATH)
        print("Deleted broken/empty twscrape DB because it had no accounts table.")

    # Try to show current status. If empty, we will import from CSV.
    before = await get_twscrape_accounts_df()
    if len(before):
        print("Existing twscrape accounts in DB:")
        display(before)
    else:
        print("No accounts currently visible in this DB.")

    if (not force) and _has_active_logged_in_account(before):
        print("At least one active twscrape account is already available. Skipping CSV import.")
        return before

    if not os.path.exists(TWS_ACCOUNT_CSV):
        raise FileNotFoundError(
            "No active accounts found, and accounts_private.csv was not found at: " + TWS_ACCOUNT_CSV
        )

    accounts_csv = read_accounts_private_table(TWS_ACCOUNT_CSV)
    print("accounts_private.csv found. Rows:", len(accounts_csv))
    if len(accounts_csv) == 0:
        raise ValueError("accounts_private.csv exists, but it has 0 rows.")

    cookie_rows = 0
    credential_rows = 0
    failed_rows = 0

    for idx, row in accounts_csv.iterrows():
        row = row.to_dict()
        account_name = _account_name_from_row(row, idx)
        cookie_str = _cookie_from_row(row)

        if cookie_str:
            # PyPI/current twscrape supports: twscrape add_cookie ACCOUNT "auth_token=...; ct0=..."
            # Cookie accounts with ct0 should become active immediately.
            result = _run_twscrape_cli(["add_cookie", account_name, cookie_str], secret_values=[cookie_str], check=False)
            if result.returncode == 0:
                cookie_rows += 1
                print(f"Imported cookie account via CLI: {account_name}")
            else:
                failed_rows += 1
                print(f"Cookie import failed for {account_name}. See sanitized CLI output above.")
        else:
            required = ["username", "password", "email"]
            missing = [c for c in required if c not in row or pd.isna(row.get(c)) or not str(row.get(c)).strip()]
            if missing:
                failed_rows += 1
                print(f"Skipped row {idx}: missing cookie fields and missing credential columns {missing}")
                continue

            # Credential rows require login after adding. Cookie rows are preferred.
            line = f'{row.get("username")}:{row.get("password")}:{row.get("email")}:{row.get("email_password", "")}'
            temp_accounts_file = os.path.join(os.path.dirname(TWS_DB_PATH), "_tmp_twscrape_accounts_to_import.txt")
            with open(temp_accounts_file, "a", encoding="utf-8") as f:
                f.write(line + "\n")
            credential_rows += 1

    if credential_rows > 0:
        temp_accounts_file = os.path.join(os.path.dirname(TWS_DB_PATH), "_tmp_twscrape_accounts_to_import.txt")
        print(f"Credential rows detected: {credential_rows}. Importing credentials through CLI.")
        _run_twscrape_cli(["add_accounts", temp_accounts_file, "username:password:email:email_password"], check=False)
        print("Credential accounts still need login. Run this manually if prompted for verification codes:")
        print(f'!twscrape --db "{TWS_DB_PATH}" login_accounts --manual')

    print(f"Import summary: {cookie_rows} cookie rows imported/attempted, {credential_rows} credential rows prepared, {failed_rows} failed/skipped rows.")

    print("CLI account status:")
    _run_twscrape_cli(["accounts"], check=False)

    after = await get_twscrape_accounts_df()
    print("twscrape accounts after import:")
    if len(after):
        display(after)
    else:
        print("Still no accounts visible through Python API.")

    if not _has_active_logged_in_account(after):
        raise RuntimeError(
            "twscrape still has no active accounts. For cookie rows, refresh auth_token and ct0 from x.com and rerun this cell. "
            "For credential rows, run: "
            f'!twscrape --db "{TWS_DB_PATH}" login_accounts --manual'
        )

    return after


# Auto-run before collection, so scraping does not silently use an empty DB.
if RUN_TWITTER_COLLECTION:
    twscrape_accounts_status = await import_accounts_private_csv_if_needed(force=False)
else:
    print("RUN_TWITTER_COLLECTION=False; skipping twscrape account setup.")

# Useful manual diagnostics:
# await import_accounts_private_csv_if_needed(force=True)
# !twscrape --db "$TWS_DB_PATH" accounts
# !twscrape --db "$TWS_DB_PATH" stats


In [ ]:
# ============================================================
# 00F_TWEET_SERIALIZATION_COLLECTION_FUNCTIONS
# ============================================================

def _safe_get(obj, attr, default=None):
    try:
        return getattr(obj, attr, default)
    except Exception:
        return default

def tweet_to_row(tweet, query_used, retrieval_stream, retrieval_substream):
    user = _safe_get(tweet, "user", None)
    raw_text = (
        _safe_get(tweet, "rawContent", None)
        or _safe_get(tweet, "content", None)
        or _safe_get(tweet, "text", None)
        or ""
    )
    return {
        "id": str(_safe_get(tweet, "id", "")),
        "date": _safe_get(tweet, "date", None),
        "text": raw_text,
        "raw_text": raw_text,
        "url": _safe_get(tweet, "url", None),
        "lang": _safe_get(tweet, "lang", None),
        "likeCount": _safe_get(tweet, "likeCount", np.nan),
        "retweetCount": _safe_get(tweet, "retweetCount", np.nan),
        "replyCount": _safe_get(tweet, "replyCount", np.nan),
        "quoteCount": _safe_get(tweet, "quoteCount", np.nan),
        "viewCount": _safe_get(tweet, "viewCount", np.nan),
        "is_retweet": bool(_safe_get(tweet, "retweetedTweet", None)) if _safe_get(tweet, "retweetedTweet", None) is not None else False,
        "is_reply": _safe_get(tweet, "inReplyToTweetId", None) is not None,
        "author_id": str(_safe_get(user, "id", "")) if user is not None else None,
        "username": _safe_get(user, "username", None) if user is not None else None,
        "displayname": _safe_get(user, "displayname", None) if user is not None else None,
        "query_used": query_used,
        "retrieval_stream": retrieval_stream,
        "retrieval_substream": retrieval_substream,
        "collection_method": "twscrape_search",
        "collected_at_utc": datetime.now(timezone.utc).isoformat(),
    }

async def collect_one_query(api, query, limit, retrieval_stream, retrieval_substream, sleep_after=1.0):
    rows = []
    full_query = add_date_and_filters(query)
    print(f"\nCollecting [{retrieval_stream} / {retrieval_substream}] limit={limit}")
    print(full_query)
    try:
        async for tweet in api.search(full_query, limit=limit):
            rows.append(tweet_to_row(tweet, full_query, retrieval_stream, retrieval_substream))
    except Exception as e:
        print("ERROR during query:", retrieval_substream, repr(e))
    if sleep_after:
        time.sleep(sleep_after)
    df = pd.DataFrame(rows)
    print("Collected rows:", len(df))
    return df

async def collect_streams(stream_dict, stream_name, limit_per_query):
    api = API(TWS_DB_PATH)
    dfs = []
    for substream, query in stream_dict.items():
        df_q = await collect_one_query(api, query, limit_per_query, stream_name, substream)
        if len(df_q):
            dfs.append(df_q)
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

def normalize_text_key(s):
    if pd.isna(s):
        return ""
    s = str(s).lower()
    s = re.sub(r"http\S+|www\.\S+", " ", s)
    s = re.sub(r"[@#]", " ", s)
    s = re.sub(r"[^a-z0-9\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def clean_basic_twitter_df(df):
    if df is None or len(df) == 0:
        return pd.DataFrame()
    df = df.copy()
    df["id"] = df["id"].astype(str)
    df["date"] = pd.to_datetime(df["date"], errors="coerce", utc=True)
    df["text"] = df["text"].fillna("").astype(str)
    df["raw_text"] = df.get("raw_text", df["text"]).fillna("").astype(str)
    df["_clean_text_key"] = df["text"].apply(normalize_text_key)
    before = len(df)
    df = df[df["_clean_text_key"].str.len() > 0].copy()
    df = df.drop_duplicates(subset=["id"], keep="first")
    for c in ["likeCount", "retweetCount", "replyCount", "quoteCount"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)
    df["_interaction_for_dedup"] = (
        df.get("likeCount", 0) + 3*df.get("retweetCount", 0) + 2*df.get("replyCount", 0) + 2*df.get("quoteCount", 0)
    )
    df = df.sort_values("_interaction_for_dedup", ascending=False)
    df = df.drop_duplicates(subset=["_clean_text_key"], keep="first")
    df = df.sort_values("date")
    print(f"Clean basic: {before} -> {len(df)} rows")
    return df


In [ ]:
# ============================================================
# 00G_LOAD_EXISTING_TWITTER_RAW_FILES_NO_RESCRAPE
# ============================================================
# This notebook version does NOT collect Twitter again.
# It loads:
#   strict stream  -> raw_data_files/master_raw.parquet
#   broad/semantic -> raw_data_files_semantic_filtering_x/master_raw_semantic_run.parquet
# Then MiniLM is applied to the broad/semantic candidate stream.

STRICT_RAW_OUT = f"{MINILM_OUT_DIR}/twitter_strict_raw.parquet"
BROAD_RAW_OUT = f"{MINILM_OUT_DIR}/twitter_broad_candidate_raw.parquet"

def _first_existing_file(paths, label):
    for p in paths:
        if os.path.exists(str(p)):
            return str(p)
    raise FileNotFoundError(f"Could not find {label}. Checked: {[str(p) for p in paths]}")

def _ensure_twitter_text_column(df, label):
    df = df.copy()
    text_candidates = ["text", "rawContent", "raw_text", "content", "tweet_text"]
    text_col = next((c for c in text_candidates if c in df.columns), None)
    if text_col is None:
        raise ValueError(f"{label} has no text column. Columns: {df.columns.tolist()}")
    if text_col != "text":
        df["text"] = df[text_col]
    return df

if RUN_TWITTER_COLLECTION:
    raise RuntimeError(
        "This is the NO-RESCRAPE notebook. RUN_TWITTER_COLLECTION should be False. "
        "Use the twscrape version only if you intentionally want new collection."
    )

strict_existing_path = _first_existing_file([V5_REGEX_RAW_PATH], "existing strict Twitter raw parquet")
broad_existing_path = _first_existing_file(
    [
        V5_SEMANTIC_RAW_PATH,
        f"{BASE}/raw_data_files_semantic_filtering_x/master_raw.parquet",
    ],
    "existing broad/semantic Twitter raw parquet",
)

strict_raw = pd.read_parquet(strict_existing_path)
broad_raw = pd.read_parquet(broad_existing_path)

strict_raw = _ensure_twitter_text_column(strict_raw, "strict_raw")
broad_raw = _ensure_twitter_text_column(broad_raw, "broad_raw")

# Basic provenance for paper/audit clarity.
strict_raw["minilm_input_source_path"] = strict_existing_path
broad_raw["minilm_input_source_path"] = broad_existing_path
if "retrieval_stream" not in strict_raw.columns:
    strict_raw["retrieval_stream"] = "strict_regex_existing"
if "retrieval_stream" not in broad_raw.columns:
    broad_raw["retrieval_stream"] = "expanded_semantic_existing"
if "retrieval_substream" not in strict_raw.columns:
    strict_raw["retrieval_substream"] = strict_raw.get("retrieval_stream", "strict_regex_existing")
if "retrieval_substream" not in broad_raw.columns:
    broad_raw["retrieval_substream"] = broad_raw.get("retrieval_stream", "expanded_semantic_existing")

# Save copies into the MiniLM output folder for traceability without overwriting v5 inputs.
strict_raw.to_parquet(STRICT_RAW_OUT, index=False)
strict_raw.to_csv(STRICT_RAW_OUT.replace(".parquet", ".csv"), index=False)
broad_raw.to_parquet(BROAD_RAW_OUT, index=False)
broad_raw.to_csv(BROAD_RAW_OUT.replace(".parquet", ".csv"), index=False)

print("Loaded existing strict raw:", strict_existing_path)
print("Loaded existing broad/semantic raw:", broad_existing_path)
print("\nStrict raw shape:", strict_raw.shape)
print("Broad raw shape:", broad_raw.shape)
print("Saved trace copies to:", MINILM_OUT_DIR)

if len(strict_raw):
    preview_cols = [c for c in ["date", "username", "text", "retrieval_substream"] if c in strict_raw.columns]
    display(strict_raw[preview_cols].head())
if len(broad_raw):
    preview_cols = [c for c in ["date", "username", "text", "retrieval_substream"] if c in broad_raw.columns]
    display(broad_raw[preview_cols].head())


In [ ]:
# ============================================================
# 00H_MINILM_SBERT_SEMANTIC_RELEVANCE_FILTER
# ============================================================

from sentence_transformers import SentenceTransformer, util

DUBAI_RE_REFERENCE_SENTENCES = [
    "Discussion about buying, selling, renting, or investing in Dubai real estate.",
    "A tweet about Dubai property prices, rent increases, landlords, tenants, leases, Ejari, or rental disputes.",
    "A tweet about Dubai off-plan property, developers, handover, payment plans, or real estate investment.",
    "A tweet about Dubai neighbourhoods such as JVC, Dubai Marina, Business Bay, Downtown Dubai, or Palm Jumeirah in relation to property or rent.",
    "A tweet about Dubai real estate market activity, sales, prices, mortgages, housing affordability, or transaction activity.",
    "Discussion about RERA, service charges, property management, rental increases, or housing regulations in Dubai.",
]

def apply_minilm_relevance_filter(df, text_col="text", threshold=MINILM_RELEVANCE_THRESHOLD, batch_size=MINILM_BATCH_SIZE):
    if df is None or len(df) == 0:
        print("No rows to score.")
        return pd.DataFrame()
    if text_col not in df.columns:
        raise ValueError(f"Missing text column: {text_col}")

    model = SentenceTransformer(MINILM_MODEL_NAME)
    reference_embeddings = model.encode(
        DUBAI_RE_REFERENCE_SENTENCES,
        convert_to_tensor=True,
        normalize_embeddings=True,
    )
    texts = df[text_col].fillna("").astype(str).tolist()
    max_scores, mean_top2_scores, best_refs, best_ref_indices = [], [], [], []

    for start in tqdm(range(0, len(texts), batch_size), desc="MiniLM scoring"):
        batch_texts = texts[start:start + batch_size]
        tweet_embeddings = model.encode(batch_texts, convert_to_tensor=True, normalize_embeddings=True)
        scores = util.cos_sim(tweet_embeddings, reference_embeddings)
        scores_np = scores.detach().cpu().numpy()
        batch_max = scores_np.max(axis=1)
        batch_best_idx = scores_np.argmax(axis=1)
        batch_top2_mean = np.sort(scores_np, axis=1)[:, -2:].mean(axis=1)
        max_scores.extend(batch_max.tolist())
        mean_top2_scores.extend(batch_top2_mean.tolist())
        best_ref_indices.extend(batch_best_idx.tolist())
        best_refs.extend([DUBAI_RE_REFERENCE_SENTENCES[i] for i in batch_best_idx])

    out = df.copy()
    out["minilm_model"] = MINILM_MODEL_NAME
    out["minilm_max_similarity"] = max_scores
    out["minilm_mean_top2_similarity"] = mean_top2_scores
    out["minilm_best_reference_index"] = best_ref_indices
    out["minilm_best_reference"] = best_refs
    out["minilm_threshold"] = threshold
    out["minilm_is_semantic_relevant"] = out["minilm_max_similarity"] >= threshold
    out["semantic_filter_method"] = "MiniLM/SBERT cosine similarity to Dubai real-estate reference sentences"
    return out

MINILM_SCORED_OUT = f"{MINILM_OUT_DIR}/twitter_broad_candidate_minilm_scored.parquet"
MINILM_KEPT_OUT = f"{MINILM_OUT_DIR}/twitter_minilm_kept_semantic_stream.parquet"
MINILM_REJECTED_OUT = f"{MINILM_OUT_DIR}/twitter_minilm_rejected.parquet"

if RUN_MINILM_FILTER:
    broad_scored = apply_minilm_relevance_filter(broad_raw, text_col="text")
    broad_scored.to_parquet(MINILM_SCORED_OUT, index=False)
    broad_scored.to_csv(MINILM_SCORED_OUT.replace(".parquet", ".csv"), index=False)

    if broad_scored is None or len(broad_scored) == 0 or "minilm_is_semantic_relevant" not in broad_scored.columns:
        print("MiniLM skipped because the broad candidate stream has 0 rows.")
        minilm_kept = pd.DataFrame(columns=list(broad_scored.columns) if broad_scored is not None else [])
        minilm_rejected = pd.DataFrame(columns=list(broad_scored.columns) if broad_scored is not None else [])
    else:
        minilm_kept = broad_scored[broad_scored["minilm_is_semantic_relevant"]].copy()
        minilm_rejected = broad_scored[~broad_scored["minilm_is_semantic_relevant"]].copy()

    if len(minilm_kept):
        minilm_kept["retrieval_stream"] = "minilm_semantic"
        minilm_kept["source_label"] = "semantic"
    if len(strict_raw):
        strict_raw["source_label"] = "regex"

    minilm_kept.to_parquet(MINILM_KEPT_OUT, index=False)
    minilm_kept.to_csv(MINILM_KEPT_OUT.replace(".parquet", ".csv"), index=False)
    minilm_rejected.to_parquet(MINILM_REJECTED_OUT, index=False)
    minilm_rejected.to_csv(MINILM_REJECTED_OUT.replace(".parquet", ".csv"), index=False)
else:
    broad_scored = pd.read_parquet(MINILM_SCORED_OUT) if os.path.exists(MINILM_SCORED_OUT) else pd.DataFrame()
    minilm_kept = pd.read_parquet(MINILM_KEPT_OUT) if os.path.exists(MINILM_KEPT_OUT) else pd.DataFrame()
    minilm_rejected = pd.read_parquet(MINILM_REJECTED_OUT) if os.path.exists(MINILM_REJECTED_OUT) else pd.DataFrame()

print("Broad scored:", broad_scored.shape)
print("MiniLM kept:", minilm_kept.shape)
print("MiniLM rejected:", minilm_rejected.shape)

if len(broad_scored):
    print("Score summary:")
    display(broad_scored["minilm_max_similarity"].describe())
    print("Kept rate:", len(minilm_kept) / len(broad_scored))
    display(broad_scored.sort_values("minilm_max_similarity", ascending=False)[["minilm_max_similarity", "retrieval_substream", "text", "minilm_best_reference"]].head(20))


In [ ]:
from pathlib import Path

BASE_PATH = Path("/content/drive/MyDrive/Dubai_Real_Estate_Data")
MINILM_OUT_DIR = BASE_PATH / "twitter_minilm_semantic_collection"

files_to_check = [
    MINILM_OUT_DIR / "twitter_broad_candidate_minilm_scored.parquet",
    MINILM_OUT_DIR / "twitter_strict_raw_flagged.parquet",
    MINILM_OUT_DIR / "twitter_minilm_kept_semantic_stream_flagged.parquet",
]

for f in files_to_check:
    print(f.name, "->", f.exists())

In [ ]:
# ============================================================
# BETTER PATCH: contrastive MiniLM filtering, no phrase blacklist
# ============================================================

from sentence_transformers import SentenceTransformer, util
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

MINILM_RELEVANCE_THRESHOLD = 0.50
MINILM_MARGIN_THRESHOLD = 0.06
MINILM_BATCH_SIZE = 128

POSITIVE_REFERENCE_SENTENCES = [
    "Discussion about buying, selling, renting, or investing in Dubai real estate.",
    "A tweet about Dubai property prices, rent increases, landlords, tenants, leases, Ejari, or rental disputes.",
    "A tweet about Dubai off-plan property, developers, handover, payment plans, or real estate investment.",
    "A tweet about Dubai neighbourhoods such as JVC, Dubai Marina, Business Bay, Downtown Dubai, or Palm Jumeirah in relation to property or rent.",
    "A tweet about Dubai real estate market activity, sales, prices, mortgages, housing affordability, or transaction activity.",
    "Discussion about RERA, service charges, property management, rental increases, or housing regulations in Dubai.",
]

NEGATIVE_REFERENCE_SENTENCES = [
    "A generic tweet about Dubai lifestyle, tourism, luxury, restaurants, hotels, airports, or vacations.",
    "A tweet about politics, war, geopolitics, missiles, airspace, military conflict, or international relations without real-estate discussion.",
    "A tweet about cryptocurrency, blockchain, token launches, Web3, NFTs, or financial trading without real-estate market discussion.",
    "A tweet about celebrities, influencers, sports, entertainment, or social media drama.",
    "A tweet using rent, property, lease, or market in a non-real-estate sense, such as rent-seeking, rental cars, politics, or general economics.",
    "A promotional advertisement unrelated to Dubai real estate discourse.",
]

model = SentenceTransformer(MINILM_MODEL_NAME)

positive_embeddings = model.encode(
    POSITIVE_REFERENCE_SENTENCES,
    convert_to_tensor=True,
    normalize_embeddings=True,
)

negative_embeddings = model.encode(
    NEGATIVE_REFERENCE_SENTENCES,
    convert_to_tensor=True,
    normalize_embeddings=True,
)

texts = broad_scored["text"].fillna("").astype(str).tolist()

pos_scores = []
neg_scores = []
best_pos_refs = []
best_neg_refs = []

for start in tqdm(range(0, len(texts), MINILM_BATCH_SIZE), desc="Contrastive MiniLM scoring"):
    batch = texts[start:start + MINILM_BATCH_SIZE]

    emb = model.encode(
        batch,
        convert_to_tensor=True,
        normalize_embeddings=True,
    )

    pos = util.cos_sim(emb, positive_embeddings).detach().cpu().numpy()
    neg = util.cos_sim(emb, negative_embeddings).detach().cpu().numpy()

    pos_max = pos.max(axis=1)
    neg_max = neg.max(axis=1)

    pos_idx = pos.argmax(axis=1)
    neg_idx = neg.argmax(axis=1)

    pos_scores.extend(pos_max.tolist())
    neg_scores.extend(neg_max.tolist())
    best_pos_refs.extend([POSITIVE_REFERENCE_SENTENCES[i] for i in pos_idx])
    best_neg_refs.extend([NEGATIVE_REFERENCE_SENTENCES[i] for i in neg_idx])

broad_scored["minilm_positive_max_similarity"] = pos_scores
broad_scored["minilm_negative_max_similarity"] = neg_scores
broad_scored["minilm_semantic_margin"] = (
    broad_scored["minilm_positive_max_similarity"]
    - broad_scored["minilm_negative_max_similarity"]
)

broad_scored["minilm_best_positive_reference"] = best_pos_refs
broad_scored["minilm_best_negative_reference"] = best_neg_refs

# Final semantic keep rule
broad_scored["minilm_is_semantic_relevant"] = (
    (broad_scored["minilm_positive_max_similarity"] >= MINILM_RELEVANCE_THRESHOLD)
    &
    (broad_scored["minilm_semantic_margin"] >= MINILM_MARGIN_THRESHOLD)
)

minilm_kept = broad_scored[broad_scored["minilm_is_semantic_relevant"]].copy()
minilm_rejected = broad_scored[~broad_scored["minilm_is_semantic_relevant"]].copy()

print("Broad scored:", len(broad_scored))
print("MiniLM kept:", len(minilm_kept))
print("MiniLM rejected:", len(minilm_rejected))
print("Kept rate:", len(minilm_kept) / len(broad_scored))

print("\nKept score summary:")
display(
    minilm_kept[
        ["minilm_positive_max_similarity", "minilm_negative_max_similarity", "minilm_semantic_margin"]
    ].describe()
)

print("\nLowest-margin kept tweets:")
display(
    minilm_kept
    .sort_values("minilm_semantic_margin", ascending=True)
    [[
        "minilm_positive_max_similarity",
        "minilm_negative_max_similarity",
        "minilm_semantic_margin",
        "text",
        "minilm_best_positive_reference",
        "minilm_best_negative_reference",
    ]]
    .head(40)
)

print("\nHighest-margin rejected tweets:")
display(
    minilm_rejected
    .sort_values("minilm_semantic_margin", ascending=False)
    [[
        "minilm_positive_max_similarity",
        "minilm_negative_max_similarity",
        "minilm_semantic_margin",
        "text",
        "minilm_best_positive_reference",
        "minilm_best_negative_reference",
    ]]
    .head(40)
)

In [ ]:
# ============================================================
# 00I_THRESHOLD_CALIBRATION_SAMPLE
# ============================================================
# Fill annotator_relevant_0_1 later to choose a defensible threshold.

CALIBRATION_SAMPLE_OUT = f"{MINILM_OUT_DIR}/twitter_minilm_threshold_calibration_sample.csv"

def make_threshold_calibration_sample(scored_df, n_per_bin=30):
    if scored_df is None or len(scored_df) == 0:
        return pd.DataFrame()
    df = scored_df.copy()
    bins = [-1, 0.20, 0.30, 0.35, 0.40, 0.42, 0.45, 0.50, 0.60, 1.00]
    labels = [f"{bins[i]}_{bins[i+1]}" for i in range(len(bins)-1)]
    df["score_bin"] = pd.cut(df["minilm_max_similarity"], bins=bins, labels=labels)
    samples = []
    for b, g in df.groupby("score_bin", observed=False):
        if len(g):
            samples.append(g.sample(n=min(n_per_bin, len(g)), random_state=42))
    out = pd.concat(samples, ignore_index=True) if samples else pd.DataFrame()
    out["annotator_relevant_0_1"] = ""
    out["annotation_notes"] = ""
    keep_cols = ["id", "date", "username", "text", "retrieval_substream", "query_used", "minilm_max_similarity", "minilm_mean_top2_similarity", "score_bin", "minilm_best_reference", "annotator_relevant_0_1", "annotation_notes"]
    keep_cols = [c for c in keep_cols if c in out.columns]
    return out[keep_cols]

calibration_sample = make_threshold_calibration_sample(broad_scored, n_per_bin=30)
calibration_sample.to_csv(CALIBRATION_SAMPLE_OUT, index=False)
print("Saved calibration sample:", CALIBRATION_SAMPLE_OUT)
display(calibration_sample.head(30))


In [ ]:
# ============================================================
# 00J_OPTIONAL_EVALUATE_THRESHOLD_AFTER_LABELS
# ============================================================
# Fill annotator_relevant_0_1 in twitter_minilm_threshold_calibration_sample.csv, then rerun.

def evaluate_thresholds_from_calibration(labelled_path=CALIBRATION_SAMPLE_OUT, thresholds=np.arange(0.25, 0.61, 0.01)):
    if not os.path.exists(labelled_path):
        print("No labelled file found:", labelled_path)
        return pd.DataFrame()
    df = pd.read_csv(labelled_path, low_memory=False)
    if "annotator_relevant_0_1" not in df.columns:
        print("Missing annotator_relevant_0_1")
        return pd.DataFrame()
    df["y_true"] = pd.to_numeric(df["annotator_relevant_0_1"], errors="coerce")
    df = df[df["y_true"].isin([0, 1])].copy()
    rows = []
    for t in thresholds:
        y_pred = (df["minilm_max_similarity"] >= t).astype(int)
        precision, recall, f1, _ = precision_recall_fscore_support(df["y_true"], y_pred, average="binary", zero_division=0)
        rows.append({"threshold": round(float(t), 3), "n_labelled": len(df), "precision": precision, "recall": recall, "f1": f1, "accuracy": accuracy_score(df["y_true"], y_pred), "kept_rate_in_sample": float(y_pred.mean())})
    out = pd.DataFrame(rows).sort_values("f1", ascending=False)
    out_path = f"{MINILM_OUT_DIR}/twitter_minilm_threshold_evaluation.csv"
    out.to_csv(out_path, index=False)
    print("Saved:", out_path)
    display(out.head(30))
    return out

threshold_eval = evaluate_thresholds_from_calibration()


In [ ]:
# ============================================================
# 00K_LISTING_SPAM_FLAGS_FOR_TWITTER_RAW_STAGE
# ============================================================
# v5 can also do this. This gives you an immediate reviewer-facing flag.

NOISE_CUES = ["crypto", "nft", "giveaway", "airdrop", "casino", "betting", "porn", "job opening", "hiring", "recruitment", "bigg boss", "biggboss"]

AGENT_LISTING_CUES = [
    "whatsapp", "call now", "dm me", "contact agent", "book viewing",
    "available now", "ready to move", "rera permit", "permit no",
    "ref no", "broker", "agent", "limited offer", "viewing",
    "for sale", "for rent", "studio available", "bedroom apartment available",
    "payment plan available", "handover soon"
]

def cue_matches(text, cues):
    t = str(text).lower()
    return [c for c in cues if c in t]

def add_quality_flags(df):
    if df is None or len(df) == 0:
        return pd.DataFrame()
    out = df.copy()
    out["noise_cues_matched"] = out["text"].apply(lambda x: "|".join(cue_matches(x, NOISE_CUES)))
    out["noise_keyword_count"] = out["noise_cues_matched"].apply(lambda x: 0 if not x else len(x.split("|")))
    out["agent_listing_cues_matched"] = out["text"].apply(lambda x: "|".join(cue_matches(x, AGENT_LISTING_CUES)))
    out["agent_listing_cue_count"] = out["agent_listing_cues_matched"].apply(lambda x: 0 if not x else len(x.split("|")))
    out["is_likely_agent_listing"] = out["agent_listing_cue_count"] >= 2
    return out

strict_flagged = add_quality_flags(strict_raw)
minilm_kept_flagged = add_quality_flags(minilm_kept)
broad_scored_flagged = add_quality_flags(broad_scored)

strict_flagged.to_parquet(f"{MINILM_OUT_DIR}/twitter_strict_raw_flagged.parquet", index=False)
minilm_kept_flagged.to_parquet(f"{MINILM_OUT_DIR}/twitter_minilm_kept_semantic_stream_flagged.parquet", index=False)
broad_scored_flagged.to_parquet(f"{MINILM_OUT_DIR}/twitter_broad_candidate_minilm_scored_flagged.parquet", index=False)

print("Strict likely listings:", strict_flagged["is_likely_agent_listing"].sum() if len(strict_flagged) else 0)
print("MiniLM kept likely listings:", minilm_kept_flagged["is_likely_agent_listing"].sum() if len(minilm_kept_flagged) else 0)


In [ ]:
# ============================================================
# PATCH: enforce Jan-Apr 2026 study window before final Twitter merge
# ============================================================

import pandas as pd
from pathlib import Path

STUDY_START = pd.Timestamp("2026-01-01", tz="UTC")
STUDY_END_EXCLUSIVE = pd.Timestamp("2026-05-01", tz="UTC")

BASE = Path("/content/drive/MyDrive/Dubai_Real_Estate_Data")
MINILM_OUT_DIR = BASE / "twitter_minilm_semantic_collection"

def enforce_study_window(df, label):
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"], errors="coerce", utc=True)

    before = len(df)
    df = df[
        (df["date"] >= STUDY_START) &
        (df["date"] < STUDY_END_EXCLUSIVE)
    ].copy()

    print(f"{label}: {before} -> {len(df)} after Jan-Apr 2026 filter")
    if len(df):
        print("Date range:", df["date"].min(), "to", df["date"].max())

    return df

# Filter all key Twitter stage dataframes if they exist
if "strict_raw" in globals():
    strict_raw = enforce_study_window(strict_raw, "strict_raw")

if "broad_raw" in globals():
    broad_raw = enforce_study_window(broad_raw, "broad_raw")

if "broad_scored" in globals():
    broad_scored = enforce_study_window(broad_scored, "broad_scored")

if "minilm_kept" in globals():
    minilm_kept = enforce_study_window(minilm_kept, "minilm_kept")

if "minilm_rejected" in globals():
    minilm_rejected = enforce_study_window(minilm_rejected, "minilm_rejected")

if "strict_flagged" in globals():
    strict_flagged = enforce_study_window(strict_flagged, "strict_flagged")

if "minilm_kept_flagged" in globals():
    minilm_kept_flagged = enforce_study_window(minilm_kept_flagged, "minilm_kept_flagged")

# Save the date-filtered semantic file that v5 should use
if "minilm_kept_flagged" in globals():
    semantic_filtered_path = MINILM_OUT_DIR / "twitter_minilm_kept_semantic_stream_flagged.parquet"
    minilm_kept_flagged.to_parquet(semantic_filtered_path, index=False)
    minilm_kept_flagged.to_csv(str(semantic_filtered_path).replace(".parquet", ".csv"), index=False)
    print("Saved filtered MiniLM semantic file:", semantic_filtered_path)

In [ ]:
# ============================================================
# 00L_MERGE_STRICT_AND_MINILM_KEPT_STREAMS
# ============================================================

MERGED_TWITTER_OUT = f"{MINILM_OUT_DIR}/twitter_merged_strict_plus_minilm.parquet"

def merge_twitter_streams(strict_df, semantic_df):
    strict_df = strict_df.copy() if strict_df is not None else pd.DataFrame()
    semantic_df = semantic_df.copy() if semantic_df is not None else pd.DataFrame()
    if len(strict_df):
        strict_df["source_label"] = "regex"
        strict_df["retrieval_streams"] = "strict_regex"
    if len(semantic_df):
        semantic_df["source_label"] = "semantic"
        semantic_df["retrieval_streams"] = "minilm_semantic"
    combined = pd.concat([strict_df, semantic_df], ignore_index=True, sort=False)
    if len(combined) == 0:
        return combined
    combined["id"] = combined["id"].astype(str)
    combined["_clean_text_key"] = combined["text"].apply(normalize_text_key)
    rows = []
    for _, g in combined.groupby(["id"], dropna=False):
        g = g.copy()
        for c in ["likeCount", "retweetCount", "replyCount", "quoteCount"]:
            if c in g.columns:
                g[c] = pd.to_numeric(g[c], errors="coerce").fillna(0)
        g["_interaction_for_dedup"] = g.get("likeCount", 0) + 3*g.get("retweetCount", 0) + 2*g.get("replyCount", 0) + 2*g.get("quoteCount", 0)
        keep = g.sort_values("_interaction_for_dedup", ascending=False).iloc[0].copy()
        labels = set(g["source_label"].dropna().astype(str).tolist())
        if labels == {"regex", "semantic"}:
            keep["source_label"] = "both"
        elif "regex" in labels:
            keep["source_label"] = "regex"
        elif "semantic" in labels:
            keep["source_label"] = "semantic"
        else:
            keep["source_label"] = "unknown"
        keep["retrieval_streams"] = "|".join(sorted(set(g["retrieval_streams"].dropna().astype(str).tolist())))
        keep["query_used_all"] = " || ".join(sorted(set(g["query_used"].dropna().astype(str).tolist())))
        rows.append(keep)
    out = pd.DataFrame(rows).sort_values("date")
    return out

twitter_merged = merge_twitter_streams(strict_flagged, minilm_kept_flagged)
twitter_merged.to_parquet(MERGED_TWITTER_OUT, index=False)
twitter_merged.to_csv(MERGED_TWITTER_OUT.replace(".parquet", ".csv"), index=False)

print("Merged Twitter shape:", twitter_merged.shape)
print("source_label distribution:")
if len(twitter_merged) and "source_label" in twitter_merged.columns:
    display(twitter_merged["source_label"].value_counts(dropna=False).reset_index(name="count"))
    display(twitter_merged[[c for c in ["date", "source_label", "retrieval_streams", "text"] if c in twitter_merged.columns]].head(20))
else:
    print("No merged Twitter rows yet.")
print("Saved:", MERGED_TWITTER_OUT)


In [ ]:
# ============================================================
# FRESH SESSION: MOUNT DRIVE + LOCK FINAL CONTRASTIVE MINILM OUTPUTS FOR V5
# ============================================================

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

from pathlib import Path
import pandas as pd

# -----------------------------
# Paths
# -----------------------------
BASE = Path("/content/drive/MyDrive/Dubai_Real_Estate_Data")
MINILM_OUT_DIR = BASE / "twitter_minilm_semantic_collection"
MINILM_OUT_DIR.mkdir(parents=True, exist_ok=True)

STUDY_START = pd.Timestamp("2026-01-01", tz="UTC")
STUDY_END_EXCLUSIVE = pd.Timestamp("2026-05-01", tz="UTC")

ORIGINAL_STRICT_RAW_PATH = BASE / "raw_data_files" / "master_raw.parquet"
FINAL_MERGED_PATH = MINILM_OUT_DIR / "twitter_merged_strict_plus_minilm.parquet"

FINAL_STRICT_FOR_V5 = MINILM_OUT_DIR / "twitter_strict_raw_date_filtered_for_v5.parquet"
FINAL_SEMANTIC_FOR_V5 = MINILM_OUT_DIR / "twitter_minilm_kept_semantic_stream_flagged.parquet"

# -----------------------------
# Helper functions
# -----------------------------
def ensure_text_col(df, label):
    df = df.copy()
    candidates = ["text", "raw_text", "rawContent", "content", "tweet_text"]
    text_col = next((c for c in candidates if c in df.columns), None)

    if text_col is None:
        raise ValueError(f"{label} has no text column. Columns: {df.columns.tolist()}")

    if text_col != "text":
        df["text"] = df[text_col]

    return df


def enforce_window(df, label):
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"], errors="coerce", utc=True)

    before = len(df)
    df = df[
        (df["date"] >= STUDY_START)
        &
        (df["date"] < STUDY_END_EXCLUSIVE)
    ].copy()

    print(f"{label}: {before} -> {len(df)}")
    if len(df):
        print("Date range:", df["date"].min(), "to", df["date"].max())

    return df


# ============================================================
# 1) Load strict regex stream from original raw file
# ============================================================

assert ORIGINAL_STRICT_RAW_PATH.exists(), f"Missing strict raw file: {ORIGINAL_STRICT_RAW_PATH}"

strict_for_v5 = pd.read_parquet(ORIGINAL_STRICT_RAW_PATH)
strict_for_v5 = ensure_text_col(strict_for_v5, "strict_for_v5")
strict_for_v5 = enforce_window(strict_for_v5, "strict_for_v5")

# Normalize strict provenance
strict_for_v5["retrieval_stream"] = "strict_regex"
if "retrieval_substream" not in strict_for_v5.columns:
    strict_for_v5["retrieval_substream"] = "strict_regex"


# ============================================================
# 2) Reconstruct final contrastive MiniLM semantic stream
#    from saved merged Twitter file
# ============================================================

assert FINAL_MERGED_PATH.exists(), f"Missing final merged file: {FINAL_MERGED_PATH}"

merged_saved = pd.read_parquet(FINAL_MERGED_PATH)
merged_saved = ensure_text_col(merged_saved, "merged_saved")
merged_saved = enforce_window(merged_saved, "merged_saved")

print("\nMerged saved shape:", merged_saved.shape)

if "source_label" in merged_saved.columns:
    print("\nMerged source labels:")
    print(merged_saved["source_label"].value_counts(dropna=False))

# Take everything that came from the MiniLM semantic side:
# this includes semantic-only + both rows.
if "retrieval_streams" in merged_saved.columns:
    semantic_mask = (
        merged_saved["retrieval_streams"]
        .fillna("")
        .astype(str)
        .str.contains("minilm_semantic", na=False)
    )
else:
    semantic_mask = merged_saved["source_label"].isin(["semantic", "both"])

semantic_for_v5 = merged_saved[semantic_mask].copy()

# Normalize semantic provenance so v5 treats this as the semantic stream
semantic_for_v5["retrieval_stream"] = "minilm_semantic"
if "retrieval_substream" not in semantic_for_v5.columns:
    semantic_for_v5["retrieval_substream"] = "contrastive_minilm"

semantic_for_v5 = ensure_text_col(semantic_for_v5, "semantic_for_v5")
semantic_for_v5 = enforce_window(semantic_for_v5, "semantic_for_v5")


# ============================================================
# 3) Save locked v5 inputs
# ============================================================

strict_for_v5.to_parquet(FINAL_STRICT_FOR_V5, index=False)
strict_for_v5.to_csv(str(FINAL_STRICT_FOR_V5).replace(".parquet", ".csv"), index=False)

semantic_for_v5.to_parquet(FINAL_SEMANTIC_FOR_V5, index=False)
semantic_for_v5.to_csv(str(FINAL_SEMANTIC_FOR_V5).replace(".parquet", ".csv"), index=False)

print("\nSaved strict v5 input:", FINAL_STRICT_FOR_V5)
print("Saved semantic v5 input:", FINAL_SEMANTIC_FOR_V5)

print("\nFinal v5 input counts:")
print("strict_for_v5:", len(strict_for_v5))
print("semantic_for_v5:", len(semantic_for_v5))

print("\nSemantic MiniLM columns:")
print([c for c in semantic_for_v5.columns if "minilm" in c.lower()])


# ============================================================
# 4) Patch v5 variables immediately
# ============================================================

BASE_PATH = BASE

V5_REGEX_RAW_PATH = str(FINAL_STRICT_FOR_V5)
V5_SEMANTIC_RAW_PATH = str(FINAL_SEMANTIC_FOR_V5)

RAW_TWITTER_REGEX_PATH = FINAL_STRICT_FOR_V5
RAW_TWITTER_SEMANTIC_PATH = FINAL_SEMANTIC_FOR_V5
RAW_TWITTER_SEMANTIC_CANDIDATES = [FINAL_SEMANTIC_FOR_V5]

RUN_TWITTER_COLLECTION = False
COLLECT_REGEX_STREAM = False
COLLECT_SEMANTIC_STREAM = False
EXPORT_TO_V5_INPUT_PATHS = False

assert FINAL_STRICT_FOR_V5.exists(), f"Missing: {FINAL_STRICT_FOR_V5}"
assert FINAL_SEMANTIC_FOR_V5.exists(), f"Missing: {FINAL_SEMANTIC_FOR_V5}"

print("\nPATCHED V5 INPUTS")
print("v5 regex input:", FINAL_STRICT_FOR_V5)
print("v5 semantic input:", FINAL_SEMANTIC_FOR_V5)
print("No rescrape. No overwrite.")

In [ ]:
# ============================================================
# 00M_TWITTER_NEAR_DUPLICATE_AUDIT
# ============================================================

NEAR_DUP_OUT = f"{MINILM_OUT_DIR}/twitter_near_duplicate_candidates.csv"
NEAR_DUP_SUMMARY_OUT = f"{MINILM_OUT_DIR}/twitter_near_duplicate_summary.csv"

def near_duplicate_audit(df, text_col="text", threshold=NEAR_DUP_THRESHOLD, max_features=50000):
    if df is None or len(df) < 2:
        return pd.DataFrame(), pd.DataFrame([{"n_rows": len(df) if df is not None else 0}])
    work = df.copy().reset_index(drop=True)
    work["_near_text"] = work[text_col].fillna("").astype(str).apply(normalize_text_key)
    work = work[work["_near_text"].str.len() > 0].reset_index(drop=True)
    if len(work) < 2:
        return pd.DataFrame(), pd.DataFrame([{"n_rows": len(work)}])
    vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2, max_features=max_features)
    X = vectorizer.fit_transform(work["_near_text"])
    nn = NearestNeighbors(n_neighbors=2, metric="cosine", algorithm="brute")
    nn.fit(X)
    distances, indices = nn.kneighbors(X)
    rows = []
    for i in range(len(work)):
        j = indices[i, 1]
        sim = 1 - distances[i, 1]
        if sim >= threshold and i < j:
            rows.append({
                "row_i": i,
                "row_j": int(j),
                "similarity": float(sim),
                "id_i": work.loc[i, "id"],
                "id_j": work.loc[j, "id"],
                "source_label_i": work.loc[i, "source_label"] if "source_label" in work.columns else None,
                "source_label_j": work.loc[j, "source_label"] if "source_label" in work.columns else None,
                "text_i": work.loc[i, text_col],
                "text_j": work.loc[j, text_col],
            })
    candidates = pd.DataFrame(rows).sort_values("similarity", ascending=False) if rows else pd.DataFrame()
    summary = pd.DataFrame([{
        "n_rows": len(work),
        "method": "TF-IDF character n-gram nearest-neighbour cosine similarity",
        "threshold": threshold,
        "near_duplicate_pair_count": len(candidates),
        "near_duplicate_pair_rate_vs_rows": len(candidates) / len(work) if len(work) else np.nan,
    }])
    return candidates, summary

if RUN_NEAR_DUPLICATE_AUDIT:
    near_dups, near_dup_summary = near_duplicate_audit(twitter_merged)
    near_dups.to_csv(NEAR_DUP_OUT, index=False)
    near_dup_summary.to_csv(NEAR_DUP_SUMMARY_OUT, index=False)
    print("Saved:", NEAR_DUP_OUT)
    print("Saved:", NEAR_DUP_SUMMARY_OUT)
    display(near_dup_summary)
    display(near_dups.head(30))


In [ ]:
# ============================================================
# 00N_COCHRAN_AND_TWO_ANNOTATOR_VALIDATION_TEMPLATES
# ============================================================

def cochran_sample_size(z=1.96, p=0.5, e=0.05, population_size=None):
    n0 = (z**2 * p * (1-p)) / (e**2)
    if population_size is None:
        return math.ceil(n0)
    n = n0 / (1 + ((n0 - 1) / population_size))
    return math.ceil(n)

population_n = len(twitter_merged)
cochran_n = cochran_sample_size(z=VALIDATION_CONFIDENCE_Z, p=VALIDATION_EXPECTED_P, e=VALIDATION_MARGIN_ERROR, population_size=population_n if population_n > 0 else None)
validation_n = min(max(cochran_n, MIN_VALIDATION_SAMPLE), MAX_VALIDATION_SAMPLE, population_n) if population_n else 0
print("Population N:", population_n)
print("Cochran recommended n:", cochran_n)
print("Chosen validation n:", validation_n)

VALIDATION_BASE_OUT = f"{MINILM_OUT_DIR}/manual_validation_sample_twitter_minilm.csv"
ANNOTATOR1_OUT = f"{MINILM_OUT_DIR}/manual_validation_sample_twitter_minilm_annotator1.csv"
ANNOTATOR2_OUT = f"{MINILM_OUT_DIR}/manual_validation_sample_twitter_minilm_annotator2.csv"
SEMANTIC_ONLY_OUT = f"{MINILM_OUT_DIR}/twitter_minilm_semantic_only_validation_sample.csv"

def make_validation_sample(df, n, semantic_only_n=SEMANTIC_ONLY_SAMPLE_SIZE):
    if df is None or len(df) == 0:
        return pd.DataFrame(), pd.DataFrame()
    samples = []
    for label, g in df.groupby("source_label", dropna=False):
        target = max(1, round(n * len(g) / len(df)))
        samples.append(g.sample(n=min(target, len(g)), random_state=42))
    sample = pd.concat(samples, ignore_index=True).drop_duplicates(subset=["id"])
    if len(sample) > n:
        sample = sample.sample(n=n, random_state=42)
    for col in ["domain_relevant_0_1", "spam_or_listing_0_1", "annotation_notes"]:
        sample[col] = ""
    keep_cols = ["id", "date", "username", "source_label", "retrieval_streams", "retrieval_substream", "minilm_max_similarity", "minilm_best_reference", "is_likely_agent_listing", "text", "domain_relevant_0_1", "spam_or_listing_0_1", "annotation_notes"]
    keep_cols = [c for c in keep_cols if c in sample.columns]
    sample = sample[keep_cols]
    sem = df[df["source_label"].isin(["semantic", "both"])].copy()
    if len(sem):
        sem = sem.sample(n=min(semantic_only_n, len(sem)), random_state=99)
        for col in ["domain_relevant_0_1", "spam_or_listing_0_1", "annotation_notes"]:
            sem[col] = ""
        sem = sem[[c for c in keep_cols if c in sem.columns]]
    else:
        sem = pd.DataFrame()
    return sample, sem

validation_sample, semantic_validation_sample = make_validation_sample(twitter_merged, validation_n)
validation_sample.to_csv(VALIDATION_BASE_OUT, index=False)
validation_sample.to_csv(ANNOTATOR1_OUT, index=False)
validation_sample.to_csv(ANNOTATOR2_OUT, index=False)
semantic_validation_sample.to_csv(SEMANTIC_ONLY_OUT, index=False)

print("Saved base validation sample:", VALIDATION_BASE_OUT)
print("Saved annotator 1 template:", ANNOTATOR1_OUT)
print("Saved annotator 2 template:", ANNOTATOR2_OUT)
print("Saved semantic-only validation sample:", SEMANTIC_ONLY_OUT)
display(validation_sample.head(20))


In [ ]:
# ============================================================
# 00O_COMPUTE_TWO_ANNOTATOR_AGREEMENT_KAPPA_PABAK
# ============================================================
# Fill the two annotator files, then rerun this cell.
# Required columns: domain_relevant_0_1 and spam_or_listing_0_1

IRR_OUT = f"{MINILM_OUT_DIR}/manual_validation_irr_results.csv"

def compute_pabak(y1, y2):
    agreement = np.mean(np.array(y1) == np.array(y2))
    return 2 * agreement - 1

def compute_irr(annotator1_path=ANNOTATOR1_OUT, annotator2_path=ANNOTATOR2_OUT):
    if not os.path.exists(annotator1_path) or not os.path.exists(annotator2_path):
        print("Annotator files not found.")
        return pd.DataFrame()
    a1 = pd.read_csv(annotator1_path, low_memory=False)
    a2 = pd.read_csv(annotator2_path, low_memory=False)
    if "id" not in a1.columns or "id" not in a2.columns:
        raise ValueError("Both annotator files must contain id column.")
    merged = a1.merge(a2, on="id", suffixes=("_a1", "_a2"))
    rows = []
    for label_col in ["domain_relevant_0_1", "spam_or_listing_0_1"]:
        c1 = f"{label_col}_a1"
        c2 = f"{label_col}_a2"
        if c1 not in merged.columns or c2 not in merged.columns:
            continue
        tmp = merged[["id", c1, c2]].copy()
        tmp[c1] = pd.to_numeric(tmp[c1], errors="coerce")
        tmp[c2] = pd.to_numeric(tmp[c2], errors="coerce")
        tmp = tmp[tmp[c1].isin([0, 1]) & tmp[c2].isin([0, 1])].copy()
        if len(tmp) == 0:
            continue
        y1 = tmp[c1].astype(int)
        y2 = tmp[c2].astype(int)
        rows.append({
            "label": label_col,
            "n_double_annotated": len(tmp),
            "percent_agreement": (y1 == y2).mean(),
            "cohen_kappa": cohen_kappa_score(y1, y2),
            "pabak": compute_pabak(y1, y2),
            "positive_rate_annotator1": y1.mean(),
            "positive_rate_annotator2": y2.mean(),
        })
    out = pd.DataFrame(rows)
    out.to_csv(IRR_OUT, index=False)
    print("Saved:", IRR_OUT)
    display(out)
    return out

irr_results = compute_irr()


In [ ]:
# ============================================================
# 00P_MINILM_MANIFEST_FOR_PAPER_METHODS
# ============================================================

manifest = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "method_name": "hybrid keyword retrieval with MiniLM/SBERT semantic relevance filtering",
    "collection": {
        "strict_stream": "explicit Dubai real-estate keyword queries",
        "broad_candidate_stream": "high-recall Dubai/UAE constrained rental, investment, developer, neighbourhood, and market-stress queries",
        "date_since": START_DATE,
        "date_until_exclusive": END_DATE,
        "query_exclusion_cues": TWITTER_QUERY_EXCLUSION_CUES,
        "twscrape_db_path": TWS_DB_PATH,
    },
    "minilm_filter": {
        "model": MINILM_MODEL_NAME,
        "threshold": MINILM_RELEVANCE_THRESHOLD,
        "reference_sentences": DUBAI_RE_REFERENCE_SENTENCES,
        "similarity": "cosine similarity over normalized sentence embeddings",
        "role": "semantic relevance filtering of broad candidate tweets, not direct Twitter vector search",
    },
    "validation": {
        "cochran_z": VALIDATION_CONFIDENCE_Z,
        "cochran_margin_error": VALIDATION_MARGIN_ERROR,
        "cochran_expected_p": VALIDATION_EXPECTED_P,
        "chosen_validation_sample_n": int(validation_n) if "validation_n" in globals() else None,
        "two_annotator_templates": [ANNOTATOR1_OUT, ANNOTATOR2_OUT],
        "irr_metrics": ["percent agreement", "Cohen's kappa", "PABAK"],
    },
    "outputs": {
        "strict_raw": STRICT_RAW_OUT,
        "broad_raw": BROAD_RAW_OUT,
        "broad_minilm_scored": MINILM_SCORED_OUT,
        "minilm_kept": MINILM_KEPT_OUT,
        "minilm_rejected": MINILM_REJECTED_OUT,
        "merged_twitter": MERGED_TWITTER_OUT,
        "validation_sample": VALIDATION_BASE_OUT,
        "semantic_only_validation_sample": SEMANTIC_ONLY_OUT,
        "near_duplicate_candidates": NEAR_DUP_OUT,
        "near_duplicate_summary": NEAR_DUP_SUMMARY_OUT,
    },
    "paper_caution": "This design supports semantic filtering and dataset construction. It does not prove causal effects or perfect relevance without manual validation.",
}

MANIFEST_OUT = f"{MINILM_OUT_DIR}/twitter_minilm_collection_manifest.json"
with open(MANIFEST_OUT, "w") as f:
    json.dump(manifest, f, indent=2)

print("Saved manifest:", MANIFEST_OUT)
print(json.dumps(manifest, indent=2)[:3000])


In [ ]:
# ============================================================
# 00Q_EXPORT_MINILM_OUTPUTS_TO_V5_INPUT_PATHS
# ============================================================
# Set EXPORT_TO_V5_INPUT_PATHS=True in config when ready.
# Then run v5 from 00_config onward with collection disabled.

def backup_file(path):
    if os.path.exists(path):
        ts = datetime.now().strftime("%Y%m%d_%H%M%S")
        backup_path = f"{path}.BACKUP_{ts}"
        shutil.copy2(path, backup_path)
        print("Backed up:", path, "->", backup_path)
    else:
        print("No existing file to backup:", path)

if EXPORT_TO_V5_INPUT_PATHS:
    if not globals().get("ALLOW_OVERWRITE_V5_INPUT_PATHS", False):
        raise RuntimeError(
            "EXPORT_TO_V5_INPUT_PATHS=True would overwrite/copy into the v5 raw input paths. "
            "Set ALLOW_OVERWRITE_V5_INPUT_PATHS=True only after you intentionally want that. "
            "By default, MiniLM outputs remain safely in MINILM_OUT_DIR."
        )
    if len(strict_flagged) == 0:
        raise ValueError("strict_flagged is empty; refusing to export to v5 regex input.")
    if len(minilm_kept_flagged) == 0:
        raise ValueError("minilm_kept_flagged is empty; refusing to export to v5 semantic input.")
    os.makedirs(os.path.dirname(V5_REGEX_RAW_PATH), exist_ok=True)
    os.makedirs(os.path.dirname(V5_SEMANTIC_RAW_PATH), exist_ok=True)
    backup_file(V5_REGEX_RAW_PATH)
    backup_file(V5_SEMANTIC_RAW_PATH)
    strict_flagged.to_parquet(V5_REGEX_RAW_PATH, index=False)
    minilm_kept_flagged.to_parquet(V5_SEMANTIC_RAW_PATH, index=False)
    strict_flagged.to_csv(V5_REGEX_RAW_PATH.replace(".parquet", ".csv"), index=False)
    minilm_kept_flagged.to_csv(V5_SEMANTIC_RAW_PATH.replace(".parquet", ".csv"), index=False)
    print("Exported to v5 input paths.")
else:
    print("EXPORT_TO_V5_INPUT_PATHS=False, so v5 inputs were not overwritten.")
    print("MiniLM outputs remain in:", MINILM_OUT_DIR)
    print("To intentionally overwrite/copy into v5 raw input paths later, set EXPORT_TO_V5_INPUT_PATHS=True and ALLOW_OVERWRITE_V5_INPUT_PATHS=True, then rerun this cell.")


In [ ]:
# ============================================================
# 00C_PATCH_V5_TO_REPROCESS_ONLY_AFTER_MINILM
# ============================================================
# Prevent v5 from scraping Twitter again.
# IMPORTANT: the later v5 00_config cell also defines Twitter paths, so a second
# path patch is inserted immediately AFTER v5 00_config. That later patch is the
# authoritative one used by 01_collect_or_load_twitter.

RUN_TWITTER_COLLECTION = False
COLLECT_REGEX_STREAM = False
COLLECT_SEMANTIC_STREAM = False

# Disable old embedding query-expansion cells if v5 still has them.
# We already do tweet-level MiniLM semantic filtering above.
USE_EMBEDDING_SEMANTIC_QUERY_EXPANSION = False

print("v5 Twitter collection flags patched:")
print("RUN_TWITTER_COLLECTION =", RUN_TWITTER_COLLECTION)
print("COLLECT_REGEX_STREAM =", COLLECT_REGEX_STREAM)
print("COLLECT_SEMANTIC_STREAM =", COLLECT_SEMANTIC_STREAM)
print("USE_EMBEDDING_SEMANTIC_QUERY_EXPANSION =", USE_EMBEDDING_SEMANTIC_QUERY_EXPANSION)
print("A second path patch after v5 00_config will force v5 to use the MiniLM-kept semantic stream.")


# Continue: v5 full processing pipeline

From this point onward, run the original v5 cells normally. v5 collection has been patched off. By default, the MiniLM outputs stay in their own folder and your older v5 raw input files are not overwritten. Only copy MiniLM outputs into the v5 input paths if you explicitly enable both `EXPORT_TO_V5_INPUT_PATHS=True` and `ALLOW_OVERWRITE_V5_INPUT_PATHS=True`.

Expected next sections:
- `00_config`
- load Twitter
- clean Twitter
- load Reddit
- reconstruct Reddit threads
- standardise schema
- merge
- domain filter
- reviewer-response filters/validation
- validation statistics
- export outputs
- optional DLD social correlation


# Dubai Real Estate Social Media Dataset Pipeline v5

One-button Colab pipeline for the ICOCO 2026 dataset/preprocessing paper.

Scope: Twitter/X + Reddit only. This notebook supports two execution modes:

1. **Collection mode**: optionally runs `twscrape` from an auditable regex/semantic query manifest and regenerates the archived raw Twitter/X parquet files.
2. **Preprocessing mode**: loads existing saved `twscrape` outputs and Reddit raw exports, cleans them, reconstructs Reddit threads, standardises the schema, applies deterministic domain relevance filtering, creates validation/statistics tables, and exports the final research-ready dataset.

Collection is disabled by default (`RUN_TWITTER_COLLECTION = False`) so the notebook can reproduce the final dataset from saved raw files without requiring Twitter/X credentials or live scraping.

Excluded by design: Facebook, Apify execution, sentiment analysis, VADER/TextBlob/AFINN/SentiWordNet, predictive modelling, manual-label claims, and BERTopic-as-results.

**v5 reviewer-response additions:** platform-native interaction metadata, explicit agent/listing cue filtering, exact/near-duplicate audit, semantic-only annotation sampling, manual validation templates with Cohen's kappa/precision/recall/F1, and an optional DLD weekly correlation framework.

**v5 addition:** embedding-based Twitter/X semantic query expansion using Sentence-BERT-style sentence embeddings, cosine similarity, and a fixed threshold for reproducible expanded query generation.


## 00_config

In [ ]:
# ============================================================
# 00_config
# ============================================================

# Install lightweight dependencies when running in Colab.
# No sentiment-analysis packages are installed or used.
import sys
import subprocess
import importlib.util

INSTALL_PACKAGES = True
_REQUIRED_PACKAGES = ["pyarrow", "emoji", "langdetect", "scikit-learn", "scipy"]

if INSTALL_PACKAGES:
    missing = [p for p in _REQUIRED_PACKAGES if importlib.util.find_spec(p.replace("-", "_")) is None]
    # scikit-learn imports as sklearn
    if importlib.util.find_spec("sklearn") is not None and "scikit-learn" in missing:
        missing.remove("scikit-learn")
    if missing:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

from pathlib import Path
import os
import re
import json
import hashlib
from collections import Counter

import numpy as np
import pandas as pd
import emoji

try:
    from langdetect import detect, DetectorFactory
    DetectorFactory.seed = 42
    LANGDETECT_AVAILABLE = True
except Exception:
    LANGDETECT_AVAILABLE = False

try:
    from sklearn.feature_extraction.text import CountVectorizer
    SKLEARN_AVAILABLE = True
except Exception:
    SKLEARN_AVAILABLE = False

# Mount Google Drive if running in Colab.
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    pass

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)
np.random.seed(42)

# -----------------------------
# Paths
# -----------------------------
BASE_PATH = Path("/content/drive/MyDrive/Dubai_Real_Estate_Data")

# Twitter/X raw output directories. These are generated by twscrape collection mode,
# or loaded directly when RUN_TWITTER_COLLECTION = False.
TWITTER_REGEX_RAW_DIR = BASE_PATH / "raw_data_files"
TWITTER_SEMANTIC_RAW_DIR = BASE_PATH / "raw_data_files_semantic_filtering_x"

RAW_TWITTER_REGEX_PATH = TWITTER_REGEX_RAW_DIR / "master_raw.parquet"
RAW_TWITTER_SEMANTIC_PATH = TWITTER_SEMANTIC_RAW_DIR / "master_raw_semantic_run.parquet"

# No-rescrape MiniLM output. If this file exists, v5 will use it as the semantic stream
# without overwriting your old semantic raw input. Prefer the flagged file because it
# carries MiniLM scores plus listing/spam flags into v5.
MINILM_KEPT_TWITTER_PATH = BASE_PATH / "twitter_minilm_semantic_collection" / "twitter_minilm_kept_semantic_stream_flagged.parquet"
MINILM_KEPT_TWITTER_UNFLAGGED_PATH = BASE_PATH / "twitter_minilm_semantic_collection" / "twitter_minilm_kept_semantic_stream.parquet"

RAW_TWITTER_SEMANTIC_CANDIDATES = [
    MINILM_KEPT_TWITTER_PATH,             # preferred after MiniLM filtering + quality flags
    MINILM_KEPT_TWITTER_UNFLAGGED_PATH,   # fallback if flags cell was skipped
    RAW_TWITTER_SEMANTIC_PATH,            # fallback: original semantic candidate stream
    TWITTER_SEMANTIC_RAW_DIR / "master_raw.parquet",
]

# Optional twscrape collection controls. Keep False for normal paper reproduction runs.
RUN_TWITTER_COLLECTION = False
FORCE_RECOLLECT_TWITTER = False
INSTALL_TWSCRAPE_IF_COLLECTION_ENABLED = True
TWSCRAPE_DB_PATH = BASE_PATH / "twscrape" / "accounts.db"
TWSCRAPE_ACCOUNT_CSV = BASE_PATH / "twscrape" / "accounts_private.csv"
TWSCRAPE_MAX_TWEETS_PER_QUERY = 5000
TWSCRAPE_BASE_COOLDOWN_SECONDS = 10

REDDIT_POSTS_PATH = BASE_PATH / "reddit_data" / "dubairealestate" / "dubairealestate_full_raw_data.csv"
REDDIT_COMMENTS_PATH = BASE_PATH / "reddit_data" / "dubairealestate" / "dubairealestate_full_raw_comments.csv"

# Optional hand-labelled Reddit thread sample from the Reddit notebook.
# If this file exists and contains labels, the pipeline evaluates the deterministic
# domain filter against it and exports precision/recall/F1.
REDDIT_GOLD_SAMPLE_CANDIDATES = [
    BASE_PATH / "praj_reddit" / "threads_reddit_gold_sample.csv",
    BASE_PATH / "reddit_data" / "dubairealestate" / "threads_reddit_gold_sample.csv",
]

FINAL_OUTPUT_DIR = BASE_PATH / "final_pipeline_outputs"
FINAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Optional separate output subfolder for validation tables.
TABLE_OUTPUT_DIR = FINAL_OUTPUT_DIR / "paper_ready_tables"
TABLE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Observation window
# -----------------------------
OBS_START = pd.Timestamp("2026-01-01", tz="UTC")
OBS_END_EXCLUSIVE = pd.Timestamp("2026-05-01", tz="UTC")  # Jan 1 through Apr 30 inclusive

# Weekly collection cohorts used for Twitter/X query execution.
# End dates are exclusive, matching Twitter/X search syntax and the preprocessing window.
TWITTER_COHORTS = [
    {"start": "2026-01-01", "end": "2026-01-08", "label": "jan_wk1"},
    {"start": "2026-01-08", "end": "2026-01-15", "label": "jan_wk2"},
    {"start": "2026-01-15", "end": "2026-01-22", "label": "jan_wk3"},
    {"start": "2026-01-22", "end": "2026-02-01", "label": "jan_wk4"},
    {"start": "2026-02-01", "end": "2026-02-08", "label": "feb_wk1"},
    {"start": "2026-02-08", "end": "2026-02-15", "label": "feb_wk2"},
    {"start": "2026-02-15", "end": "2026-02-22", "label": "feb_wk3"},
    {"start": "2026-02-22", "end": "2026-03-01", "label": "feb_wk4"},
    {"start": "2026-03-01", "end": "2026-03-08", "label": "mar_wk1"},
    {"start": "2026-03-08", "end": "2026-03-15", "label": "mar_wk2"},
    {"start": "2026-03-15", "end": "2026-03-22", "label": "mar_wk3"},
    {"start": "2026-03-22", "end": "2026-04-01", "label": "mar_wk4"},
    {"start": "2026-04-01", "end": "2026-04-08", "label": "apr_wk1"},
    {"start": "2026-04-08", "end": "2026-04-15", "label": "apr_wk2"},
    {"start": "2026-04-15", "end": "2026-04-22", "label": "apr_wk3"},
    {"start": "2026-04-22", "end": "2026-05-01", "label": "apr_wk4"},
]

# -----------------------------
# Twitter/X query manifest components
# -----------------------------
# The regex stream is the original keyword/regex retrieval strategy.
TWITTER_REGEX_STREAMS = {
    "macro_investment": (
        '("dubai property" OR "dubai real estate" OR "offplan dubai" OR '
        '"dubai market" OR "buy property dubai" OR '
        '"buying property in dubai" OR "buy apartment dubai" OR '
        '"buy villa dubai" OR "dubai apartment" OR '
        '"dubai villa" OR "property investment dubai" OR '
        '"dubai home" OR "owning property in dubai")'
    ),
    "rental_painpoints": (
        '("dubai rent" OR "dubai rental" OR "dubai landlord" OR "dubai tenant" OR '
        '"rent increase dubai" OR "dubai rent increase" OR "eviction dubai" OR '
        '"dubai eviction" OR "service charge dubai" OR "ejari" OR '
        '"rera rent" OR "dubai lease")'
    ),
    "developer_activity": (
        '(emaar OR damac OR nakheel OR sobha OR danube OR binghatti OR azizi OR '
        'aldar OR omniyat) (dubai OR uae OR rera)'
    ),
    "neighborhood_retail": (
        '(("Dubai Marina" OR "JVC" OR "Business Bay" OR '
        '"Palm Jumeirah" OR "Dubai Hills" OR '
        '"Downtown Dubai" OR "JLT" OR '
        '"Dubai Creek Harbour" OR "Arabian Ranches" OR '
        '"Motor City" OR "Dubai Sports City" OR '
        '"Al Furjan" OR "Town Square" OR '
        '"Dubai Silicon Oasis" OR "Mirdif" OR '
        '"Discovery Gardens" OR "The Greens" OR '
        '"The Springs" OR "The Meadows" OR '
        '"DAMAC Hills" OR "Dubai South" OR '
        '"International City") '
        'AND '
        '(property OR apartment OR villa OR rent OR rental OR buying OR investment))'
    ),
}

# The semantic stream is a broader query-expansion strategy.
# Subqueries are still tagged under the same conceptual retrieval_stream for clean provenance.
TWITTER_SEMANTIC_STANDARD_STREAMS = {
    "macro_investment": TWITTER_REGEX_STREAMS["macro_investment"],
    "developer_activity": TWITTER_REGEX_STREAMS["developer_activity"],
}

TWITTER_SEMANTIC_RENTAL_SUBQUERIES = {
    "rental_market": (
        '(rent OR rental OR tenant OR landlord OR lease OR renewal) '
        'AND (dubai OR uae)'
    ),
    "rental_legal": (
        '(eviction OR ejari OR "service charge" OR "rent increase" OR '
        '"notice period" OR "rental dispute" OR "security deposit") '
        'AND (dubai OR uae)'
    ),
}

TWITTER_SEMANTIC_NEIGHBORHOOD_SUBQUERIES = {
    "nb_core": (
        '(("Dubai Marina" OR "JVC" OR "Business Bay" OR '
        '"Palm Jumeirah" OR "Downtown Dubai" OR "JLT" OR '
        '"Dubai Hills" OR "Arabian Ranches") '
        'AND (property OR apartment OR villa OR rent OR buying))'
    ),
    "nb_emerging": (
        '(("Dubai Creek Harbour" OR "Motor City" OR "Al Furjan" OR '
        '"Town Square" OR "Dubai South" OR "DAMAC Hills" OR '
        '"Mirdif" OR "Dubai Sports City") '
        'AND (property OR apartment OR villa OR rent OR buying))'
    ),
    "nb_affordable": (
        '(("Discovery Gardens" OR "The Greens" OR "The Springs" OR '
        '"The Meadows" OR "International City" OR '
        '"Dubai Silicon Oasis" OR "Remraam") '
        'AND (property OR apartment OR villa OR rent OR buying))'
    ),
}

TWITTER_QUERY_EXCLUSION_CUES = (
    '-crypto -giveaway -nft -job -hiring '
    '-biggboss -bb19 -#biggboss19 -#bb19 '
    '-#tanyamittal -#farrhanabhatt -#gauravkhanna'
)

# -----------------------------
# Filtering controls
# -----------------------------
APPLY_LANGDETECT_IF_NO_LANG_COLUMN = True
APPLY_LANGDETECT_TO_REDDIT = True
MIN_TOKEN_COUNT = 3
MAX_NOISE_CUES_FOR_DOMAIN_ROW = 1
ANONYMIZE_AUTHORS_FOR_PUBLIC_RELEASE = False  # Set True before publishing externally.

# Reddit text duplicate policy:
# The final default is ID deduplication + normalized-text duplicate audit only.
# This is deliberate: identical comments across different Reddit threads are not
# automatically false duplicates because repeated short advice can be contextually valid.
# Set this True only if you decide to remove exact repeated text within the same thread/post type.
REDDIT_REMOVE_NORMALIZED_TEXT_DUPLICATES = False

# Labels accepted for optional gold-sample evaluation. Adjust only if your CSV uses other labels.
GOLD_POSITIVE_LABELS = {"1", "yes", "y", "true", "relevant", "domain", "in_domain", "include", "keep", "real_estate", "real estate"}
GOLD_NEGATIVE_LABELS = {"0", "no", "n", "false", "irrelevant", "not relevant", "out_of_domain", "exclude", "remove", "noise"}

# -----------------------------
# Domain keyword lexicon
# -----------------------------
DOMAIN_KEYWORDS = sorted(set([
    # General real estate / housing
    "real estate", "property", "properties", "housing", "home", "homes", "unit", "units",
    "residential", "commercial property", "land", "plot",

    # Property types
    "apartment", "apartments", "flat", "flats", "villa", "villas", "townhouse", "townhouses",
    "studio", "studios", "penthouse", "penthouses", "mansion", "bedroom", "maid room",

    # Dubai / UAE real estate institutions and transaction terms
    "dubai", "uae", "rera", "dld", "dubai land department", "ejari", "oqood", "title deed",
    "dewa", "service charge", "maintenance fee",

    # Renting / tenancy
    "rent", "rental", "renting", "tenant", "tenants", "landlord", "landlords", "lease", "leasing",
    "eviction", "rent increase", "rental increase", "security deposit", "tenancy contract",

    # Buying / selling / investment
    "buy", "buying", "buyer", "buyers", "sell", "selling", "seller", "sellers", "purchase",
    "mortgage", "home loan", "down payment", "payment plan", "roi", "yield", "rental yield",
    "capital appreciation", "investment property", "property investment", "flipping", "speculation",
    "transaction", "transactions", "listing", "listings", "asking price", "market value",
    "price per sqft", "sqft", "sq ft", "square foot", "handover",

    # Off-plan / development
    "off plan", "off-plan", "offplan", "developer", "developers", "launch", "project", "handover",
    "construction", "under construction", "freehold", "leasehold",

    # Brokers / agents
    "broker", "brokers", "agent", "agents", "real estate agent", "property management",

    # Major developers
    "emaar", "damac", "nakheel", "sobha", "azizi", "ellington", "meraas", "danube",
    "binghatti", "aldar", "omniyat", "reportage", "mag", "dubai properties",

    # Popular Dubai communities / locations
    "dubai marina", "marina", "business bay", "downtown", "downtown dubai", "jvc",
    "jumeirah village circle", "jlt", "jumeirah lake towers", "arabian ranches", "dubai hills",
    "palm jumeirah", "creek harbour", "dubai creek harbour", "silicon oasis", "dubai silicon oasis",
    "motor city", "dubai sports city", "al furjan", "town square", "mirdif", "discovery gardens",
    "the greens", "the springs", "the meadows", "damac hills", "dubai south", "international city",
    "dip", "dubai investment park", "verdana",
]), key=lambda x: (-len(x), x))

NOISE_CUES = sorted(set([
    "crypto", "bitcoin", "ethereum", "nft", "airdrop", "token", "giveaway", "promo code", "coupon",
    "job", "jobs", "hiring", "vacancy", "apply now", "career", "salary", "recruitment",
    "biggboss", "bb19", "tanyamittal", "farrhana", "gauravkhanna", "celebrity", "bollywood",
    "restaurant", "food delivery", "hotel booking", "flight ticket", "tourist visa",
    "forex signal", "trading signal", "casino", "betting",
]), key=lambda x: (-len(x), x))

BOT_USERNAME_PATTERN = re.compile(r"(bot|auto|automoderator|crawler|scraper|spam)", re.IGNORECASE)
URL_PATTERN = re.compile(r"https?://\S+|www\.\S+", re.IGNORECASE)
MENTION_PATTERN = re.compile(r"@\w+")
HASHTAG_PATTERN = re.compile(r"#(\w+)")
WHITESPACE_PATTERN = re.compile(r"\s+")
DELETED_PATTERN = re.compile(r"^\s*\[(removed|deleted)\]\s*$", re.IGNORECASE)

EXCLUDED_FINAL_COLUMNS = {
    "sentiment_score", "sentiment_vader", "sentiment_lexicon", "vader_score",
    "emoji_score", "afinn", "textblob", "sentiwordnet", "polarity", "subjectivity",
    "topic", "topic_name", "Topic", "Topic_Name",
    "forward_looking", "retroactive", "speculative",  # discourse/sentiment-adjacent, excluded for dataset-only paper
}

UNIFIED_SCHEMA = [
    "id", "raw_text", "clean_text", "date", "platform", "source_label", "post_type",
    "platform_interaction_value", "platform_interaction_definition",
    "likeCount", "retweetCount", "replyCount", "quoteCount",
    "is_retweet", "is_reply", "username", "url", "retrieval_stream", "retrieval_streams",
    "extraction_cohort", "query_used", "collected_at", "subreddit", "parent_id", "link_id",
    "thread_id", "domain_keyword_count", "domain_keywords_matched", "noise_keyword_count",
    "agent_listing_cue_count", "agent_listing_cues_matched", "is_likely_agent_listing",
    "is_domain_relevant", "pipeline_keep_final", "yearweek", "month", "week_num"
]


# -----------------------------
# Reviewer-response controls (v4)
# -----------------------------
# Interaction metadata is kept as platform-native only; no cross-platform engagement comparison.
# The legacy engagement_score column may be used internally for sorting old raw rows, but it is
# not exported in the final paper-facing schema.

# Agent/listing filtering. Use True when the paper focuses on genuine discourse rather than listings.
REMOVE_LIKELY_AGENT_LISTINGS_FROM_MASTER = True
AGENT_LISTING_CUE_THRESHOLD = 2

# Near-duplicate audit. This is an audit by default; candidates are not silently removed.
RUN_NEAR_DUPLICATE_AUDIT = True
NEAR_DUPLICATE_METHOD = "tfidf_char_cosine_nearest_neighbor"
NEAR_DUPLICATE_SIMILARITY_THRESHOLD = 0.90
NEAR_DUPLICATE_MAX_FEATURES = 50000
NEAR_DUPLICATE_NGRAM_RANGE = (3, 5)
NEAR_DUPLICATE_AUDIT_MAX_ROWS_PER_PLATFORM = None  # set e.g. 20000 for faster approximate audit

# Manual validation sampling.
VALIDATION_RANDOM_SEED = 42
MANUAL_VALIDATION_SAMPLE_SIZE = 500
TWITTER_SEMANTIC_ONLY_SAMPLE_SIZE = 150

MANUAL_VALIDATION_ANNOTATOR1_PATH = TABLE_OUTPUT_DIR / "manual_validation_sample_annotator1_completed.csv"
MANUAL_VALIDATION_ANNOTATOR2_PATH = TABLE_OUTPUT_DIR / "manual_validation_sample_annotator2_completed.csv"
TWITTER_SEMANTIC_ANNOTATOR1_PATH = TABLE_OUTPUT_DIR / "twitter_semantic_only_validation_annotator1_completed.csv"
TWITTER_SEMANTIC_ANNOTATOR2_PATH = TABLE_OUTPUT_DIR / "twitter_semantic_only_validation_annotator2_completed.csv"

# Semantic query expansion controls (v5 reviewer fix).
# The archived semantic file from earlier runs may still be manual/domain-informed. To claim
# embedding-based semantic expansion in the paper, rerun Twitter collection with
# RUN_TWITTER_COLLECTION = True and USE_EMBEDDING_SEMANTIC_QUERY_EXPANSION = True.
USE_EMBEDDING_SEMANTIC_QUERY_EXPANSION = True
INSTALL_SENTENCE_TRANSFORMERS_IF_NEEDED = True
SEMANTIC_QUERY_EXPANSION_TYPE = "sbert_embedding_similarity_query_expansion"
SEMANTIC_QUERY_EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
SEMANTIC_QUERY_SIMILARITY_THRESHOLD = 0.52
SEMANTIC_QUERY_MAX_SELECTED_TERMS_PER_STREAM = 18
SEMANTIC_QUERY_TERMS_PER_TWITTER_QUERY = 6
SEMANTIC_QUERY_VALIDATION_REQUIRED = True
SEMANTIC_QUERY_EXPANSION_OUTPUT_DIR = FINAL_OUTPUT_DIR / "semantic_query_expansion"
SEMANTIC_QUERY_EXPANSION_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Optional DLD/social-media correlation extension.
RUN_DLD_CORRELATION_EXTENSION = True
DLD_TRANSACTIONS_PATH = BASE_PATH / "dld" / "transactions-2026-06-19.csv"
DLD_OUTPUT_DIR = FINAL_OUTPUT_DIR / "dld_social_correlation"
DLD_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DLD_LEAD_LAG_WEEKS = list(range(-4, 5))

retention_log = []
paper_tables = {}


def first_existing_path(paths, label):
    for p in paths:
        p = Path(p)
        if p.exists():
            return p
    raise FileNotFoundError(f"No existing {label} file found. Checked: {[str(Path(p)) for p in paths]}")


def first_existing_optional_path(paths):
    for p in paths:
        p = Path(p)
        if p.exists():
            return p
    return None


def read_table(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing input file: {path}")
    if path.suffix.lower() == ".parquet":
        return pd.read_parquet(path)
    if path.suffix.lower() in {".csv", ".txt"}:
        return pd.read_csv(path, low_memory=False)
    raise ValueError(f"Unsupported file extension for {path}")


def safe_to_datetime(series):
    return pd.to_datetime(series, errors="coerce", utc=True)


def normalise_text(raw):
    if pd.isna(raw):
        return ""
    text = str(raw)
    text = emoji.demojize(text, delimiters=(" :", ": "))
    text = URL_PATTERN.sub(" ", text)
    text = MENTION_PATTERN.sub(" ", text)
    text = HASHTAG_PATTERN.sub(r"\1", text)
    text = text.replace("&amp;", "&")
    text = WHITESPACE_PATTERN.sub(" ", text).strip()
    return text


def token_count(text):
    return len(re.findall(r"\b\w+\b", str(text)))


def _contains_keyword(text, keyword):
    # Boundary-aware but still phrase-friendly.
    pattern = r"(?<!\w)" + re.escape(keyword.lower()) + r"(?!\w)"
    return re.search(pattern, str(text).lower()) is not None


def matched_keywords(text, keywords):
    lowered = str(text).lower()
    return [kw for kw in keywords if _contains_keyword(lowered, kw)]


def count_noise_cues(text):
    return len(matched_keywords(text, NOISE_CUES))


def has_deleted_marker(text):
    return bool(DELETED_PATTERN.match(str(text)))


def detect_is_english(text):
    if not LANGDETECT_AVAILABLE:
        return True
    try:
        sample = str(text).strip()
        if len(sample) < 20:
            # Very short English/domain rows are difficult for language detection.
            return True
        return detect(sample) == "en"
    except Exception:
        return False


def language_mask(df, text_col="raw_text", lang_col="lang", fallback_detect=True):
    if lang_col in df.columns:
        lang = df[lang_col].astype(str).str.lower().str.strip()
        mask = lang.eq("en") | lang.eq("english")
        # If lang is missing for some rows and fallback is enabled, detect those rows only.
        missing_lang = lang.isna() | lang.eq("") | lang.eq("nan") | lang.eq("none")
        if fallback_detect and missing_lang.any():
            detected = df.loc[missing_lang, text_col].apply(detect_is_english)
            mask.loc[missing_lang] = detected
        return mask.fillna(False)
    if fallback_detect:
        return df[text_col].apply(detect_is_english)
    return pd.Series(True, index=df.index)


def compute_engagement_score(df):
    # Twitter weighted engagement if available; Reddit score otherwise.
    for col in ["likeCount", "retweetCount", "replyCount", "quoteCount", "score"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)
    if "engagement_score" in df.columns:
        df["engagement_score"] = pd.to_numeric(df["engagement_score"], errors="coerce").fillna(0)
    elif {"likeCount", "retweetCount", "replyCount", "quoteCount"}.issubset(df.columns):
        df["engagement_score"] = (
            df["likeCount"] * 1 + df["replyCount"] * 2 + df["retweetCount"] * 3 + df["quoteCount"] * 2
        )
    elif "score" in df.columns:
        df["engagement_score"] = pd.to_numeric(df["score"], errors="coerce").fillna(0)
    else:
        df["engagement_score"] = 0
    return df


def add_time_features(df):
    df = df.copy()
    df["date"] = safe_to_datetime(df["date"])
    df["yearweek"] = df["date"].dt.strftime("%G-W%V")
    df["month"] = df["date"].dt.strftime("%Y-%m")
    df["week_num"] = df["date"].dt.isocalendar().week.astype("Int64")
    return df


def filter_observation_window(df, date_col="date"):
    df = df.copy()
    df[date_col] = safe_to_datetime(df[date_col])
    return df[(df[date_col] >= OBS_START) & (df[date_col] < OBS_END_EXCLUSIVE)].copy()


def log_retention(platform, stage, count, note=""):
    retention_log.append({
        "platform": platform,
        "stage_order": len(retention_log),
        "stage": stage,
        "count": int(count),
        "note": note,
    })


def make_retention_table():
    rt = pd.DataFrame(retention_log)
    if rt.empty:
        return rt
    rt = rt.sort_values(["platform", "stage_order"]).copy()
    rt["previous_count"] = rt.groupby("platform")["count"].shift(1)
    rt["removed_from_previous"] = rt["previous_count"] - rt["count"]
    rt["retained_from_previous_pct"] = np.where(
        rt["previous_count"].notna() & (rt["previous_count"] != 0),
        (rt["count"] / rt["previous_count"] * 100).round(2),
        np.nan,
    )
    raw = rt.groupby("platform")["count"].transform("first")
    rt["retained_from_platform_raw_pct"] = np.where(raw != 0, (rt["count"] / raw * 100).round(2), np.nan)
    return rt


def write_df(df, path, index=False):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.suffix.lower() == ".parquet":
        df.to_parquet(path, index=index, engine="pyarrow")
    elif path.suffix.lower() == ".csv":
        df.to_csv(path, index=index, encoding="utf-8")
    else:
        raise ValueError(f"Unsupported output extension: {path}")
    return path


def clean_for_final_schema(df):
    df = df.copy()
    drop_cols = [c for c in df.columns if c in EXCLUDED_FINAL_COLUMNS or c.lower() in {x.lower() for x in EXCLUDED_FINAL_COLUMNS}]
    df = df.drop(columns=drop_cols, errors="ignore")
    for col in UNIFIED_SCHEMA:
        if col not in df.columns:
            df[col] = pd.NA
    return df[UNIFIED_SCHEMA].copy()


def hash_identifier(value):
    if pd.isna(value):
        return pd.NA
    return hashlib.sha256(str(value).encode("utf-8")).hexdigest()[:16]


def maybe_anonymize(df):
    df = df.copy()
    if ANONYMIZE_AUTHORS_FOR_PUBLIC_RELEASE and "username" in df.columns:
        df["username"] = df["username"].apply(hash_identifier)
    return df


def make_reddit_text_duplicate_audit(df):
    """Audit normalized-text duplicates without removing them by default.

    Reddit comments are first deduplicated by Reddit ID. This audit quantifies exact
    normalized-text repeats so the paper can state the choice explicitly.
    """
    if df is None or len(df) == 0:
        return pd.DataFrame([{
            "metric": "reddit_rows_audited", "value": 0,
            "note": "No Reddit rows available for duplicate audit."
        }])

    tmp = df.copy()
    tmp["_norm_text"] = tmp["clean_text"].fillna("").astype(str).str.lower().str.replace(r"\s+", " ", regex=True).str.strip()
    tmp = tmp[tmp["_norm_text"].ne("")].copy()

    exact_groups = tmp.groupby(["post_type", "_norm_text"], dropna=False).agg(
        rows=("id", "count"),
        unique_threads=("thread_id", pd.Series.nunique),
    ).reset_index()
    duplicate_groups = exact_groups[exact_groups["rows"] > 1]
    cross_thread_groups = duplicate_groups[duplicate_groups["unique_threads"] > 1]

    within_thread_groups = tmp.groupby(["post_type", "thread_id", "_norm_text"], dropna=False).size().reset_index(name="rows")
    within_thread_duplicate_groups = within_thread_groups[within_thread_groups["rows"] > 1]

    return pd.DataFrame([
        {"metric": "reddit_rows_audited", "value": int(len(tmp)), "note": "Rows with non-empty normalized text."},
        {"metric": "exact_normalized_text_duplicate_groups", "value": int(len(duplicate_groups)), "note": "Groups with the same post_type and normalized clean_text appearing more than once."},
        {"metric": "exact_normalized_text_duplicate_rows", "value": int(duplicate_groups["rows"].sum() if not duplicate_groups.empty else 0), "note": "Rows involved in exact normalized-text duplicate groups."},
        {"metric": "cross_thread_duplicate_groups", "value": int(len(cross_thread_groups)), "note": "Duplicate text groups appearing across more than one Reddit thread."},
        {"metric": "within_thread_duplicate_groups", "value": int(len(within_thread_duplicate_groups)), "note": "Duplicate text groups repeated within the same thread and post_type."},
        {"metric": "normalized_text_duplicate_policy", "value": "audit_only" if not REDDIT_REMOVE_NORMALIZED_TEXT_DUPLICATES else "remove_within_thread_exact_duplicates", "note": "Default is audit-only; Reddit is ID-deduplicated but not globally text-deduplicated."},
    ])


def _gold_context_from_columns(df):
    context_priority = ["clean_thread_text", "full_thread_text", "post_text", "clean_text", "raw_text", "text"]
    for col in context_priority:
        if col in df.columns:
            return df[col].fillna("").astype(str)
    parts = []
    for col in ["title", "selftext", "all_comments", "body"]:
        if col in df.columns:
            parts.append(df[col].fillna("").astype(str))
    if parts:
        context = parts[0]
        for part in parts[1:]:
            context = context + " " + part
        return context
    return pd.Series("", index=df.index)


def _normalise_gold_label(value):
    if pd.isna(value):
        return pd.NA
    label = str(value).strip().lower()
    if label == "":
        return pd.NA
    if label in GOLD_POSITIVE_LABELS:
        return True
    if label in GOLD_NEGATIVE_LABELS:
        return False
    return pd.NA


def evaluate_reddit_gold_sample(gold_path):
    """Evaluate the deterministic domain filter against a hand-labelled sample when available.

    The function is non-fatal: missing files or empty/unrecognised labels create an
    explanatory summary row rather than stopping the full one-button pipeline.
    """
    if gold_path is None:
        summary = pd.DataFrame([{
            "metric": "gold_sample_status",
            "value": "missing",
            "note": "No threads_reddit_gold_sample.csv found in configured candidate paths.",
        }])
        predictions = pd.DataFrame()
        return summary, predictions

    gold = read_table(gold_path).copy()
    context = _gold_context_from_columns(gold).apply(normalise_text)
    gold["filter_context_text"] = context
    gold["domain_keywords_matched"] = context.apply(lambda t: "|".join(matched_keywords(t, DOMAIN_KEYWORDS)))
    gold["domain_keyword_count"] = gold["domain_keywords_matched"].apply(lambda s: 0 if not s else len(str(s).split("|")))
    gold["noise_keyword_count"] = context.apply(count_noise_cues).astype(int)
    gold["predicted_relevant"] = (gold["domain_keyword_count"] > 0) & (gold["noise_keyword_count"] <= MAX_NOISE_CUES_FOR_DOMAIN_ROW)

    label_col = next((c for c in ["actual_label", "label", "is_relevant", "relevance_label", "domain_label", "manual_label"] if c in gold.columns), None)
    if label_col is None:
        summary = pd.DataFrame([{
            "metric": "gold_sample_status",
            "value": "no_label_column",
            "note": f"Found {gold_path}, but no recognised label column was present.",
        }, {
            "metric": "gold_sample_rows",
            "value": int(len(gold)),
            "note": "Rows in the gold sample file.",
        }])
        return summary, gold

    gold["actual_relevant"] = gold[label_col].apply(_normalise_gold_label)
    labelled = gold[gold["actual_relevant"].notna()].copy()

    if labelled.empty:
        summary = pd.DataFrame([{
            "metric": "gold_sample_status",
            "value": "unlabelled",
            "note": f"Found {gold_path}, but {label_col} has no recognised positive/negative labels yet.",
        }, {
            "metric": "gold_sample_rows",
            "value": int(len(gold)),
            "note": "Rows in the gold sample file.",
        }])
        return summary, gold

    y_true = labelled["actual_relevant"].astype(bool)
    y_pred = labelled["predicted_relevant"].astype(bool)
    tp = int((y_true & y_pred).sum())
    fp = int((~y_true & y_pred).sum())
    tn = int((~y_true & ~y_pred).sum())
    fn = int((y_true & ~y_pred).sum())
    precision = tp / (tp + fp) if (tp + fp) else np.nan
    recall = tp / (tp + fn) if (tp + fn) else np.nan
    f1 = 2 * precision * recall / (precision + recall) if precision == precision and recall == recall and (precision + recall) else np.nan
    accuracy = (tp + tn) / len(labelled) if len(labelled) else np.nan

    summary = pd.DataFrame([
        {"metric": "gold_sample_status", "value": "evaluated", "note": f"Evaluated deterministic filter against {gold_path}."},
        {"metric": "gold_sample_rows", "value": int(len(gold)), "note": "Rows in the gold sample file."},
        {"metric": "gold_sample_labelled_rows", "value": int(len(labelled)), "note": f"Rows with recognised labels in {label_col}."},
        {"metric": "gold_sample_precision", "value": round(precision, 4) if precision == precision else np.nan, "note": "TP / (TP + FP)."},
        {"metric": "gold_sample_recall", "value": round(recall, 4) if recall == recall else np.nan, "note": "TP / (TP + FN)."},
        {"metric": "gold_sample_f1", "value": round(f1, 4) if f1 == f1 else np.nan, "note": "Harmonic mean of precision and recall."},
        {"metric": "gold_sample_accuracy", "value": round(accuracy, 4) if accuracy == accuracy else np.nan, "note": "Overall accuracy."},
        {"metric": "gold_sample_tp", "value": tp, "note": "True positives."},
        {"metric": "gold_sample_fp", "value": fp, "note": "False positives."},
        {"metric": "gold_sample_tn", "value": tn, "note": "True negatives."},
        {"metric": "gold_sample_fn", "value": fn, "note": "False negatives."},
        {"metric": "domain_filter_rule", "value": f"domain_keyword_count > 0 and noise_keyword_count <= {MAX_NOISE_CUES_FOR_DOMAIN_ROW}", "note": "Deterministic relevance rule used for Twitter, Reddit rows, and Reddit threads."},
    ])
    return summary, gold

print("Configuration complete.")
print(f"Base path: {BASE_PATH}")
print(f"Final output directory: {FINAL_OUTPUT_DIR}")
print(f"Observation window: {OBS_START.date()} to {(OBS_END_EXCLUSIVE - pd.Timedelta(days=1)).date()}")


In [ ]:
# ============================================================
# FRESH SESSION PATCH AFTER v5 00_config:
# use locked contrastive MiniLM Twitter inputs
# ============================================================

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

from pathlib import Path

BASE_PATH = Path("/content/drive/MyDrive/Dubai_Real_Estate_Data")
MINILM_OUT_DIR = BASE_PATH / "twitter_minilm_semantic_collection"

RAW_TWITTER_REGEX_PATH = MINILM_OUT_DIR / "twitter_strict_raw_date_filtered_for_v5.parquet"
RAW_TWITTER_SEMANTIC_PATH = MINILM_OUT_DIR / "twitter_minilm_kept_semantic_stream_flagged.parquet"
RAW_TWITTER_SEMANTIC_CANDIDATES = [RAW_TWITTER_SEMANTIC_PATH]

V5_REGEX_RAW_PATH = str(RAW_TWITTER_REGEX_PATH)
V5_SEMANTIC_RAW_PATH = str(RAW_TWITTER_SEMANTIC_PATH)

RUN_TWITTER_COLLECTION = False
FORCE_RECOLLECT_TWITTER = False
COLLECT_REGEX_STREAM = False
COLLECT_SEMANTIC_STREAM = False
USE_EMBEDDING_SEMANTIC_QUERY_EXPANSION = False
EXPORT_TO_V5_INPUT_PATHS = False

assert RAW_TWITTER_REGEX_PATH.exists(), f"Missing regex file: {RAW_TWITTER_REGEX_PATH}"
assert RAW_TWITTER_SEMANTIC_PATH.exists(), f"Missing semantic file: {RAW_TWITTER_SEMANTIC_PATH}"

print("v5 will load locked regex from:", RAW_TWITTER_REGEX_PATH)
print("v5 will load locked MiniLM semantic from:", RAW_TWITTER_SEMANTIC_PATH)
print("Confirmed: no rescrape, no overwrite.")

In [ ]:
# ============================================================
# 00_config_POST_PATCH_USE_MINILM_FILTERED_TWITTER
# ============================================================
# This is the key no-overwrite patch:
# v5 keeps the old raw files untouched, but final processing loads:
#   regex stream    -> raw_data_files/master_raw.parquet
#   semantic stream -> twitter_minilm_semantic_collection/twitter_minilm_kept_semantic_stream_flagged.parquet

from pathlib import Path

BASE_PATH = Path("/content/drive/MyDrive/Dubai_Real_Estate_Data")
MINILM_OUT_DIR = BASE_PATH / "twitter_minilm_semantic_collection"

RAW_TWITTER_REGEX_PATH = BASE_PATH / "raw_data_files" / "master_raw.parquet"
RAW_TWITTER_SEMANTIC_PATH = MINILM_OUT_DIR / "twitter_minilm_kept_semantic_stream_flagged.parquet"
MINILM_KEPT_TWITTER_PATH = RAW_TWITTER_SEMANTIC_PATH

RAW_TWITTER_SEMANTIC_CANDIDATES = [
    RAW_TWITTER_SEMANTIC_PATH
]

RUN_TWITTER_COLLECTION = False
FORCE_RECOLLECT_TWITTER = False
COLLECT_REGEX_STREAM = False
COLLECT_SEMANTIC_STREAM = False
USE_EMBEDDING_SEMANTIC_QUERY_EXPANSION = False

print("v5 will load regex from:", RAW_TWITTER_REGEX_PATH)
print("v5 will load MiniLM semantic from:", RAW_TWITTER_SEMANTIC_PATH)

assert RAW_TWITTER_REGEX_PATH.exists(), f"Missing regex file: {RAW_TWITTER_REGEX_PATH}"
assert RAW_TWITTER_SEMANTIC_PATH.exists(), (
    "Missing MiniLM semantic file. Run the MiniLM cells above through "
    "00K_LISTING_SPAM_FLAGS_FOR_TWITTER_RAW_STAGE first: "
    f"{RAW_TWITTER_SEMANTIC_PATH}"
)

print("Confirmed: v5 will use MiniLM-filtered semantic stream without overwriting old raw files.")



## 00b_literature_grounding_for_reviewer_fixes

This notebook documents the reviewer-response methods directly in the pipeline:

- **Query expansion** is documented as a retrieval strategy, not assumed to be a magic semantic model. Query expansion is an established information-retrieval idea for reducing vocabulary mismatch, but it must specify whether expansion is manual, relevance-feedback-based, embedding-based, or LLM-generated.
- **Semantic-only Twitter validation** is exported as an annotation sample. If two annotators complete the files, the notebook computes Cohen's kappa and precision/recall/F1 for the semantic-only retrieval subset.
- **Near-duplicate detection** is treated as an audit using TF--IDF character n-gram cosine nearest neighbours at a stated threshold. This gives a reportable duplicate rate without silently removing valid repeated Reddit discourse.
- **Ad/listing content** is handled separately from domain relevance. Real estate advertisements and genuine discourse both contain domain terms, so the pipeline counts listing-style cues such as direct contact prompts, viewing invitations, permit/reference numbers, and broker language.
- **DLD correlation** is optional and kept outside the main dataset-construction output. It aggregates social-media indicators and DLD transactions weekly, then reports contemporaneous and lead/lag correlations. This supports a later market-activity paper without turning the dataset paper into a prediction paper.


## 01a_embedding_based_semantic_query_expansion

This v5 section implements a reproducible semantic query expansion method for Twitter/X. It uses Sentence-BERT-style sentence embeddings (`sentence-transformers/all-MiniLM-L6-v2`) and cosine similarity to select candidate expansion phrases from an auditable candidate pool. The threshold is fixed in `00_config`.

Important: if `RUN_TWITTER_COLLECTION = False`, this section exports the expanded query manifest but does not change the already-archived Twitter semantic parquet. To claim that the final Twitter raw data was collected using embedding-based expansion, rerun collection with `RUN_TWITTER_COLLECTION = True` after this section succeeds.


In [ ]:

# ============================================================
# 01a_embedding_based_semantic_query_expansion
# ============================================================
# This section makes the Twitter/X semantic expansion strategy explicit and reproducible.
# It uses Sentence-BERT-style sentence embeddings and cosine similarity to select candidate
# expansion phrases for each retrieval stream. Query expansion is only used for live collection
# when RUN_TWITTER_COLLECTION = True. If collection is skipped, the notebook still exports the
# expansion specification and selected terms, but the existing archived semantic parquet remains
# whatever strategy was used when it was originally collected.

SEMANTIC_EXPANSION_SEEDS = {
    "macro_investment": [
        "dubai real estate investment", "buy property in dubai", "dubai off plan property",
        "dubai property market", "dubai property prices", "dubai mortgage property"
    ],
    "rental_painpoints": [
        "dubai rent increase", "dubai tenant landlord", "ejari rental dispute",
        "rental renewal dubai", "service charges dubai", "eviction notice dubai"
    ],
    "developer_activity": [
        "dubai property developer", "emaar dubai property", "damac dubai property",
        "nakheel dubai projects", "sobha dubai real estate", "developer handover dubai"
    ],
    "neighborhood_retail": [
        "dubai marina apartment", "jvc rent dubai", "business bay apartment",
        "downtown dubai property", "dubai hills villa", "palm jumeirah real estate"
    ],
}

# Candidate pool. These are intentionally auditable terms/phrases, not hidden LLM outputs.
# The embedding model only ranks/selects them by similarity to stream seed phrases.
SEMANTIC_EXPANSION_CANDIDATE_TERMS = [
    "dubai housing market", "dubai property prices", "dubai realty", "uae real estate",
    "home buying dubai", "buying flat in dubai", "property buyers dubai", "first time buyer dubai",
    "property investment uae", "real estate investment dubai", "investment property dubai",
    "off plan investment", "off plan launch", "off plan project", "handover dubai property",
    "property handover", "capital appreciation dubai", "rental yield dubai", "roi dubai property",
    "property valuation dubai", "asking price dubai", "sale price dubai", "transaction price dubai",
    "mortgage dubai", "home loan dubai", "down payment dubai property", "service charges dubai",
    "rent hike dubai", "rent increase notice", "rental index dubai", "rera rental index",
    "ejari renewal", "tenant rights dubai", "landlord dispute dubai", "rental dispute dubai",
    "eviction notice dubai", "security deposit dubai", "lease renewal dubai", "rent cheque dubai",
    "rent affordability dubai", "moving apartments dubai", "apartment renewal dubai",
    "emaar project", "damac project", "nakheel project", "sobha project", "danube property",
    "binghatti project", "azizi development", "omniyat project", "aldar dubai", "developer launch",
    "new launch dubai", "project handover", "delayed handover", "payment plan dubai",
    "construction update dubai", "master developer dubai", "rera approved project",
    "studio apartment dubai", "one bedroom dubai", "two bedroom apartment dubai", "villa community dubai",
    "townhouse dubai", "penthouse dubai", "branded residences dubai", "serviced apartment dubai",
    "dubai marina rent", "jvc apartment", "business bay rent", "downtown dubai apartment",
    "dubai hills estate", "palm jumeirah villa", "dubai creek harbour", "arabian ranches villa",
    "motor city apartment", "dubai south property", "mirdif villa", "dubai silicon oasis rent",
    "international city rent", "discovery gardens rent", "jlt apartment", "al furjan property",
    "town square dubai", "damac hills villa", "the greens dubai", "the springs dubai",
]


def install_sentence_transformers_if_needed():
    if not USE_EMBEDDING_SEMANTIC_QUERY_EXPANSION:
        return False
    if importlib.util.find_spec("sentence_transformers") is None:
        if INSTALL_SENTENCE_TRANSFORMERS_IF_NEEDED:
            print("Installing sentence-transformers for embedding-based query expansion...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers"])
        else:
            raise ImportError("sentence-transformers is required for embedding-based query expansion.")
    return True


def build_embedding_semantic_expansion():
    """Return selected expansion terms and chunked Twitter/X query bases."""
    if not USE_EMBEDDING_SEMANTIC_QUERY_EXPANSION:
        empty = pd.DataFrame(columns=[
            "retrieval_stream", "candidate_term", "best_seed", "cosine_similarity", "selected"
        ])
        return empty, empty, {}

    try:
        install_sentence_transformers_if_needed()
        from sentence_transformers import SentenceTransformer
        from sklearn.metrics.pairwise import cosine_similarity

        model = SentenceTransformer(SEMANTIC_QUERY_EMBEDDING_MODEL)
        candidate_terms = sorted(set(str(t).strip() for t in SEMANTIC_EXPANSION_CANDIDATE_TERMS if str(t).strip()))
        candidate_emb = model.encode(candidate_terms, normalize_embeddings=True, show_progress_bar=False)

        all_rows = []
        selected_rows = []
        expanded_query_bases = {}

        for stream_name, seeds in SEMANTIC_EXPANSION_SEEDS.items():
            seeds = [str(s).strip() for s in seeds if str(s).strip()]
            seed_emb = model.encode(seeds, normalize_embeddings=True, show_progress_bar=False)
            sims = cosine_similarity(candidate_emb, seed_emb)

            rows = []
            for i, term in enumerate(candidate_terms):
                best_idx = int(np.argmax(sims[i]))
                best_score = float(sims[i][best_idx])
                rows.append({
                    "retrieval_stream": stream_name,
                    "candidate_term": term,
                    "best_seed": seeds[best_idx],
                    "cosine_similarity": round(best_score, 4),
                    "selected": best_score >= float(SEMANTIC_QUERY_SIMILARITY_THRESHOLD),
                    "embedding_model": SEMANTIC_QUERY_EMBEDDING_MODEL,
                    "similarity_threshold": SEMANTIC_QUERY_SIMILARITY_THRESHOLD,
                })

            stream_candidates = pd.DataFrame(rows).sort_values("cosine_similarity", ascending=False)
            stream_selected = stream_candidates[stream_candidates["selected"]].head(SEMANTIC_QUERY_MAX_SELECTED_TERMS_PER_STREAM).copy()
            all_rows.append(stream_candidates)
            selected_rows.append(stream_selected)

            selected_terms = stream_selected["candidate_term"].tolist()
            chunks = [selected_terms[i:i + SEMANTIC_QUERY_TERMS_PER_TWITTER_QUERY]
                      for i in range(0, len(selected_terms), SEMANTIC_QUERY_TERMS_PER_TWITTER_QUERY)]
            for j, chunk in enumerate(chunks, start=1):
                if not chunk:
                    continue
                quoted = " OR ".join([f'"{t}"' if " " in t else t for t in chunk])
                # Guard with Dubai/UAE/property context to reduce semantic topic drift.
                query_base = f'(({quoted}) AND (dubai OR uae OR property OR rent OR apartment OR villa OR rera OR ejari))'
                expanded_query_bases[(stream_name, f"sbert_expansion_{j:02d}")] = query_base

        candidates_df = pd.concat(all_rows, ignore_index=True) if all_rows else pd.DataFrame()
        selected_df = pd.concat(selected_rows, ignore_index=True) if selected_rows else pd.DataFrame()
        return candidates_df, selected_df, expanded_query_bases

    except Exception as e:
        print(f"Embedding-based semantic query expansion could not be built: {e}")
        print("Falling back to manual semantic query manifest only. Do not claim embedding-based collection unless this section runs successfully and collection is rerun.")
        empty = pd.DataFrame([{
            "retrieval_stream": "all",
            "candidate_term": pd.NA,
            "best_seed": pd.NA,
            "cosine_similarity": pd.NA,
            "selected": False,
            "embedding_model": SEMANTIC_QUERY_EMBEDDING_MODEL,
            "similarity_threshold": SEMANTIC_QUERY_SIMILARITY_THRESHOLD,
            "error": str(e),
        }])
        return empty, empty.iloc[0:0].copy(), {}


semantic_expansion_candidates, semantic_expansion_selected, TWITTER_EMBEDDING_SEMANTIC_SUBQUERIES = build_embedding_semantic_expansion()

# Export immediately so the method is documented even if collection mode is off.
semantic_expansion_candidates.to_csv(SEMANTIC_QUERY_EXPANSION_OUTPUT_DIR / "semantic_query_expansion_candidates.csv", index=False)
semantic_expansion_selected.to_csv(SEMANTIC_QUERY_EXPANSION_OUTPUT_DIR / "semantic_query_expansion_selected.csv", index=False)
semantic_expansion_selected.to_csv(TABLE_OUTPUT_DIR / "semantic_query_expansion_selected.csv", index=False)
semantic_expansion_candidates.to_csv(TABLE_OUTPUT_DIR / "semantic_query_expansion_candidates.csv", index=False)

paper_tables["semantic_query_expansion_candidates"] = semantic_expansion_candidates
paper_tables["semantic_query_expansion_selected"] = semantic_expansion_selected

print("Embedding-based semantic query expansion specification:")
print(f"  Enabled: {USE_EMBEDDING_SEMANTIC_QUERY_EXPANSION}")
print(f"  Model: {SEMANTIC_QUERY_EMBEDDING_MODEL}")
print(f"  Similarity threshold: {SEMANTIC_QUERY_SIMILARITY_THRESHOLD}")
print(f"  Selected terms: {len(semantic_expansion_selected):,}")
print(f"  Generated query chunks: {len(TWITTER_EMBEDDING_SEMANTIC_SUBQUERIES):,}")
if len(semantic_expansion_selected):
    display(semantic_expansion_selected.head(20))


## 01_collect_or_load_twitter

In [ ]:
# ============================================================
# 01_collect_or_load_twitter
# ============================================================

# This section makes the Twitter/X part paper-defensible in two modes:
#   1. Collection mode: RUN_TWITTER_COLLECTION = True
#      - runs twscrape over the auditable regex + semantic query manifest
#      - writes raw weekly parquet checkpoints
#      - consolidates them into the expected master parquet files
#   2. Preprocessing mode: RUN_TWITTER_COLLECTION = False
#      - loads the already-saved twscrape outputs from Drive
#      - still exports the query manifest and regex/semantic comparison tables
#
# Credentials are intentionally NOT stored in this notebook.
# If collection mode is enabled, configure twscrape accounts separately using either:
#   A. a pre-existing twscrape accounts DB at TWSCRAPE_DB_PATH, or
#   B. a private CSV at TWSCRAPE_ACCOUNT_CSV with columns:
#      username,password,email,email_password,cookies

import asyncio
import time


def build_twitter_query_manifest():
    """Create a row-level manifest for all Twitter/X collection jobs."""
    rows = []

    def add_rows(strategy, stream_name, subquery_name, query_base, output_dir, master_output_path):
        for cohort in TWITTER_COHORTS:
            full_query = (
                f'{query_base} lang:en '
                f'since:{cohort["start"]} until:{cohort["end"]} '
                f'{TWITTER_QUERY_EXCLUSION_CUES}'
            )
            rows.append({
                "retrieval_strategy": strategy,
                "retrieval_stream": stream_name,
                "subquery_name": subquery_name,
                "cohort_label": cohort["label"],
                "cohort_start": cohort["start"],
                "cohort_end_exclusive": cohort["end"],
                "query_base": query_base,
                "query_used": full_query,
                "query_length": len(full_query),
                "within_512_chars": len(full_query) <= 512,
                "checkpoint_path": str(Path(output_dir) / f"raw_{cohort['label']}_{stream_name}_{subquery_name}.parquet"),
                "master_output_path": str(master_output_path),
            })

    for stream_name, query_base in TWITTER_REGEX_STREAMS.items():
        add_rows(
            strategy="regex",
            stream_name=stream_name,
            subquery_name="main",
            query_base=query_base,
            output_dir=TWITTER_REGEX_RAW_DIR,
            master_output_path=RAW_TWITTER_REGEX_PATH,
        )

    for stream_name, query_base in TWITTER_SEMANTIC_STANDARD_STREAMS.items():
        add_rows(
            strategy="semantic",
            stream_name=stream_name,
            subquery_name="main",
            query_base=query_base,
            output_dir=TWITTER_SEMANTIC_RAW_DIR,
            master_output_path=RAW_TWITTER_SEMANTIC_PATH,
        )

    for subquery_name, query_base in TWITTER_SEMANTIC_RENTAL_SUBQUERIES.items():
        add_rows(
            strategy="semantic",
            stream_name="rental_painpoints",
            subquery_name=subquery_name,
            query_base=query_base,
            output_dir=TWITTER_SEMANTIC_RAW_DIR,
            master_output_path=RAW_TWITTER_SEMANTIC_PATH,
        )

    for subquery_name, query_base in TWITTER_SEMANTIC_NEIGHBORHOOD_SUBQUERIES.items():
        add_rows(
            strategy="semantic",
            stream_name="neighborhood_retail",
            subquery_name=subquery_name,
            query_base=query_base,
            output_dir=TWITTER_SEMANTIC_RAW_DIR,
            master_output_path=RAW_TWITTER_SEMANTIC_PATH,
        )

    # v5: embedding-based semantic expansion jobs. These are only collected when
    # RUN_TWITTER_COLLECTION = True. If collection is skipped, they still document the
    # reproducible expansion strategy but do not change the archived raw semantic parquet.
    for (stream_name, subquery_name), query_base in TWITTER_EMBEDDING_SEMANTIC_SUBQUERIES.items():
        add_rows(
            strategy="semantic_embedding",
            stream_name=stream_name,
            subquery_name=subquery_name,
            query_base=query_base,
            output_dir=TWITTER_SEMANTIC_RAW_DIR,
            master_output_path=RAW_TWITTER_SEMANTIC_PATH,
        )

    manifest = pd.DataFrame(rows)
    manifest = manifest.sort_values([
        "retrieval_strategy", "cohort_label", "retrieval_stream", "subquery_name"
    ]).reset_index(drop=True)
    return manifest


twitter_query_manifest = build_twitter_query_manifest()
paper_tables["twitter_query_manifest"] = twitter_query_manifest

print("Twitter/X query manifest created.")
print(f"  Jobs: {len(twitter_query_manifest):,}")
print("  By strategy:")
display(twitter_query_manifest.groupby("retrieval_strategy").size().reset_index(name="collection_jobs"))
print("  Query length check:")
display(
    twitter_query_manifest
    .groupby(["retrieval_strategy", "retrieval_stream", "subquery_name"], dropna=False)
    .agg(max_query_length=("query_length", "max"), all_within_512=("within_512_chars", "all"))
    .reset_index()
)


def install_twscrape_if_needed():
    if not RUN_TWITTER_COLLECTION:
        return
    missing = []
    if importlib.util.find_spec("twscrape") is None:
        missing.append("twscrape")
    if importlib.util.find_spec("nest_asyncio") is None:
        missing.append("nest_asyncio")
    if missing and INSTALL_TWSCRAPE_IF_COLLECTION_ENABLED:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    elif missing:
        raise ImportError(f"Missing optional collection packages: {missing}")


def make_tweet_row(tweet, strategy, stream_name, subquery_name, cohort_label, query, collected_at):
    """Convert a twscrape tweet object into the raw schema used downstream."""
    return {
        "text": getattr(tweet, "rawContent", None),
        "date": getattr(tweet, "date", None),
        "id": str(getattr(tweet, "id", "")),
        "url": f"https://x.com/i/web/status/{getattr(tweet, 'id', '')}",
        "lang": getattr(tweet, "lang", None),
        "is_retweet": getattr(tweet, "retweetedTweet", None) is not None,
        "is_reply": getattr(tweet, "inReplyToTweetId", None) is not None,
        "likeCount": getattr(tweet, "likeCount", 0),
        "retweetCount": getattr(tweet, "retweetCount", 0),
        "replyCount": getattr(tweet, "replyCount", 0),
        "quoteCount": getattr(tweet, "quoteCount", 0),
        "engagement_score": (
            getattr(tweet, "likeCount", 0) * 1 +
            getattr(tweet, "replyCount", 0) * 2 +
            getattr(tweet, "retweetCount", 0) * 3 +
            getattr(tweet, "quoteCount", 0) * 2
        ),
        "author_id": str(getattr(getattr(tweet, "user", None), "id", "")) if getattr(tweet, "user", None) else None,
        "username": getattr(getattr(tweet, "user", None), "username", None) if getattr(tweet, "user", None) else None,
        "retrieval_strategy": strategy,
        "retrieval_stream": stream_name,
        "subquery_name": subquery_name,
        "extraction_cohort": cohort_label,
        "query_used": query,
        "collected_at": collected_at,
    }


async def configure_twscrape_api():
    """Initialise twscrape API without exposing credentials in the notebook."""
    from twscrape import API

    TWSCRAPE_DB_PATH.parent.mkdir(parents=True, exist_ok=True)
    try:
        api = API(str(TWSCRAPE_DB_PATH))
    except TypeError:
        # Older twscrape versions may not accept a DB path in the constructor.
        api = API()

    # Optional private account CSV; do not commit/share this file.
    if TWSCRAPE_ACCOUNT_CSV.exists():
        accounts = pd.read_csv(TWSCRAPE_ACCOUNT_CSV)
        required_cols = {"username", "password", "email"}
        missing_cols = required_cols - set(accounts.columns)
        if missing_cols:
            raise ValueError(f"TWSCRAPE_ACCOUNT_CSV is missing columns: {sorted(missing_cols)}")
        for _, acc in accounts.iterrows():
            try:
                await api.pool.add_account(
                    username=str(acc.get("username", "")),
                    password=str(acc.get("password", "")),
                    email=str(acc.get("email", "")),
                    email_password=str(acc.get("email_password", "")) if pd.notna(acc.get("email_password", "")) else "",
                    cookies=str(acc.get("cookies", "")) if pd.notna(acc.get("cookies", "")) else None,
                )
            except Exception as e:
                print(f"Account add skipped/failed for {acc.get('username', '<unknown>')}: {e}")

    try:
        accounts_loaded = await api.pool.get_all()
    except Exception:
        accounts_loaded = []

    if not accounts_loaded:
        raise RuntimeError(
            "No twscrape accounts are configured. Either create a private account CSV at "
            f"{TWSCRAPE_ACCOUNT_CSV}, or configure the twscrape account pool/DB at {TWSCRAPE_DB_PATH}. "
            "For normal reproduction, keep RUN_TWITTER_COLLECTION = False and load existing parquet files."
        )

    print(f"twscrape accounts available: {len(accounts_loaded)}")
    return api


async def scrape_manifest_row(api, row):
    """Run one query-manifest row and save a checkpoint parquet."""
    checkpoint_path = Path(row["checkpoint_path"])
    checkpoint_path.parent.mkdir(parents=True, exist_ok=True)

    if checkpoint_path.exists() and not FORCE_RECOLLECT_TWITTER:
        return {
            "retrieval_strategy": row["retrieval_strategy"],
            "retrieval_stream": row["retrieval_stream"],
            "subquery_name": row["subquery_name"],
            "cohort_label": row["cohort_label"],
            "checkpoint_path": str(checkpoint_path),
            "status": "checkpoint_exists",
            "rows_collected": len(pd.read_parquet(checkpoint_path)),
            "cap_hit": False,
        }

    tweets = []
    collected_at = pd.Timestamp.utcnow()
    try:
        async for tweet in api.search(row["query_used"], limit=TWSCRAPE_MAX_TWEETS_PER_QUERY):
            tweets.append(make_tweet_row(
                tweet=tweet,
                strategy=row["retrieval_strategy"],
                stream_name=row["retrieval_stream"],
                subquery_name=row["subquery_name"],
                cohort_label=row["cohort_label"],
                query=row["query_used"],
                collected_at=collected_at,
            ))
    except Exception as e:
        return {
            "retrieval_strategy": row["retrieval_strategy"],
            "retrieval_stream": row["retrieval_stream"],
            "subquery_name": row["subquery_name"],
            "cohort_label": row["cohort_label"],
            "checkpoint_path": str(checkpoint_path),
            "status": f"error: {e}",
            "rows_collected": 0,
            "cap_hit": False,
        }

    df = pd.DataFrame(tweets)
    if not df.empty:
        # De-duplicate inside a single checkpoint if subqueries accidentally overlap.
        df = (
            df.sort_values("engagement_score", ascending=False)
            .drop_duplicates(subset=["id"], keep="first")
            .reset_index(drop=True)
        )
        df.to_parquet(checkpoint_path, engine="pyarrow", index=False)
    else:
        # Save an empty but schema-visible checkpoint for reproducibility.
        pd.DataFrame(columns=[
            "text", "date", "id", "url", "lang", "is_retweet", "is_reply",
            "likeCount", "retweetCount", "replyCount", "quoteCount", "engagement_score",
            "author_id", "username", "retrieval_strategy", "retrieval_stream", "subquery_name",
            "extraction_cohort", "query_used", "collected_at"
        ]).to_parquet(checkpoint_path, engine="pyarrow", index=False)

    return {
        "retrieval_strategy": row["retrieval_strategy"],
        "retrieval_stream": row["retrieval_stream"],
        "subquery_name": row["subquery_name"],
        "cohort_label": row["cohort_label"],
        "checkpoint_path": str(checkpoint_path),
        "status": "collected",
        "rows_collected": len(df),
        "cap_hit": len(df) >= TWSCRAPE_MAX_TWEETS_PER_QUERY,
    }


def assign_collection_provenance(master_df):
    """Store all conceptual retrieval streams/subqueries that retrieved each tweet."""
    master_df = master_df.copy()
    if master_df.empty:
        master_df["retrieval_streams"] = pd.NA
        master_df["subquery_names"] = pd.NA
        return master_df

    stream_prov = (
        master_df.groupby("id")["retrieval_stream"]
        .apply(lambda s: "|".join(sorted(set(s.dropna().astype(str)))))
        .reset_index(name="retrieval_streams")
    )
    subquery_prov = (
        master_df.groupby("id")["subquery_name"]
        .apply(lambda s: "|".join(sorted(set(s.dropna().astype(str)))))
        .reset_index(name="subquery_names")
    )
    master_df = master_df.merge(stream_prov, on="id", how="left").merge(subquery_prov, on="id", how="left")
    return master_df


def consolidate_twitter_strategy(raw_dir, master_output_path, strategy):
    """Combine weekly checkpoint parquet files into one raw master parquet/csv."""
    raw_dir = Path(raw_dir)
    master_output_path = Path(master_output_path)
    files = sorted(raw_dir.glob("raw_*.parquet"))
    if not files:
        raise FileNotFoundError(f"No checkpoint parquet files found for {strategy} in {raw_dir}")

    frames = [pd.read_parquet(f) for f in files]
    frames = [f for f in frames if not f.empty]
    if not frames:
        master_df = pd.DataFrame()
    else:
        master_df = pd.concat(frames, ignore_index=True)
        master_df = assign_collection_provenance(master_df)
        if "engagement_score" not in master_df.columns:
            master_df = compute_engagement_score(master_df)
        master_df = (
            master_df.sort_values("engagement_score", ascending=False)
            .drop_duplicates(subset=["id"], keep="first")
            .reset_index(drop=True)
        )

    master_output_path.parent.mkdir(parents=True, exist_ok=True)
    master_df.to_parquet(master_output_path, engine="pyarrow", index=False)

    # Keep a compatibility file name for older semantic notebook conventions.
    if strategy == "semantic":
        compatibility_path = master_output_path.parent / "master_raw.parquet"
        master_df.to_parquet(compatibility_path, engine="pyarrow", index=False)

    master_df.to_csv(master_output_path.with_suffix(".csv"), index=False, encoding="utf-8")
    return {
        "retrieval_strategy": strategy,
        "raw_dir": str(raw_dir),
        "master_output_path": str(master_output_path),
        "checkpoint_files": len(files),
        "rows_after_strategy_dedup": len(master_df),
        "unique_ids_after_strategy_dedup": int(master_df["id"].nunique()) if "id" in master_df.columns else 0,
    }


async def collect_twitter_with_twscrape():
    install_twscrape_if_needed()
    api = await configure_twscrape_api()

    telemetry = []
    for _, row in twitter_query_manifest.iterrows():
        print(
            f"Collecting {row['retrieval_strategy']} / {row['cohort_label']} / "
            f"{row['retrieval_stream']} / {row['subquery_name']}"
        )
        result = await scrape_manifest_row(api, row)
        telemetry.append(result)
        time.sleep(TWSCRAPE_BASE_COOLDOWN_SECONDS)

    twitter_collection_telemetry = pd.DataFrame(telemetry)
    strategy_summaries = pd.DataFrame([
        consolidate_twitter_strategy(TWITTER_REGEX_RAW_DIR, RAW_TWITTER_REGEX_PATH, "regex"),
        consolidate_twitter_strategy(TWITTER_SEMANTIC_RAW_DIR, RAW_TWITTER_SEMANTIC_PATH, "semantic"),
    ])

    twitter_collection_telemetry.to_csv(FINAL_OUTPUT_DIR / "twitter_collection_telemetry.csv", index=False)
    strategy_summaries.to_csv(FINAL_OUTPUT_DIR / "twitter_collection_strategy_outputs.csv", index=False)
    return twitter_collection_telemetry, strategy_summaries


def run_async(coro):
    """Run async code safely in notebooks and scripts."""
    try:
        loop = asyncio.get_event_loop()
        if loop.is_running():
            import nest_asyncio
            nest_asyncio.apply()
        return loop.run_until_complete(coro)
    except RuntimeError:
        return asyncio.run(coro)


if RUN_TWITTER_COLLECTION:
    twitter_collection_telemetry, twitter_collection_strategy_outputs = run_async(collect_twitter_with_twscrape())
else:
    twitter_collection_telemetry = pd.DataFrame(columns=[
        "retrieval_strategy", "retrieval_stream", "subquery_name", "cohort_label",
        "checkpoint_path", "status", "rows_collected", "cap_hit"
    ])
    twitter_collection_strategy_outputs = pd.DataFrame([
        {
            "retrieval_strategy": "regex",
            "raw_dir": str(TWITTER_REGEX_RAW_DIR),
            "master_output_path": str(RAW_TWITTER_REGEX_PATH),
            "collection_mode": "skipped_loaded_existing_raw_files",
        },
        {
            "retrieval_strategy": "semantic",
            "raw_dir": str(TWITTER_SEMANTIC_RAW_DIR),
            "master_output_path": str(RAW_TWITTER_SEMANTIC_PATH),
            "collection_mode": "skipped_loaded_existing_raw_files",
        },
    ])
    print("RUN_TWITTER_COLLECTION = False. Skipping live twscrape collection and loading existing raw parquet files.")

paper_tables["twitter_collection_telemetry"] = twitter_collection_telemetry
paper_tables["twitter_collection_strategy_outputs"] = twitter_collection_strategy_outputs


def standardise_twitter_raw(df, source_name):
    df = df.copy()

    # Flexible text field handling across notebook variants.
    text_candidates = ["text", "rawContent", "raw_text", "content"]
    text_col = next((c for c in text_candidates if c in df.columns), None)
    if text_col is None:
        raise ValueError(f"Could not find a Twitter text column in {source_name}. Columns: {df.columns.tolist()}")
    if text_col != "text":
        df["text"] = df[text_col]

    if "id" not in df.columns:
        raise ValueError(f"Twitter {source_name} dataset has no id column.")
    df["id"] = df["id"].astype(str)

    if "date" not in df.columns:
        raise ValueError(f"Twitter {source_name} dataset has no date column.")
    df["date"] = safe_to_datetime(df["date"])

    for col in ["likeCount", "retweetCount", "replyCount", "quoteCount"]:
        if col not in df.columns:
            df[col] = 0
    df = compute_engagement_score(df)

    for col in [
        "lang", "is_retweet", "is_reply", "author_id", "username", "url",
        "retrieval_strategy", "retrieval_stream", "retrieval_streams", "subquery_name",
        "subquery_names", "extraction_cohort", "query_used", "collected_at",
    ]:
        if col not in df.columns:
            df[col] = pd.NA

    df["_collection_source"] = source_name
    return df


SEMANTIC_TWITTER_PATH = first_existing_path(RAW_TWITTER_SEMANTIC_CANDIDATES, "semantic Twitter raw")

regex_raw_all = standardise_twitter_raw(read_table(RAW_TWITTER_REGEX_PATH), "regex")
semantic_raw_all = standardise_twitter_raw(read_table(SEMANTIC_TWITTER_PATH), "semantic")

# Keep raw loaded stats before observation-window filtering.
twitter_input_inventory = pd.DataFrame([
    {"dataset": "regex", "path": str(RAW_TWITTER_REGEX_PATH), "rows_loaded": len(regex_raw_all), "unique_ids_loaded": regex_raw_all["id"].nunique(), "min_date_loaded": regex_raw_all["date"].min(), "max_date_loaded": regex_raw_all["date"].max()},
    {"dataset": "semantic", "path": str(SEMANTIC_TWITTER_PATH), "rows_loaded": len(semantic_raw_all), "unique_ids_loaded": semantic_raw_all["id"].nunique(), "min_date_loaded": semantic_raw_all["date"].min(), "max_date_loaded": semantic_raw_all["date"].max()},
])

regex_raw = filter_observation_window(regex_raw_all)
semantic_raw = filter_observation_window(semantic_raw_all)

regex_ids = set(regex_raw["id"])
semantic_ids = set(semantic_raw["id"])
shared_ids = regex_ids & semantic_ids
regex_only_ids = regex_ids - semantic_ids
semantic_only_ids = semantic_ids - regex_ids

# Merge and label by true collection provenance.
twitter_raw = pd.concat([regex_raw, semantic_raw], ignore_index=True)
twitter_raw["source_label"] = np.select(
    [twitter_raw["id"].isin(shared_ids), twitter_raw["id"].isin(regex_only_ids), twitter_raw["id"].isin(semantic_only_ids)],
    ["both", "regex", "semantic"],
    default=twitter_raw["_collection_source"],
)

# Deduplicate by tweet ID, keeping the highest-engagement copy while preserving source_label.
twitter_raw = (
    twitter_raw
    .sort_values("engagement_score", ascending=False)
    .drop_duplicates(subset=["id"], keep="first")
    .reset_index(drop=True)
)

twitter_raw["platform"] = "twitter"
twitter_raw["post_type"] = "tweet"
twitter_raw["raw_text"] = twitter_raw["text"].astype(str)

twitter_collection_strategy_comparison = pd.DataFrame([
    {"metric": "regex_rows_in_window", "value": len(regex_raw)},
    {"metric": "semantic_rows_in_window", "value": len(semantic_raw)},
    {"metric": "regex_unique_ids_in_window", "value": len(regex_ids)},
    {"metric": "semantic_unique_ids_in_window", "value": len(semantic_ids)},
    {"metric": "shared_unique_ids_in_window", "value": len(shared_ids)},
    {"metric": "regex_only_unique_ids_in_window", "value": len(regex_only_ids)},
    {"metric": "semantic_only_unique_ids_in_window", "value": len(semantic_only_ids)},
    {"metric": "merged_unique_tweets_in_window", "value": len(twitter_raw)},
])

# Optional stream and source-label distributions for the paper appendix/methodology.
twitter_stream_distribution = (
    twitter_raw["retrieval_stream"]
    .fillna("unknown")
    .value_counts()
    .rename_axis("retrieval_stream")
    .reset_index(name="count")
)

twitter_source_label_distribution = (
    twitter_raw["source_label"]
    .fillna("unknown")
    .value_counts()
    .rename_axis("source_label")
    .reset_index(name="count")
)

paper_tables["twitter_input_inventory"] = twitter_input_inventory
paper_tables["twitter_collection_strategy_comparison"] = twitter_collection_strategy_comparison
paper_tables["twitter_stream_distribution"] = twitter_stream_distribution
paper_tables["twitter_source_label_distribution"] = twitter_source_label_distribution

log_retention("twitter", "00_loaded_raw_regex_semantic", len(twitter_raw), "Loaded regex and semantic twscrape outputs, filtered to observation window, and deduplicated by tweet ID.")

print("Twitter raw loaded and merged.")
display(twitter_input_inventory)
display(twitter_collection_strategy_comparison)


the below was run before collect or load twitter.

In [ ]:
# ============================================================
# PATCH: define missing optional query-manifest variables
# Needed because we are skipping old embedding semantic expansion
# ============================================================

# We already completed contrastive MiniLM filtering separately.
# So the old v5 embedding semantic query expansion is intentionally disabled.
if "TWITTER_EMBEDDING_SEMANTIC_SUBQUERIES" not in globals():
    TWITTER_EMBEDDING_SEMANTIC_SUBQUERIES = {}

if "TWITTER_EMBEDDING_SEMANTIC_REFERENCE_TERMS" not in globals():
    TWITTER_EMBEDDING_SEMANTIC_REFERENCE_TERMS = []

if "USE_EMBEDDING_SEMANTIC_QUERY_EXPANSION" not in globals():
    USE_EMBEDDING_SEMANTIC_QUERY_EXPANSION = False

print("Patched optional semantic expansion variables.")
print("TWITTER_EMBEDDING_SEMANTIC_SUBQUERIES:", TWITTER_EMBEDDING_SEMANTIC_SUBQUERIES)
print("USE_EMBEDDING_SEMANTIC_QUERY_EXPANSION:", USE_EMBEDDING_SEMANTIC_QUERY_EXPANSION)

In [ ]:
print("twitter_raw:", twitter_raw.shape)
print("date:", twitter_raw["date"].min(), "to", twitter_raw["date"].max())
print(twitter_raw["source_label"].value_counts(dropna=False))
print([c for c in twitter_raw.columns if "minilm" in c.lower()])

## 02_clean_twitter

In [ ]:
# ============================================================
# 02_clean_twitter
# ============================================================

twitter_clean = twitter_raw.copy()

# Text normalisation.
twitter_clean["raw_text"] = twitter_clean["raw_text"].fillna("").astype(str)
twitter_clean["clean_text"] = twitter_clean["raw_text"].apply(normalise_text)
twitter_clean = twitter_clean[twitter_clean["clean_text"].str.strip().ne("")].copy()
log_retention("twitter", "01_non_empty_text", len(twitter_clean), "Removed empty text after URL/mention/hashtag cleanup.")

# Language filtering: prefer Twitter metadata, fallback to detection only if needed.
twitter_clean = twitter_clean[language_mask(
    twitter_clean,
    text_col="raw_text",
    lang_col="lang",
    fallback_detect=APPLY_LANGDETECT_IF_NO_LANG_COLUMN,
)].copy()
log_retention("twitter", "02_language_filtered", len(twitter_clean), "Kept English rows using lang metadata and optional fallback detection.")

# Text-level deduplication after normalisation.
twitter_clean["_clean_text_key"] = twitter_clean["clean_text"].str.lower().str.strip()
twitter_clean = (
    twitter_clean
    .sort_values("engagement_score", ascending=False)
    .drop_duplicates(subset=["_clean_text_key"], keep="first")
    .drop(columns=["_clean_text_key"], errors="ignore")
    .reset_index(drop=True)
)
log_retention("twitter", "03_duplicates_removed", len(twitter_clean), "Removed duplicated normalised text.")

# Minimum text validity: keep short posts when they contain a clear domain term.
def keep_by_length_or_domain(text):
    if token_count(text) >= MIN_TOKEN_COUNT:
        return True
    return len(matched_keywords(text, DOMAIN_KEYWORDS)) > 0

twitter_clean = twitter_clean[twitter_clean["clean_text"].apply(keep_by_length_or_domain)].copy()
log_retention("twitter", "04_length_filtered", len(twitter_clean), f"Kept rows with >= {MIN_TOKEN_COUNT} tokens or a domain keyword.")

# Spam/noise filter before domain relevance.
def twitter_is_spam(row):
    username = row.get("username", "")
    text = row.get("clean_text", "")
    username_is_bot = bool(BOT_USERNAME_PATTERN.search(str(username))) if pd.notna(username) else False
    noise_count = count_noise_cues(text)
    domain_count = len(matched_keywords(text, DOMAIN_KEYWORDS))
    spam_without_domain = noise_count >= 2 and domain_count == 0
    return username_is_bot or spam_without_domain

twitter_clean["_is_spam_or_bot"] = twitter_clean.apply(twitter_is_spam, axis=1)
twitter_clean = twitter_clean[~twitter_clean["_is_spam_or_bot"]].drop(columns=["_is_spam_or_bot"], errors="ignore").copy()
log_retention("twitter", "05_spam_noise_removed", len(twitter_clean), "Removed bot-like usernames and high-noise non-domain rows.")

# Context text for domain filtering.
twitter_clean["domain_context_text"] = twitter_clean["clean_text"]

twitter_clean = add_time_features(twitter_clean)

print(f"Twitter cleaned before domain filter: {len(twitter_clean):,}")
print("Date coverage:", twitter_clean["date"].min(), "to", twitter_clean["date"].max())
display(twitter_clean[["id", "date", "source_label", "post_type", "clean_text"]].head())

In [ ]:
print("twitter_clean:", twitter_clean.shape)
print("date:", twitter_clean["date"].min(), "to", twitter_clean["date"].max())
print(twitter_clean["source_label"].value_counts(dropna=False))
print([c for c in twitter_clean.columns if "minilm" in c.lower()])

display(
    twitter_clean[["date", "source_label", "clean_text"]]
    .sample(20, random_state=42)
)

In [ ]:
# ============================================================
# LOAD PRAJWAL REDDIT FILES WITH CONTEXT-AWARE COMMENT ROWS
# posts + comments become reddit_clean
# threads are kept separately as reddit_threads
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import re

REDDIT_DIR = Path("/content/drive/MyDrive/Dubai_Real_Estate_Data/Prajwal final reddit")

POSTS_PATH = REDDIT_DIR / "combined_reddit_posts.csv"
COMMENTS_PATH = REDDIT_DIR / "combined_reddit_comments.csv"
THREADS_PATH = REDDIT_DIR / "combined_reddit_threads.csv"

assert POSTS_PATH.exists(), f"Missing: {POSTS_PATH}"
assert COMMENTS_PATH.exists(), f"Missing: {COMMENTS_PATH}"
assert THREADS_PATH.exists(), f"Missing: {THREADS_PATH}"

reddit_posts_raw = pd.read_csv(POSTS_PATH, low_memory=False)
reddit_comments_raw = pd.read_csv(COMMENTS_PATH, low_memory=False)
reddit_threads = pd.read_csv(THREADS_PATH, low_memory=False)

print("Loaded:")
print("posts:", reddit_posts_raw.shape)
print("comments:", reddit_comments_raw.shape)
print("threads:", reddit_threads.shape)

print("\nPost columns:")
print(reddit_posts_raw.columns.tolist())

print("\nComment columns:")
print(reddit_comments_raw.columns.tolist())

print("\nThread columns:")
print(reddit_threads.columns.tolist())


# ----------------------------
# Helpers
# ----------------------------

OBS_START = pd.Timestamp("2026-01-01", tz="UTC")
OBS_END_EXCLUSIVE = pd.Timestamp("2026-05-01", tz="UTC")

def first_existing_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

def clean_id(x):
    if pd.isna(x):
        return pd.NA
    x = str(x)
    x = x.replace("t3_", "").replace("t1_", "")
    return x.strip()

def basic_clean_text(x):
    x = "" if pd.isna(x) else str(x)
    x = re.sub(r"http\S+|www\.\S+", " ", x)
    x = re.sub(r"\s+", " ", x)
    return x.strip()

def parse_reddit_date(df, label):
    date_col = first_existing_col(
        df,
        ["date", "created_datetime", "created_at", "timestamp", "created_utc", "created"]
    )
    if date_col is None:
        raise ValueError(f"{label}: no date column found. Columns: {df.columns.tolist()}")

    maybe_num = pd.to_numeric(df[date_col], errors="coerce")

    if maybe_num.notna().mean() > 0.8 and date_col in ["created_utc", "created", "timestamp"]:
        return pd.to_datetime(maybe_num, unit="s", errors="coerce", utc=True)

    return pd.to_datetime(df[date_col], errors="coerce", utc=True)


# ============================================================
# 1) Standardise posts
# ============================================================

p = reddit_posts_raw.copy()

post_id_col = first_existing_col(p, ["id", "post_id", "submission_id", "reddit_id"])
post_title_col = first_existing_col(p, ["title", "post_title"])
post_body_col = first_existing_col(p, ["selftext", "body", "text", "post_text", "raw_text", "clean_text"])
post_author_col = first_existing_col(p, ["author", "username", "post_author"])
post_score_col = first_existing_col(p, ["score", "post_score", "upvotes", "engagement_score"])
post_url_col = first_existing_col(p, ["url", "permalink", "post_url"])

reddit_posts_clean = pd.DataFrame(index=p.index)

reddit_posts_clean["id"] = p[post_id_col].apply(clean_id) if post_id_col else ["reddit_post_" + str(i) for i in range(len(p))]
reddit_posts_clean["thread_id"] = reddit_posts_clean["id"]
reddit_posts_clean["link_id"] = reddit_posts_clean["thread_id"]
reddit_posts_clean["parent_id"] = pd.NA

reddit_posts_clean["date"] = parse_reddit_date(p, "posts")
reddit_posts_clean["platform"] = "reddit"
reddit_posts_clean["source_label"] = "reddit"
reddit_posts_clean["post_type"] = "reddit_post"

post_title = p[post_title_col].fillna("").astype(str) if post_title_col else ""
post_body = p[post_body_col].fillna("").astype(str) if post_body_col else ""

if post_title_col:
    reddit_posts_clean["raw_text"] = (post_title + " " + post_body).str.strip()
    reddit_posts_clean["parent_post_title"] = post_title
else:
    reddit_posts_clean["raw_text"] = post_body
    reddit_posts_clean["parent_post_title"] = ""

reddit_posts_clean["parent_post_text"] = post_body
reddit_posts_clean["clean_text"] = reddit_posts_clean["raw_text"].apply(basic_clean_text)

reddit_posts_clean["context_text"] = reddit_posts_clean["clean_text"]
reddit_posts_clean["domain_context_text"] = reddit_posts_clean["clean_text"]
reddit_posts_clean["analysis_text_for_filter"] = reddit_posts_clean["clean_text"]
reddit_posts_clean["context_available"] = True

reddit_posts_clean["username"] = p[post_author_col] if post_author_col else pd.NA
reddit_posts_clean["url"] = p[post_url_col] if post_url_col else pd.NA

if "subreddit" in p.columns:
    reddit_posts_clean["subreddit"] = p["subreddit"].fillna("unknown").astype(str)
else:
    reddit_posts_clean["subreddit"] = "unknown"

reddit_posts_clean["platform_interaction_value"] = pd.to_numeric(p[post_score_col], errors="coerce") if post_score_col else pd.NA
reddit_posts_clean["platform_interaction_definition"] = "Reddit native score for post/comment at collection time"

# Preserve useful classifier/provenance columns
for c in p.columns:
    c_low = c.lower()
    if "deberta" in c_low or "real_estate" in c_low or "confidence" in c_low or "prob" in c_low:
        reddit_posts_clean[c] = p[c]


# ============================================================
# 2) Build post-context lookup for comments
# ============================================================

post_context_lookup = reddit_posts_clean[
    [
        "thread_id",
        "parent_post_title",
        "parent_post_text",
        "clean_text",
        "subreddit",
    ]
].copy()

post_context_lookup = post_context_lookup.rename(columns={
    "clean_text": "parent_post_clean_text",
    "subreddit": "parent_post_subreddit",
})

post_context_lookup = post_context_lookup.drop_duplicates(subset=["thread_id"], keep="first")


# ============================================================
# 3) Optional thread fallback lookup
# ============================================================

t = reddit_threads.copy()

thread_id_col = first_existing_col(t, ["thread_id", "id", "post_id", "submission_id", "link_id"])
thread_text_col = first_existing_col(t, ["thread_text", "clean_thread_text", "text", "raw_text", "combined_text"])
thread_title_col = first_existing_col(t, ["title", "post_title", "thread_title"])

thread_fallback_lookup = pd.DataFrame()

if thread_id_col:
    thread_fallback_lookup["thread_id"] = t[thread_id_col].apply(clean_id)

    if thread_title_col:
        thread_title = t[thread_title_col].fillna("").astype(str)
    else:
        thread_title = ""

    if thread_text_col:
        thread_text = t[thread_text_col].fillna("").astype(str)
    else:
        thread_text = ""

    if thread_title_col:
        thread_fallback_lookup["thread_fallback_text"] = (thread_title + " " + thread_text).str.strip().apply(basic_clean_text)
    else:
        thread_fallback_lookup["thread_fallback_text"] = thread_text.apply(basic_clean_text)

    thread_fallback_lookup = thread_fallback_lookup.drop_duplicates(subset=["thread_id"], keep="first")


# ============================================================
# 4) Standardise comments and attach parent post context
# ============================================================

c = reddit_comments_raw.copy()

comment_id_col = first_existing_col(c, ["id", "comment_id", "reddit_id"])
comment_body_col = first_existing_col(c, ["body", "comment_text", "text", "raw_text", "clean_text"])
comment_author_col = first_existing_col(c, ["author", "username", "comment_author"])
comment_score_col = first_existing_col(c, ["score", "comment_score", "upvotes", "engagement_score"])
comment_url_col = first_existing_col(c, ["url", "permalink", "comment_url"])

comment_link_col = first_existing_col(c, ["link_id", "post_id", "submission_id", "thread_id"])

reddit_comments_clean = pd.DataFrame(index=c.index)

reddit_comments_clean["id"] = c[comment_id_col].apply(clean_id) if comment_id_col else ["reddit_comment_" + str(i) for i in range(len(c))]

if comment_link_col:
    reddit_comments_clean["thread_id"] = c[comment_link_col].apply(clean_id)
else:
    reddit_comments_clean["thread_id"] = pd.NA

reddit_comments_clean["link_id"] = reddit_comments_clean["thread_id"]

if "parent_id" in c.columns:
    reddit_comments_clean["parent_id"] = c["parent_id"].apply(clean_id)
else:
    reddit_comments_clean["parent_id"] = pd.NA

reddit_comments_clean["date"] = parse_reddit_date(c, "comments")
reddit_comments_clean["platform"] = "reddit"
reddit_comments_clean["source_label"] = "reddit"
reddit_comments_clean["post_type"] = "reddit_comment"

reddit_comments_clean["raw_text"] = c[comment_body_col].fillna("").astype(str) if comment_body_col else ""
reddit_comments_clean["clean_text"] = reddit_comments_clean["raw_text"].apply(basic_clean_text)

reddit_comments_clean["username"] = c[comment_author_col] if comment_author_col else pd.NA
reddit_comments_clean["url"] = c[comment_url_col] if comment_url_col else pd.NA

if "subreddit" in c.columns:
    reddit_comments_clean["subreddit"] = c["subreddit"].fillna("unknown").astype(str)
else:
    reddit_comments_clean["subreddit"] = "unknown"

reddit_comments_clean["platform_interaction_value"] = pd.to_numeric(c[comment_score_col], errors="coerce") if comment_score_col else pd.NA
reddit_comments_clean["platform_interaction_definition"] = "Reddit native score for post/comment at collection time"

# Preserve useful classifier/provenance columns
for col in c.columns:
    col_low = col.lower()
    if "deberta" in col_low or "real_estate" in col_low or "confidence" in col_low or "prob" in col_low:
        reddit_comments_clean[col] = c[col]

# Attach parent post context
reddit_comments_clean = reddit_comments_clean.merge(
    post_context_lookup,
    on="thread_id",
    how="left",
)

# Optional fallback from threads if post context is missing
if len(thread_fallback_lookup):
    reddit_comments_clean = reddit_comments_clean.merge(
        thread_fallback_lookup,
        on="thread_id",
        how="left",
    )
else:
    reddit_comments_clean["thread_fallback_text"] = pd.NA

# Fill context fields
reddit_comments_clean["parent_post_title"] = reddit_comments_clean["parent_post_title"].fillna("")
reddit_comments_clean["parent_post_text"] = reddit_comments_clean["parent_post_text"].fillna("")
reddit_comments_clean["parent_post_clean_text"] = reddit_comments_clean["parent_post_clean_text"].fillna("")
reddit_comments_clean["thread_fallback_text"] = reddit_comments_clean["thread_fallback_text"].fillna("")

reddit_comments_clean["context_text"] = np.where(
    reddit_comments_clean["parent_post_clean_text"].str.strip().ne(""),
    reddit_comments_clean["parent_post_clean_text"],
    reddit_comments_clean["thread_fallback_text"]
)

# This is the text used for domain filtering/validation display.
# It keeps the comment, but gives it parent post context.
reddit_comments_clean["domain_context_text"] = (
    "POST CONTEXT: "
    + reddit_comments_clean["context_text"].fillna("").astype(str)
    + " COMMENT: "
    + reddit_comments_clean["clean_text"].fillna("").astype(str)
).str.strip()

reddit_comments_clean["analysis_text_for_filter"] = reddit_comments_clean["domain_context_text"]

reddit_comments_clean["context_available"] = reddit_comments_clean["context_text"].fillna("").astype(str).str.strip().ne("")

# If subreddit missing on comment, fill from parent post
if "parent_post_subreddit" in reddit_comments_clean.columns:
    missing_sub = reddit_comments_clean["subreddit"].isin(["unknown", "", "nan"]) | reddit_comments_clean["subreddit"].isna()
    reddit_comments_clean.loc[missing_sub, "subreddit"] = reddit_comments_clean.loc[missing_sub, "parent_post_subreddit"]


# ============================================================
# 5) Combine posts + context-aware comments
# ============================================================

reddit_clean = pd.concat(
    [reddit_posts_clean, reddit_comments_clean],
    ignore_index=True,
    sort=False,
)

# Date window
before = len(reddit_clean)
reddit_clean = reddit_clean[
    (reddit_clean["date"] >= OBS_START)
    &
    (reddit_clean["date"] < OBS_END_EXCLUSIVE)
].copy()
print(f"Reddit date window: {before} -> {len(reddit_clean)}")

# Remove empty/deleted/removed target texts
deleted_markers = {"", "[deleted]", "[removed]", "deleted", "removed"}
before = len(reddit_clean)
reddit_clean = reddit_clean[
    ~reddit_clean["clean_text"].fillna("").astype(str).str.strip().str.lower().isin(deleted_markers)
].copy()
print(f"Reddit non-empty/deleted filter: {before} -> {len(reddit_clean)}")

# Dedup by platform row id
before = len(reddit_clean)
reddit_clean = reddit_clean.drop_duplicates(subset=["id"], keep="first").copy()
print(f"Reddit ID dedup: {before} -> {len(reddit_clean)}")

print("\nreddit_clean:", reddit_clean.shape)

print("\nPost type counts:")
print(reddit_clean["post_type"].value_counts(dropna=False))

print("\nSubreddit counts:")
print(reddit_clean["subreddit"].value_counts(dropna=False).head(20))

print("\nComment context availability:")
print(
    reddit_clean.loc[reddit_clean["post_type"].eq("reddit_comment"), "context_available"]
    .value_counts(dropna=False)
)

display(
    reddit_clean[["id", "date", "subreddit", "post_type", "clean_text", "context_text"]]
    .sample(min(20, len(reddit_clean)), random_state=42)
)

In [ ]:
# ============================================================
# COMBINE TWITTER CLEAN + CONTEXT-AWARE REDDIT CLEAN
# ============================================================

import pandas as pd
import numpy as np

assert "twitter_clean" in globals(), "twitter_clean not found. Run 02_clean_twitter first."
assert "reddit_clean" in globals(), "reddit_clean not found. Run Reddit context-aware cell first."

COMMON_COLS = [
    "id",
    "platform",
    "source_label",
    "post_type",
    "date",
    "raw_text",
    "clean_text",
    "domain_context_text",
    "analysis_text_for_filter",
    "context_text",
    "context_available",
    "parent_post_title",
    "parent_post_text",
    "username",
    "url",
    "subreddit",
    "thread_id",
    "parent_id",
    "link_id",
    "platform_interaction_value",
    "platform_interaction_definition",
    "likeCount",
    "retweetCount",
    "replyCount",
    "quoteCount",
    "is_retweet",
    "is_reply",
    "retrieval_stream",
    "retrieval_streams",
    "query_used",
    "extraction_cohort",
    "minilm_positive_max_similarity",
    "minilm_negative_max_similarity",
    "minilm_semantic_margin",
    "minilm_best_positive_reference",
    "minilm_best_negative_reference",
]

def align_to_common_schema(df, cols):
    df = df.copy()
    for col in cols:
        if col not in df.columns:
            df[col] = pd.NA
    return df[cols]

twitter_for_master = twitter_clean.copy()
twitter_for_master["platform"] = "twitter"
twitter_for_master["post_type"] = "tweet"

# Twitter has no parent thread context, so context = tweet itself
twitter_for_master["context_text"] = twitter_for_master["clean_text"]
twitter_for_master["domain_context_text"] = twitter_for_master["clean_text"]
twitter_for_master["analysis_text_for_filter"] = twitter_for_master["clean_text"]
twitter_for_master["context_available"] = True

twitter_for_master["platform_interaction_value"] = (
    pd.to_numeric(twitter_for_master.get("likeCount", 0), errors="coerce").fillna(0)
    + pd.to_numeric(twitter_for_master.get("retweetCount", 0), errors="coerce").fillna(0)
    + pd.to_numeric(twitter_for_master.get("replyCount", 0), errors="coerce").fillna(0)
    + pd.to_numeric(twitter_for_master.get("quoteCount", 0), errors="coerce").fillna(0)
)

twitter_for_master["platform_interaction_definition"] = (
    "Twitter/X total interactions: likeCount + retweetCount + replyCount + quoteCount"
)

reddit_for_master = reddit_clean.copy()

master = pd.concat(
    [
        align_to_common_schema(twitter_for_master, COMMON_COLS),
        align_to_common_schema(reddit_for_master, COMMON_COLS),
    ],
    ignore_index=True,
)

master["date"] = pd.to_datetime(master["date"], errors="coerce", utc=True)
master["clean_text"] = master["clean_text"].fillna("").astype(str)
master["domain_context_text"] = master["domain_context_text"].fillna(master["clean_text"]).astype(str)

before = len(master)
master = master[
    (master["date"] >= pd.Timestamp("2026-01-01", tz="UTC"))
    &
    (master["date"] < pd.Timestamp("2026-05-01", tz="UTC"))
    &
    (master["clean_text"].str.strip().ne(""))
].copy()

print(f"master combined: {before} -> {len(master)}")
print("date:", master["date"].min(), "to", master["date"].max())

print("\nPlatform counts:")
print(master["platform"].value_counts(dropna=False))

print("\nPost type counts:")
print(master["post_type"].value_counts(dropna=False))

print("\nReddit subreddit counts:")
print(master.loc[master["platform"].eq("reddit"), "subreddit"].value_counts(dropna=False).head(20))

print("\nReddit comment context availability:")
print(
    master.loc[
        master["post_type"].eq("reddit_comment"),
        "context_available"
    ].value_counts(dropna=False)
)

In [ ]:
# ============================================================
# REBUILD REDDIT CLEAN USING:
# posts file = reddit_post rows
# threads file = reddit_comment rows with parent post context
# comments file = audit only, not used in master
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import re

REDDIT_DIR = Path("/content/drive/MyDrive/Dubai_Real_Estate_Data/Prajwal final reddit")

POSTS_PATH = REDDIT_DIR / "combined_reddit_posts.csv"
COMMENTS_PATH = REDDIT_DIR / "combined_reddit_comments.csv"   # audit only
THREADS_PATH = REDDIT_DIR / "combined_reddit_threads.csv"     # contextual comments

assert POSTS_PATH.exists(), POSTS_PATH
assert COMMENTS_PATH.exists(), COMMENTS_PATH
assert THREADS_PATH.exists(), THREADS_PATH

reddit_posts_raw = pd.read_csv(POSTS_PATH, low_memory=False)
reddit_comments_raw_audit = pd.read_csv(COMMENTS_PATH, low_memory=False)
reddit_threads_raw = pd.read_csv(THREADS_PATH, low_memory=False)

print("Loaded:")
print("posts:", reddit_posts_raw.shape)
print("comments audit only:", reddit_comments_raw_audit.shape)
print("threads/contextual comments:", reddit_threads_raw.shape)

OBS_START = pd.Timestamp("2026-01-01", tz="UTC")
OBS_END_EXCLUSIVE = pd.Timestamp("2026-05-01", tz="UTC")

def clean_id(x):
    if pd.isna(x):
        return pd.NA
    return str(x).replace("t3_", "").replace("t1_", "").strip()

def clean_text_basic(x):
    x = "" if pd.isna(x) else str(x)
    x = re.sub(r"http\S+|www\.\S+", " ", x)
    x = re.sub(r"\s+", " ", x)
    return x.strip()

def parse_date(s):
    return pd.to_datetime(s, errors="coerce", utc=True)

# ============================================================
# 1) Posts as reddit_post rows
# ============================================================

p = reddit_posts_raw.copy()

reddit_posts_clean = pd.DataFrame()

reddit_posts_clean["id"] = p["post_id"].apply(clean_id)
reddit_posts_clean["platform"] = "reddit"
reddit_posts_clean["source_label"] = "reddit"
reddit_posts_clean["post_type"] = "reddit_post"
reddit_posts_clean["date"] = parse_date(p["date"])

reddit_posts_clean["raw_text"] = (
    p["title"].fillna("").astype(str)
    + " "
    + p["post_text"].fillna(p["selftext"]).fillna("").astype(str)
).str.strip()

reddit_posts_clean["clean_text"] = reddit_posts_clean["raw_text"].apply(clean_text_basic)

reddit_posts_clean["thread_id"] = reddit_posts_clean["id"]
reddit_posts_clean["link_id"] = reddit_posts_clean["id"]
reddit_posts_clean["parent_id"] = pd.NA

reddit_posts_clean["parent_post_title"] = p["title"].fillna("").astype(str)
reddit_posts_clean["parent_post_text"] = p["post_text"].fillna(p["selftext"]).fillna("").astype(str)

reddit_posts_clean["context_text"] = reddit_posts_clean["clean_text"]
reddit_posts_clean["domain_context_text"] = reddit_posts_clean["clean_text"]
reddit_posts_clean["analysis_text_for_filter"] = reddit_posts_clean["clean_text"]
reddit_posts_clean["context_available"] = True

reddit_posts_clean["username"] = p["post_author"]
reddit_posts_clean["subreddit"] = p["subreddit"]
reddit_posts_clean["url"] = p["url"].fillna(p["permalink"])

reddit_posts_clean["platform_interaction_value"] = pd.to_numeric(p["post_score"], errors="coerce")
reddit_posts_clean["platform_interaction_definition"] = "Reddit native post score at collection time"

# Preserve classifier/provenance fields from posts
for col in ["deberta_score", "is_real_estate", "prediction", "upvote_ratio", "num_comments"]:
    if col in p.columns:
        reddit_posts_clean[col] = p[col]

# ============================================================
# 2) Threads file as contextual reddit_comment rows
# ============================================================

t = reddit_threads_raw.copy()

reddit_comments_clean = pd.DataFrame()

reddit_comments_clean["id"] = t["comment_id"].apply(clean_id)
reddit_comments_clean["platform"] = "reddit"
reddit_comments_clean["source_label"] = "reddit"
reddit_comments_clean["post_type"] = "reddit_comment"
reddit_comments_clean["date"] = parse_date(t["date_comment"])

reddit_comments_clean["raw_text"] = t["body"].fillna("").astype(str)
reddit_comments_clean["clean_text"] = reddit_comments_clean["raw_text"].apply(clean_text_basic)

reddit_comments_clean["thread_id"] = t["post_id"].apply(clean_id)
reddit_comments_clean["link_id"] = reddit_comments_clean["thread_id"]
reddit_comments_clean["parent_id"] = t["parent_id"].apply(clean_id)

reddit_comments_clean["parent_post_title"] = t["title"].fillna("").astype(str)
reddit_comments_clean["parent_post_text"] = (
    t["post_text"].fillna(t["selftext"]).fillna("").astype(str)
)

reddit_comments_clean["context_text"] = (
    reddit_comments_clean["parent_post_title"]
    + " "
    + reddit_comments_clean["parent_post_text"]
).apply(clean_text_basic)

reddit_comments_clean["domain_context_text"] = (
    "POST CONTEXT: "
    + reddit_comments_clean["context_text"].fillna("").astype(str)
    + " COMMENT: "
    + reddit_comments_clean["clean_text"].fillna("").astype(str)
).str.strip()

reddit_comments_clean["analysis_text_for_filter"] = reddit_comments_clean["domain_context_text"]
reddit_comments_clean["context_available"] = reddit_comments_clean["context_text"].str.strip().ne("")

reddit_comments_clean["username"] = t["comment_author"]
reddit_comments_clean["subreddit"] = t["subreddit"]
reddit_comments_clean["url"] = t["permalink_comment"]

reddit_comments_clean["platform_interaction_value"] = pd.to_numeric(t["comment_score"], errors="coerce")
reddit_comments_clean["platform_interaction_definition"] = "Reddit native comment score at collection time"

# Preserve post-level fields useful for audit
for col in ["post_score", "upvote_ratio", "num_comments", "is_submitter", "controversiality"]:
    if col in t.columns:
        reddit_comments_clean[col] = t[col]

# ============================================================
# 3) Combine posts + contextual comments
# ============================================================

reddit_clean = pd.concat(
    [reddit_posts_clean, reddit_comments_clean],
    ignore_index=True,
    sort=False
)

before = len(reddit_clean)
reddit_clean = reddit_clean[
    (reddit_clean["date"] >= OBS_START)
    &
    (reddit_clean["date"] < OBS_END_EXCLUSIVE)
].copy()
print(f"Reddit date window: {before} -> {len(reddit_clean)}")

deleted_markers = {"", "[deleted]", "[removed]", "deleted", "removed"}
before = len(reddit_clean)
reddit_clean = reddit_clean[
    ~reddit_clean["clean_text"].fillna("").astype(str).str.strip().str.lower().isin(deleted_markers)
].copy()
print(f"Reddit non-empty/deleted filter: {before} -> {len(reddit_clean)}")

before = len(reddit_clean)
reddit_clean = reddit_clean.drop_duplicates(subset=["id"], keep="first").copy()
print(f"Reddit ID dedup: {before} -> {len(reddit_clean)}")

print("\nreddit_clean:", reddit_clean.shape)

print("\nPost type counts:")
print(reddit_clean["post_type"].value_counts(dropna=False))

print("\nSubreddit counts:")
print(reddit_clean["subreddit"].value_counts(dropna=False).head(20))

print("\nComment context availability:")
print(
    reddit_clean.loc[
        reddit_clean["post_type"].eq("reddit_comment"),
        "context_available"
    ].value_counts(dropna=False)
)

print("\nAudit: raw comments file vs contextual comments file")
print("raw comments audit rows:", len(reddit_comments_raw_audit))
print("contextual comment rows before filters:", len(reddit_threads_raw))
print("difference:", len(reddit_comments_raw_audit) - len(reddit_threads_raw))

display(
    reddit_clean[["id", "date", "subreddit", "post_type", "clean_text", "context_text"]]
    .sample(min(20, len(reddit_clean)), random_state=42)
)

In [ ]:
# ============================================================
# REBUILD MASTER: TWITTER CLEAN + REDDIT CLEAN
# ============================================================

import pandas as pd
import numpy as np

assert "twitter_clean" in globals(), "twitter_clean not found. Run 02_clean_twitter first."
assert "reddit_clean" in globals(), "reddit_clean not found. Run Reddit rebuild cell first."

COMMON_COLS = [
    "id",
    "platform",
    "source_label",
    "post_type",
    "date",
    "raw_text",
    "clean_text",
    "domain_context_text",
    "analysis_text_for_filter",
    "context_text",
    "context_available",
    "parent_post_title",
    "parent_post_text",
    "username",
    "url",
    "subreddit",
    "thread_id",
    "parent_id",
    "link_id",
    "platform_interaction_value",
    "platform_interaction_definition",
    "likeCount",
    "retweetCount",
    "replyCount",
    "quoteCount",
    "is_retweet",
    "is_reply",
    "retrieval_stream",
    "retrieval_streams",
    "query_used",
    "extraction_cohort",
    "minilm_positive_max_similarity",
    "minilm_negative_max_similarity",
    "minilm_semantic_margin",
    "minilm_best_positive_reference",
    "minilm_best_negative_reference",
    "deberta_score",
    "is_real_estate",
    "prediction",
]

def align_to_common_schema(df, cols):
    df = df.copy()
    for col in cols:
        if col not in df.columns:
            df[col] = pd.NA
    return df[cols]

twitter_for_master = twitter_clean.copy()
twitter_for_master["platform"] = "twitter"
twitter_for_master["post_type"] = "tweet"
twitter_for_master["context_text"] = twitter_for_master["clean_text"]
twitter_for_master["domain_context_text"] = twitter_for_master["clean_text"]
twitter_for_master["analysis_text_for_filter"] = twitter_for_master["clean_text"]
twitter_for_master["context_available"] = True

twitter_for_master["platform_interaction_value"] = (
    pd.to_numeric(twitter_for_master.get("likeCount", 0), errors="coerce").fillna(0)
    + pd.to_numeric(twitter_for_master.get("retweetCount", 0), errors="coerce").fillna(0)
    + pd.to_numeric(twitter_for_master.get("replyCount", 0), errors="coerce").fillna(0)
    + pd.to_numeric(twitter_for_master.get("quoteCount", 0), errors="coerce").fillna(0)
)

twitter_for_master["platform_interaction_definition"] = (
    "Twitter/X total interactions: likeCount + retweetCount + replyCount + quoteCount"
)

reddit_for_master = reddit_clean.copy()

master = pd.concat(
    [
        align_to_common_schema(twitter_for_master, COMMON_COLS),
        align_to_common_schema(reddit_for_master, COMMON_COLS),
    ],
    ignore_index=True
)

master["date"] = pd.to_datetime(master["date"], errors="coerce", utc=True)
master["clean_text"] = master["clean_text"].fillna("").astype(str)
master["domain_context_text"] = master["domain_context_text"].fillna(master["clean_text"]).astype(str)
master["analysis_text_for_filter"] = master["analysis_text_for_filter"].fillna(master["domain_context_text"]).astype(str)

master = master[
    (master["date"] >= pd.Timestamp("2026-01-01", tz="UTC"))
    &
    (master["date"] < pd.Timestamp("2026-05-01", tz="UTC"))
    &
    (master["clean_text"].str.strip().ne(""))
].copy()

print("master:", master.shape)
print("date:", master["date"].min(), "to", master["date"].max())

print("\nPlatform counts:")
print(master["platform"].value_counts(dropna=False))

print("\nPost type counts:")
print(master["post_type"].value_counts(dropna=False))

print("\nReddit subreddit counts:")
print(master.loc[master["platform"].eq("reddit"), "subreddit"].value_counts(dropna=False).head(20))

print("\nReddit comment context availability:")
print(
    master.loc[
        master["post_type"].eq("reddit_comment"),
        "context_available"
    ].value_counts(dropna=False)
)

In [ ]:
# ============================================================
# SAVE CHECKPOINT BEFORE DOMAIN / LISTING / DUP FILTERS
# ============================================================

from pathlib import Path

CHECKPOINT_DIR = Path("/content/drive/MyDrive/Dubai_Real_Estate_Data/final_pipeline_outputs/checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

reddit_clean.to_parquet(CHECKPOINT_DIR / "reddit_clean_contextual_comments_pre_domain.parquet", index=False)
reddit_clean.to_csv(CHECKPOINT_DIR / "reddit_clean_contextual_comments_pre_domain.csv", index=False)

reddit_threads_raw.to_parquet(CHECKPOINT_DIR / "reddit_contextual_comments_source_table.parquet", index=False)
reddit_threads_raw.to_csv(CHECKPOINT_DIR / "reddit_contextual_comments_source_table.csv", index=False)

reddit_comments_raw_audit.to_parquet(CHECKPOINT_DIR / "reddit_comments_raw_audit_only.parquet", index=False)
reddit_comments_raw_audit.to_csv(CHECKPOINT_DIR / "reddit_comments_raw_audit_only.csv", index=False)

master.to_parquet(CHECKPOINT_DIR / "master_twitter_reddit_pre_domain.parquet", index=False)
master.to_csv(CHECKPOINT_DIR / "master_twitter_reddit_pre_domain.csv", index=False)

print("Saved checkpoints to:", CHECKPOINT_DIR)
for p in sorted(CHECKPOINT_DIR.glob("*")):
    print(p.name)

In [ ]:
# ============================================================
# 07_CONTEXT_AWARE_DOMAIN_AND_LISTING_FILTER
# Uses analysis_text_for_filter / domain_context_text for Reddit comments
# Creates:
#   master_domain_scored
#   master_domain_filtered
#   master_processed
# ============================================================

import pandas as pd
import numpy as np
import re
from pathlib import Path

assert "master" in globals(), "master missing. Load checkpoint or rebuild master first."

df = master.copy()

# ------------------------------------------------------------
# Text used for filtering
# ------------------------------------------------------------
if "analysis_text_for_filter" in df.columns:
    filter_text_col = "analysis_text_for_filter"
elif "domain_context_text" in df.columns:
    filter_text_col = "domain_context_text"
else:
    filter_text_col = "clean_text"

df["filter_text_used"] = df[filter_text_col].fillna(df["clean_text"]).fillna("").astype(str)

print("Using filter text column:", filter_text_col)
print("Input master:", df.shape)

# ------------------------------------------------------------
# Keyword/cue lists
# Use notebook globals if they exist; otherwise fallback lists.
# ------------------------------------------------------------

DOMAIN_TERMS = globals().get("DOMAIN_KEYWORDS", [
    # Core real estate
    "real estate", "property", "properties", "apartment", "apartments",
    "villa", "villas", "townhouse", "townhouses", "studio", "penthouse",
    "unit", "land", "plot",

    # Dubai RE market terms
    "dubai real estate", "dubai property", "property market", "housing market",
    "rent", "rental", "rents", "renting", "tenant", "landlord", "lease",
    "renewal", "ejari", "rera", "service charges", "mortgage",

    # Buying/selling/investment
    "buy property", "buying property", "sell property", "selling property",
    "off plan", "off-plan", "handover", "payment plan", "roi",
    "capital appreciation", "distress", "distressed", "below market",
    "secondary market", "transaction", "transactions", "valuation",

    # Developers/neighbourhood cues
    "emaar", "damac", "nakheel", "sobha", "danube", "azizi", "binghatti",
    "dubai marina", "downtown dubai", "business bay", "jvc",
    "palm jumeirah", "dubai hills", "dubai south", "creek harbour",
    "arabian ranches", "jumeirah village circle", "meydan", "sobha hartland",
])

NOISE_TERMS = globals().get("NOISE_KEYWORDS", globals().get("NOISE_CUES", [
    "crypto", "nft", "giveaway", "airdrop", "job hiring", "hiring now",
    "casino", "betting", "porn", "escort", "dating", "football", "ufc",
    "movie", "celebrity", "restaurant", "hotel stay", "vacation",
]))

AGENT_LISTING_TERMS = globals().get("AGENT_LISTING_CUES", [
    "whatsapp", "call now", "contact us", "dm me", "dm for", "inbox",
    "book now", "book your unit", "book viewing", "schedule a viewing",
    "viewing", "register now", "limited offer", "exclusive offer",
    "for sale", "for rent", "available now", "ready to move",
    "fully furnished", "brand new", "vacant", "single row",
    "motivated seller", "distress deal", "distressed property",
    "below op", "below original price", "payment plan", "handover soon",
    "dld waiver", "commission", "broker", "agent", "rera permit",
    "permit no", "ref no", "reference no", "roi", "guaranteed return",
    "starting at", "aed", "sqft", "sq ft", "1br", "2br", "3br", "4br", "5br",
])

MAX_NOISE = globals().get("MAX_NOISE_CUES_FOR_DOMAIN_ROW", 1)
AGENT_THRESHOLD = globals().get("AGENT_LISTING_CUE_THRESHOLD", 2)

def normalise_for_match(x):
    x = "" if pd.isna(x) else str(x).lower()
    x = re.sub(r"https?://\S+|www\.\S+", " ", x)
    x = re.sub(r"[^a-z0-9\s\-_/+.%]", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x

def matched_terms(text, terms):
    text_n = normalise_for_match(text)
    found = []
    for term in terms:
        term_n = normalise_for_match(term)
        if not term_n:
            continue
        # phrase/substr match, safer for phrases like "dubai real estate", "1br", "aed"
        if term_n in text_n:
            found.append(term)
    # unique while preserving order
    return list(dict.fromkeys(found))

df["domain_keywords_matched"] = df["filter_text_used"].apply(lambda x: matched_terms(x, DOMAIN_TERMS))
df["domain_keyword_count"] = df["domain_keywords_matched"].apply(len)

df["noise_keywords_matched"] = df["filter_text_used"].apply(lambda x: matched_terms(x, NOISE_TERMS))
df["noise_keyword_count"] = df["noise_keywords_matched"].apply(len)

df["agent_listing_cues_matched"] = df["filter_text_used"].apply(lambda x: matched_terms(x, AGENT_LISTING_TERMS))
df["agent_listing_cue_count"] = df["agent_listing_cues_matched"].apply(len)

df["is_domain_relevant"] = (
    (df["domain_keyword_count"] > 0)
    &
    (df["noise_keyword_count"] <= MAX_NOISE)
)

df["is_likely_agent_listing"] = df["agent_listing_cue_count"] >= AGENT_THRESHOLD

# Final keep = domain relevant and not likely listing/ad
df["pipeline_keep_final"] = (
    df["is_domain_relevant"]
    &
    (~df["is_likely_agent_listing"])
)

# Time columns
df["date"] = pd.to_datetime(df["date"], errors="coerce", utc=True)
df["yearweek"] = df["date"].dt.strftime("%G-W%V")
df["month"] = df["date"].dt.to_period("M").astype(str)
df["week_num"] = df["date"].dt.isocalendar().week.astype("Int64")

master_domain_scored = df.copy()
master_domain_filtered = df[df["is_domain_relevant"]].copy()
master_processed = df[df["pipeline_keep_final"]].copy()

print("\n=== FILTER SUMMARY ===")
print("Input rows:", len(df))
print("Domain relevant:", len(master_domain_filtered))
print("Likely agent/listing rows:", int(df["is_likely_agent_listing"].sum()))
print("Final kept after domain + listing filter:", len(master_processed))

print("\nDomain relevant by platform:")
print(master_domain_filtered["platform"].value_counts(dropna=False))

print("\nFinal kept by platform:")
print(master_processed["platform"].value_counts(dropna=False))

print("\nFinal kept by post_type:")
print(master_processed["post_type"].value_counts(dropna=False))

print("\nLikely agent/listing by platform:")
print(df.groupby("platform")["is_likely_agent_listing"].sum())

print("\nSample removed likely listings:")
display(
    df[df["is_likely_agent_listing"]][
        ["platform", "post_type", "date", "clean_text", "agent_listing_cues_matched"]
    ].head(20)
)

print("\nSample final kept:")
display(
    master_processed[
        ["platform", "post_type", "date", "clean_text", "domain_keywords_matched", "agent_listing_cue_count"]
    ].sample(min(20, len(master_processed)), random_state=42)
)

In [ ]:
# ============================================================
# SAVE AGGRESSIVE FILTER VERSION FOR COMPARISON / AUDIT
# ============================================================

from pathlib import Path

FILTER_DIR = Path("/content/drive/MyDrive/Dubai_Real_Estate_Data/final_pipeline_outputs/filter_checkpoints")
FILTER_DIR.mkdir(parents=True, exist_ok=True)

master_domain_scored.to_parquet(FILTER_DIR / "master_domain_scored_aggressive_listing.parquet", index=False)
master_domain_scored.to_csv(FILTER_DIR / "master_domain_scored_aggressive_listing.csv", index=False)

master_processed.to_parquet(FILTER_DIR / "master_processed_aggressive_listing.parquet", index=False)
master_processed.to_csv(FILTER_DIR / "master_processed_aggressive_listing.csv", index=False)

print("Saved aggressive filter version to:", FILTER_DIR)

In [ ]:
# ============================================================
# REFINED LISTING FILTER
# Keeps domain filter, improves agent/listing detection
# Avoids removing analytical posts just because they mention AED/sqft/BR
# ============================================================

import pandas as pd
import numpy as np
import re

assert "master_domain_scored" in globals(), "Run domain scoring first."

df = master_domain_scored.copy()

def normalise_for_match(x):
    x = "" if pd.isna(x) else str(x).lower()
    x = re.sub(r"https?://\S+|www\.\S+", " ", x)
    x = re.sub(r"[^a-z0-9\s\-_/+.%]", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x

def matched_terms(text, terms):
    text_n = normalise_for_match(text)
    found = []
    for term in terms:
        term_n = normalise_for_match(term)
        if term_n and term_n in text_n:
            found.append(term)
    return list(dict.fromkeys(found))

# Strong cues: actual promotional / contact / listing intent
STRONG_LISTING_CUES = [
    "whatsapp",
    "call now",
    "contact us",
    "contact me",
    "dm me",
    "dm for",
    "send me a dm",
    "inbox",
    "book viewing",
    "schedule a viewing",
    "viewing today",
    "register now",
    "dm to register",
    "limited offer",
    "exclusive offer",
    "available now",
    "ready to move",
    "for sale",
    "for rent",
    "rent with",
    "selling my",
    "selling 1br",
    "selling 2br",
    "selling 3br",
    "motivated seller",
    "distress deal",
    "distressed property",
    "below op",
    "below original price",
    "dld waiver",
    "commission",
    "broker",
    "agent",
    "rera permit",
    "permit no",
    "ref no",
    "reference no",
    "guaranteed return",
    "book your unit",
]

# Weak cues: property specs / numbers. These alone do NOT mean ad/listing.
WEAK_LISTING_CUES = [
    "aed",
    "sqft",
    "sq ft",
    "1br",
    "2br",
    "3br",
    "4br",
    "5br",
    "studio",
    "fully furnished",
    "brand new",
    "vacant",
    "single row",
    "payment plan",
    "handover soon",
    "starting at",
    "roi",
]

# Analysis/news cues that should protect rows from false removal
ANALYSIS_PROTECTION_CUES = [
    "analysis",
    "market",
    "price drop",
    "prices",
    "rent prices",
    "transaction",
    "transactions",
    "recorded",
    "data",
    "report",
    "index",
    "bubble",
    "panic",
    "collapsed",
    "overvalued",
    "undervalued",
    "demand",
    "supply",
    "trend",
    "investors who bought",
    "everyone panicking",
    "looked at",
    "scanned",
]

df["strong_listing_cues_matched"] = df["filter_text_used"].apply(
    lambda x: matched_terms(x, STRONG_LISTING_CUES)
)
df["weak_listing_cues_matched"] = df["filter_text_used"].apply(
    lambda x: matched_terms(x, WEAK_LISTING_CUES)
)
df["analysis_protection_cues_matched"] = df["filter_text_used"].apply(
    lambda x: matched_terms(x, ANALYSIS_PROTECTION_CUES)
)

df["strong_listing_cue_count"] = df["strong_listing_cues_matched"].apply(len)
df["weak_listing_cue_count"] = df["weak_listing_cues_matched"].apply(len)
df["analysis_protection_cue_count"] = df["analysis_protection_cues_matched"].apply(len)

# Refined listing rule
df["is_likely_agent_listing_refined"] = (
    # Direct strong promotional intent
    (df["strong_listing_cue_count"] >= 2)

    |

    # One strong cue + at least one property-spec cue
    (
        (df["strong_listing_cue_count"] >= 1)
        &
        (df["weak_listing_cue_count"] >= 1)
    )

    |

    # Many weak specs can indicate a listing template, but only without analysis/news cues
    (
        (df["weak_listing_cue_count"] >= 4)
        &
        (df["analysis_protection_cue_count"] == 0)
    )
)

# Extra protection: if a tweet/reddit row is clearly analytical, don't remove it only because of weak specs
df.loc[
    (df["analysis_protection_cue_count"] >= 1)
    &
    (df["strong_listing_cue_count"] == 0),
    "is_likely_agent_listing_refined"
] = False

# Final keep
df["pipeline_keep_final_refined"] = (
    df["is_domain_relevant"]
    &
    (~df["is_likely_agent_listing_refined"])
)

master_domain_scored_refined = df.copy()
master_domain_filtered = df[df["is_domain_relevant"]].copy()
master_processed = df[df["pipeline_keep_final_refined"]].copy()

print("=== REFINED FILTER SUMMARY ===")
print("Input rows:", len(df))
print("Domain relevant:", int(df["is_domain_relevant"].sum()))
print("Old likely listing rows:", int(df["is_likely_agent_listing"].sum()) if "is_likely_agent_listing" in df.columns else "NA")
print("Refined likely listing rows:", int(df["is_likely_agent_listing_refined"].sum()))
print("Final kept after refined filter:", len(master_processed))

print("\nFinal kept by platform:")
print(master_processed["platform"].value_counts(dropna=False))

print("\nFinal kept by post_type:")
print(master_processed["post_type"].value_counts(dropna=False))

print("\nRefined likely listing by platform:")
print(df.groupby("platform")["is_likely_agent_listing_refined"].sum())

print("\nSample refined removed listings:")
display(
    df[df["is_likely_agent_listing_refined"]][
        [
            "platform",
            "post_type",
            "date",
            "clean_text",
            "strong_listing_cues_matched",
            "weak_listing_cues_matched",
            "analysis_protection_cues_matched",
        ]
    ].sample(min(25, int(df["is_likely_agent_listing_refined"].sum())), random_state=42)
)

print("\nSample final kept:")
display(
    master_processed[
        [
            "platform",
            "post_type",
            "date",
            "clean_text",
            "domain_keywords_matched",
            "strong_listing_cue_count",
            "weak_listing_cue_count",
            "analysis_protection_cue_count",
        ]
    ].sample(min(25, len(master_processed)), random_state=42)
)

refined list

In [ ]:
# ============================================================
# REFINED LISTING + LOW-VALUE CONTACT COMMENT FILTER
# Fixes rows like:
#   "DM", "Sent a DM", "Inbox", "Pls connect", "Private chat"
# ============================================================

import pandas as pd
import numpy as np
import re

assert "master_domain_scored" in globals(), "Run the domain scoring cell first."

df = master_domain_scored.copy()

def norm_text(x):
    x = "" if pd.isna(x) else str(x).lower()
    x = re.sub(r"https?://\S+|www\.\S+", " ", x)
    x = re.sub(r"[^a-z0-9\s\-_/+.%]", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x

def matched_terms(text, terms):
    text_n = norm_text(text)
    found = []
    for term in terms:
        term_n = norm_text(term)
        if term_n and term_n in text_n:
            found.append(term)
    return list(dict.fromkeys(found))

# ------------------------------------------------------------
# Strong promotional/contact/listing intent
# ------------------------------------------------------------
STRONG_LISTING_CUES = [
    # direct contact / conversion
    "whatsapp",
    "call now",
    "contact us",
    "contact me",
    "contact for",
    "dm me",
    "dm for",
    "dm details",
    "send me a dm",
    "sent a dm",
    "sent you a dm",
    "sent you in private chat",
    "private chat",
    "pm me",
    "inbox",
    "pls connect",
    "please connect",
    "connect with me",
    "book viewing",
    "schedule a viewing",
    "viewing today",
    "register now",
    "dm to register",

    # listing/agent wording
    "available now",
    "ready to move",
    "for sale",
    "for rent",
    "selling my",
    "motivated seller",
    "distress deal",
    "distressed property",
    "below op",
    "below original price",
    "dld waiver",
    "commission",
    "broker",
    "agent",
    "rera permit",
    "permit no",
    "ref no",
    "reference no",
    "guaranteed return",
    "book your unit",
    "limited offer",
    "exclusive offer",
]

# Specs/numeric/property-ad style cues.
# These alone should not remove analytical posts.
WEAK_LISTING_CUES = [
    "aed",
    "sqft",
    "sq ft",
    "1br",
    "2br",
    "3br",
    "4br",
    "5br",
    "studio",
    "fully furnished",
    "brand new",
    "vacant",
    "single row",
    "payment plan",
    "handover soon",
    "starting at",
    "roi",
]

# Market/news/analysis cues that protect rows from being removed only for weak specs.
ANALYSIS_PROTECTION_CUES = [
    "analysis",
    "market",
    "price drop",
    "prices",
    "rent prices",
    "transaction",
    "transactions",
    "recorded",
    "data",
    "report",
    "index",
    "bubble",
    "panic",
    "collapsed",
    "overvalued",
    "undervalued",
    "demand",
    "supply",
    "trend",
    "investors who bought",
    "everyone panicking",
    "scanned",
    "forecast",
    "valuation",
]

# Contact-only short comments: these are not useful discourse rows.
CONTACT_ONLY_PATTERNS = [
    r"^\s*d+\.?m+\.?\s*$",
    r"^\s*dm me\s*$",
    r"^\s*dm\s+details\s*$",
    r"^\s*send me (a )?dm\s*$",
    r"^\s*sent (a )?dm\s*$",
    r"^\s*sent you (a )?dm\s*$",
    r"^\s*i sent you (a )?dm\s*$",
    r"^\s*i just sent you.*private chat\s*$",
    r"^\s*private chat\s*$",
    r"^\s*inbox\s*$",
    r"^\s*pm me\s*$",
    r"^\s*pls connect\s*$",
    r"^\s*please connect\s*$",
    r"^\s*hi\.?\s*pls connect\s*$",
    r"^\s*hi\.?\s*please connect\s*$",
    r"^\s*connect\s*$",
    r"^\s*interested\s*$",
    r"^\s*interested\.?\s*looking for myself\.?\s*$",
    r"^\s*details\s*$",
    r"^\s*share details\s*$",
    r"^\s*send details\s*$",
]

CONTACT_ONLY_REGEX = re.compile("|".join(CONTACT_ONLY_PATTERNS), flags=re.IGNORECASE)

def word_count(x):
    return len(re.findall(r"\b\w+\b", "" if pd.isna(x) else str(x)))

df["strong_listing_cues_matched"] = df["filter_text_used"].apply(
    lambda x: matched_terms(x, STRONG_LISTING_CUES)
)
df["weak_listing_cues_matched"] = df["filter_text_used"].apply(
    lambda x: matched_terms(x, WEAK_LISTING_CUES)
)
df["analysis_protection_cues_matched"] = df["filter_text_used"].apply(
    lambda x: matched_terms(x, ANALYSIS_PROTECTION_CUES)
)

df["strong_listing_cue_count"] = df["strong_listing_cues_matched"].apply(len)
df["weak_listing_cue_count"] = df["weak_listing_cues_matched"].apply(len)
df["analysis_protection_cue_count"] = df["analysis_protection_cues_matched"].apply(len)

# Use clean_text, not context text, for short-contact detection.
df["clean_text_norm"] = df["clean_text"].fillna("").astype(str).apply(norm_text)
df["clean_text_word_count"] = df["clean_text"].fillna("").astype(str).apply(word_count)

df["is_short_contact_only_comment"] = (
    df["post_type"].eq("reddit_comment")
    &
    (
        df["clean_text"].fillna("").astype(str).str.match(CONTACT_ONLY_REGEX)
        |
        (
            df["clean_text_word_count"] <= 5
            &
            df["clean_text_norm"].str.contains(
                r"\b(dm|inbox|connect|details|private chat|pm)\b",
                regex=True,
                na=False,
            )
        )
    )
)

# Main refined listing rule
df["is_likely_agent_listing_refined"] = (
    # Direct short contact comments are removed
    df["is_short_contact_only_comment"]

    |

    # Strong promotional intent
    (df["strong_listing_cue_count"] >= 2)

    |

    # One strong cue + at least one weak property-spec cue
    (
        (df["strong_listing_cue_count"] >= 1)
        &
        (df["weak_listing_cue_count"] >= 1)
    )

    |

    # Many weak specs = listing template, unless clearly analytical
    (
        (df["weak_listing_cue_count"] >= 4)
        &
        (df["analysis_protection_cue_count"] == 0)
    )
)

# Protect clearly analytical posts from removal when they only have weak specs
df.loc[
    (df["analysis_protection_cue_count"] >= 1)
    &
    (df["strong_listing_cue_count"] == 0)
    &
    (~df["is_short_contact_only_comment"]),
    "is_likely_agent_listing_refined"
] = False

df["pipeline_keep_final_refined"] = (
    df["is_domain_relevant"]
    &
    (~df["is_likely_agent_listing_refined"])
)

master_domain_scored_refined = df.copy()
master_domain_filtered = df[df["is_domain_relevant"]].copy()
master_processed = df[df["pipeline_keep_final_refined"]].copy()

print("=== REFINED FILTER WITH CONTACT-ONLY COMMENT REMOVAL ===")
print("Input rows:", len(df))
print("Domain relevant:", int(df["is_domain_relevant"].sum()))
print("Old aggressive likely listing rows:", int(df["is_likely_agent_listing"].sum()) if "is_likely_agent_listing" in df.columns else "NA")
print("Refined likely listing rows:", int(df["is_likely_agent_listing_refined"].sum()))
print("Short contact-only comments removed:", int(df["is_short_contact_only_comment"].sum()))
print("Final kept:", len(master_processed))

print("\nFinal kept by platform:")
print(master_processed["platform"].value_counts(dropna=False))

print("\nFinal kept by post_type:")
print(master_processed["post_type"].value_counts(dropna=False))

print("\nRefined likely listing by platform:")
print(df.groupby("platform")["is_likely_agent_listing_refined"].sum())

print("\nShort contact-only examples removed:")
display(
    df[df["is_short_contact_only_comment"]][
        ["platform", "post_type", "date", "clean_text", "context_text"]
    ].sample(min(25, int(df["is_short_contact_only_comment"].sum())), random_state=42)
)

print("\nRefined removed listing examples:")
display(
    df[df["is_likely_agent_listing_refined"]][
        [
            "platform",
            "post_type",
            "date",
            "clean_text",
            "strong_listing_cues_matched",
            "weak_listing_cues_matched",
            "analysis_protection_cues_matched",
            "is_short_contact_only_comment",
        ]
    ].sample(min(25, int(df["is_likely_agent_listing_refined"].sum())), random_state=42)
)

print("\nFinal kept examples:")
display(
    master_processed[
        [
            "platform",
            "post_type",
            "date",
            "clean_text",
            "domain_keywords_matched",
            "strong_listing_cue_count",
            "weak_listing_cue_count",
            "analysis_protection_cue_count",
        ]
    ].sample(min(25, len(master_processed)), random_state=42)
)

In [ ]:
# ============================================================
# CHECK IF DM / INBOX / CONNECT COMMENTS SURVIVED
# ============================================================

suspect = master_processed[
    master_processed["clean_text"].fillna("").astype(str).str.lower().str.contains(
        r"^\s*(dm|dm me|sent a dm|send me a dm|inbox|pls connect|please connect|private chat|pm me|details|share details)\s*\.?\s*$",
        regex=True,
        na=False,
    )
]

print("Obvious contact-only rows still kept:", len(suspect))

if len(suspect):
    display(suspect[["platform", "post_type", "date", "clean_text", "context_text"]].head(50))

In [ ]:
# ============================================================
# SAVE REFINED FILTER VERSION
# ============================================================

from pathlib import Path

FILTER_DIR = Path("/content/drive/MyDrive/Dubai_Real_Estate_Data/final_pipeline_outputs/filter_checkpoints")
FILTER_DIR.mkdir(parents=True, exist_ok=True)

master_domain_scored_refined.to_parquet(
    FILTER_DIR / "master_domain_scored_refined_listing_contact_clean.parquet",
    index=False
)
master_domain_scored_refined.to_csv(
    FILTER_DIR / "master_domain_scored_refined_listing_contact_clean.csv",
    index=False
)

master_domain_filtered.to_parquet(
    FILTER_DIR / "master_domain_filtered.parquet",
    index=False
)
master_domain_filtered.to_csv(
    FILTER_DIR / "master_domain_filtered.csv",
    index=False
)

master_processed.to_parquet(
    FILTER_DIR / "master_processed_after_refined_domain_listing_contact_clean.parquet",
    index=False
)
master_processed.to_csv(
    FILTER_DIR / "master_processed_after_refined_domain_listing_contact_clean.csv",
    index=False
)

print("Saved refined filter checkpoint to:", FILTER_DIR)

print("\nFinal master_processed:", master_processed.shape)

print("\nPlatform counts:")
print(master_processed["platform"].value_counts(dropna=False))

print("\nPost type counts:")
print(master_processed["post_type"].value_counts(dropna=False))

print("\nSaved files:")
for p in sorted(FILTER_DIR.glob("*refined*")):
    print("-", p.name)

In [ ]:
# ============================================================
# FRESH SESSION: LOAD SAVED REFINED master_processed
# THEN RUN DUPLICATE + NEAR-DUPLICATE AUDIT
# ============================================================

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

from pathlib import Path
import pandas as pd
import numpy as np
import re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

# ------------------------------------------------------------
# 1) Load saved refined final filtered dataset
# ------------------------------------------------------------

BASE = Path("/content/drive/MyDrive/Dubai_Real_Estate_Data")
FILTER_DIR = BASE / "final_pipeline_outputs" / "filter_checkpoints"
AUDIT_DIR = BASE / "final_pipeline_outputs" / "paper_ready_tables"
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

MASTER_PROCESSED_PATH = FILTER_DIR / "master_processed_after_refined_domain_listing_contact_clean.parquet"

assert MASTER_PROCESSED_PATH.exists(), f"Missing: {MASTER_PROCESSED_PATH}"

master_processed = pd.read_parquet(MASTER_PROCESSED_PATH)

print("Loaded master_processed:", master_processed.shape)
print("\nPlatform counts:")
print(master_processed["platform"].value_counts(dropna=False))
print("\nPost type counts:")
print(master_processed["post_type"].value_counts(dropna=False))


# ============================================================
# 2) DUPLICATE + NEAR-DUPLICATE AUDIT
# Audit only. Does not remove rows.
# Uses clean_text, NOT domain_context_text.
# ============================================================

df = master_processed.copy()

def normalise_text_for_dup(x):
    x = "" if pd.isna(x) else str(x).lower()
    x = re.sub(r"https?://\S+|www\.\S+", " ", x)
    x = re.sub(r"@\w+", " ", x)
    x = re.sub(r"#", " ", x)
    x = re.sub(r"[^a-z0-9\s]", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x

def word_count(x):
    return len(re.findall(r"\b\w+\b", "" if pd.isna(x) else str(x)))

df["_norm_text_dup"] = df["clean_text"].fillna("").astype(str).apply(normalise_text_for_dup)
df["_norm_text_len"] = df["_norm_text_dup"].str.len()
df["_norm_word_count"] = df["_norm_text_dup"].apply(word_count)

# ------------------------------------------------------------
# A) Platform ID duplicate audit
# ------------------------------------------------------------

id_dup_rows = df.duplicated(subset=["platform", "id"], keep=False).sum()

id_dup_groups = (
    df.groupby(["platform", "id"], dropna=False)
    .size()
    .reset_index(name="rows")
)

id_dup_groups = id_dup_groups[id_dup_groups["rows"] > 1].copy()

# ------------------------------------------------------------
# B) Exact normalised text duplicate audit
# ------------------------------------------------------------

text_base = df[df["_norm_text_dup"].ne("")].copy()

exact_groups = (
    text_base
    .groupby(["post_type", "_norm_text_dup"], dropna=False)
    .agg(
        rows=("id", "count"),
        unique_platforms=("platform", pd.Series.nunique),
        unique_threads=("thread_id", pd.Series.nunique),
        example_text=("clean_text", "first"),
    )
    .reset_index()
)

exact_duplicate_groups = exact_groups[exact_groups["rows"] > 1].copy()

exact_duplicate_rows = (
    int(exact_duplicate_groups["rows"].sum())
    if len(exact_duplicate_groups)
    else 0
)

exact_duplicate_groups = exact_duplicate_groups.sort_values(
    ["rows", "unique_threads"],
    ascending=False
)

dup_norms = set(exact_duplicate_groups["_norm_text_dup"])
df["exact_normalized_text_duplicate"] = df["_norm_text_dup"].isin(dup_norms)

# ------------------------------------------------------------
# C) Near-duplicate audit using TF-IDF char n-grams
# ------------------------------------------------------------
# Avoid tiny comments like "yes exactly", "same", etc.
# Near-duplicates are audited by post_type to reduce false matches.
# ------------------------------------------------------------

NEAR_DUP_THRESHOLD = 0.90
near_duplicate_candidates_all = []

eligible = df[
    (df["_norm_text_len"] >= 40)
    &
    (df["_norm_word_count"] >= 6)
].copy()

print("\nRows eligible for near-duplicate audit:", len(eligible))

for group_name, group_df in eligible.groupby("post_type", dropna=False):
    group_df = group_df.copy()

    if len(group_df) < 2:
        continue

    print(f"Running near-duplicate audit for {group_name}: {len(group_df)} rows")

    group_df = group_df.reset_index().rename(columns={"index": "original_index"})

    vectorizer = TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(3, 5),
        min_df=2,
        max_features=50000,
        lowercase=False,
    )

    X = vectorizer.fit_transform(group_df["_norm_text_dup"])

    nn = NearestNeighbors(
        n_neighbors=2,
        metric="cosine",
        algorithm="brute",
        n_jobs=-1,
    )

    nn.fit(X)
    distances, indices = nn.kneighbors(X)

    nearest_idx = indices[:, 1]
    nearest_distance = distances[:, 1]
    nearest_similarity = 1 - nearest_distance

    group_df["nearest_original_index"] = group_df.loc[nearest_idx, "original_index"].values
    group_df["near_duplicate_similarity"] = nearest_similarity

    cand = group_df[group_df["near_duplicate_similarity"] >= NEAR_DUP_THRESHOLD].copy()

    if len(cand):
        lookup = df[
            [
                "platform",
                "post_type",
                "id",
                "date",
                "clean_text",
                "thread_id",
                "subreddit",
            ]
        ].copy()

        lookup_left = lookup.copy()
        lookup_left["original_index"] = lookup_left.index

        lookup_right = lookup.copy()
        lookup_right["nearest_original_index"] = lookup_right.index

        cand = cand[
            [
                "original_index",
                "nearest_original_index",
                "near_duplicate_similarity",
            ]
        ].copy()

        cand = cand.merge(lookup_left, on="original_index", how="left")

        cand = cand.merge(
            lookup_right,
            on="nearest_original_index",
            how="left",
            suffixes=("_row", "_nearest"),
        )

        cand["pair_key"] = cand.apply(
            lambda r: tuple(
                sorted(
                    [
                        int(r["original_index"]),
                        int(r["nearest_original_index"]),
                    ]
                )
            ),
            axis=1,
        )

        cand = cand.drop_duplicates(subset=["pair_key"]).copy()
        near_duplicate_candidates_all.append(cand)

if near_duplicate_candidates_all:
    near_duplicate_candidates = pd.concat(
        near_duplicate_candidates_all,
        ignore_index=True,
    ).sort_values("near_duplicate_similarity", ascending=False)
else:
    near_duplicate_candidates = pd.DataFrame()

# ------------------------------------------------------------
# D) Summaries
# ------------------------------------------------------------

duplicate_audit_summary = pd.DataFrame([
    {
        "metric": "final_rows_audited",
        "value": int(len(df)),
        "note": "Rows in master_processed after refined domain/listing/contact filter.",
    },
    {
        "metric": "platform_id_duplicate_groups",
        "value": int(len(id_dup_groups)),
        "note": "Duplicate groups by platform + id.",
    },
    {
        "metric": "platform_id_duplicate_rows",
        "value": int(id_dup_rows),
        "note": "Rows involved in platform + id duplicates.",
    },
    {
        "metric": "normalized_text_duplicate_groups",
        "value": int(len(exact_duplicate_groups)),
        "note": "Groups with identical normalized clean_text.",
    },
    {
        "metric": "normalized_text_duplicate_rows",
        "value": int(exact_duplicate_rows),
        "note": "Rows involved in normalized clean_text duplicate groups.",
    },
    {
        "metric": "near_duplicate_candidate_pairs",
        "value": int(len(near_duplicate_candidates)),
        "note": "TF-IDF char n-gram cosine similarity >= 0.90; audit only.",
    },
    {
        "metric": "near_duplicate_policy",
        "value": "audit_only",
        "note": "Near duplicates are reported, not automatically removed.",
    },
])

dup_by_platform = (
    df.groupby(["platform", "post_type"], dropna=False)
    .agg(
        rows=("id", "count"),
        exact_norm_text_duplicate_rows=("exact_normalized_text_duplicate", "sum"),
    )
    .reset_index()
)

if len(near_duplicate_candidates):
    near_duplicate_summary_by_type = (
        near_duplicate_candidates
        .groupby(["platform_row", "post_type_row"], dropna=False)
        .size()
        .reset_index(name="near_duplicate_candidate_pairs")
        .sort_values("near_duplicate_candidate_pairs", ascending=False)
    )
else:
    near_duplicate_summary_by_type = pd.DataFrame(
        columns=[
            "platform_row",
            "post_type_row",
            "near_duplicate_candidate_pairs",
        ]
    )

# ------------------------------------------------------------
# E) Save audit outputs
# ------------------------------------------------------------

duplicate_audit_summary.to_csv(
    AUDIT_DIR / "duplicate_audit_summary.csv",
    index=False,
)

id_dup_groups.to_csv(
    AUDIT_DIR / "id_duplicate_groups.csv",
    index=False,
)

exact_duplicate_groups.to_csv(
    AUDIT_DIR / "exact_normalized_text_duplicate_groups.csv",
    index=False,
)

dup_by_platform.to_csv(
    AUDIT_DIR / "duplicate_audit_by_platform_post_type.csv",
    index=False,
)

near_duplicate_candidates.to_csv(
    AUDIT_DIR / "near_duplicate_candidates.csv",
    index=False,
)

near_duplicate_summary_by_type.to_csv(
    AUDIT_DIR / "near_duplicate_summary_by_post_type.csv",
    index=False,
)

df.drop(
    columns=["_norm_text_dup", "_norm_text_len", "_norm_word_count"],
    errors="ignore",
).to_parquet(
    AUDIT_DIR / "master_processed_with_duplicate_flags.parquet",
    index=False,
)

df.drop(
    columns=["_norm_text_dup", "_norm_text_len", "_norm_word_count"],
    errors="ignore",
).to_csv(
    AUDIT_DIR / "master_processed_with_duplicate_flags.csv",
    index=False,
)

print("\nSaved duplicate audit outputs to:", AUDIT_DIR)

print("\nDuplicate audit summary:")
display(duplicate_audit_summary)

print("\nDuplicate audit by platform/post_type:")
display(dup_by_platform)

print("\nNear-duplicate summary by post_type:")
display(near_duplicate_summary_by_type)

print("\nTop exact normalized duplicate groups:")
display(exact_duplicate_groups.head(20))

print("\nTop near-duplicate candidates:")
display(near_duplicate_candidates.head(20))

In [ ]:
from pathlib import Path

BASE = Path("/content/drive/MyDrive/Dubai_Real_Estate_Data")
FILTER_DIR = BASE / "final_pipeline_outputs" / "filter_checkpoints"

QUALITY_PATH = FILTER_DIR / "master_processed_final_quality_clean.parquet"
REFINED_PATH = FILTER_DIR / "master_processed_after_refined_domain_listing_contact_clean.parquet"

print("Quality-clean exists:", QUALITY_PATH.exists())
print("Refined exists:", REFINED_PATH.exists())
print("Quality path:", QUALITY_PATH)

In [ ]:
# ============================================================
# FINAL LOW-INFORMATION CLEANUP
# + CORRECTED DUPLICATE / NEAR-DUPLICATE AUDIT
# ============================================================

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

from pathlib import Path
import pandas as pd
import numpy as np
import re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

BASE = Path("/content/drive/MyDrive/Dubai_Real_Estate_Data")

FILTER_DIR = BASE / "final_pipeline_outputs" / "filter_checkpoints"
PAPER_DIR = BASE / "final_pipeline_outputs" / "paper_ready_tables"
FINAL_DIR = BASE / "final_pipeline_outputs"

FILTER_DIR.mkdir(parents=True, exist_ok=True)
PAPER_DIR.mkdir(parents=True, exist_ok=True)
FINAL_DIR.mkdir(parents=True, exist_ok=True)

INPUT_PATH = FILTER_DIR / "master_processed_after_refined_domain_listing_contact_clean.parquet"
assert INPUT_PATH.exists(), f"Missing input file: {INPUT_PATH}"

master_processed = pd.read_parquet(INPUT_PATH)

print("Loaded refined master_processed:", master_processed.shape)

print("\nPlatform counts before low-info cleanup:")
print(master_processed["platform"].value_counts(dropna=False))

print("\nPost type counts before low-info cleanup:")
print(master_processed["post_type"].value_counts(dropna=False))


# ============================================================
# 1) LOW-INFORMATION COMMENT CLEANUP
# ============================================================

def norm_text(x):
    x = "" if pd.isna(x) else str(x).lower()
    x = re.sub(r"https?://\S+|www\.\S+", " ", x)
    x = re.sub(r"[^a-z0-9\s]", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x

def word_count(x):
    return len(re.findall(r"\b\w+\b", "" if pd.isna(x) else str(x)))

df = master_processed.copy()

df["_norm_clean_text"] = df["clean_text"].fillna("").astype(str).apply(norm_text)
df["_clean_word_count"] = df["clean_text"].fillna("").astype(str).apply(word_count)

# Exact low-information standalone Reddit comments.
# These are removed only when the whole normalized comment equals one of these.
LOW_INFO_EXACT = {
    # acknowledgements / reactions
    "yes", "yeah", "yep", "no", "nope", "ok", "okay",
    "thanks", "thank you", "thx", "sure", "done", "lol", "lmao",
    "this", "same", "agreed", "true", "very true", "exactly",
    "following", "bump",

    # low-information price/detail requests
    "price", "details", "details please", "share details",
    "send details", "interested",

    # contact-only / DM-only
    "dm", "dm me", "please dm", "pls dm", "dm please",
    "dm sent", "sent dm", "sent a dm", "check dm", "check your dm",
    "please check your dm", "kindly check your dm",
    "inbox", "pm me", "private chat", "pls connect", "please connect",
    "connect", "hi pls connect", "hi please connect",
}

MOD_REMOVAL_REGEX = re.compile(
    r"(?:your post has been removed because it does not|"
    r"your comment has been removed because it does not|"
    r"removed by moderator|"
    r"this post was removed|"
    r"this comment was removed)",
    flags=re.IGNORECASE,
)

CONTACT_NOISE_REGEX = re.compile(
    r"(?:check your dm|check dm|dm sent|sent dm|sent a dm|"
    r"please dm|pls dm|dm please|kindly check your dm|"
    r"private chat|pls connect|please connect|inbox|pm me)",
    flags=re.IGNORECASE,
)

def has_regex(text, regex):
    return bool(regex.search("" if pd.isna(text) else str(text)))

df["is_moderator_removal_boilerplate"] = df["clean_text"].apply(
    lambda x: has_regex(x, MOD_REMOVAL_REGEX)
)

df["is_short_contact_noise"] = (
    df["_clean_word_count"].le(8)
    &
    df["clean_text"].apply(lambda x: has_regex(x, CONTACT_NOISE_REGEX))
)

df["is_low_information_comment"] = (
    df["post_type"].eq("reddit_comment")
    &
    (
        df["_norm_clean_text"].isin(LOW_INFO_EXACT)
        |
        df["is_short_contact_noise"]
        |
        df["is_moderator_removal_boilerplate"]
    )
)

low_info_count = int(df["is_low_information_comment"].sum())

print("\nLow-information Reddit comments flagged for removal:", low_info_count)

if low_info_count > 0:
    print("\nExamples of low-information comments removed:")
    display(
        df[df["is_low_information_comment"]][
            [
                "platform",
                "post_type",
                "date",
                "username",
                "clean_text",
                "context_text",
            ]
        ].sample(min(30, low_info_count), random_state=42)
    )

master_processed_final = df[~df["is_low_information_comment"]].copy()

# Save removed rows for audit
removed_low_information_comments = df[df["is_low_information_comment"]].copy()

removed_low_information_comments.to_csv(
    PAPER_DIR / "removed_low_information_reddit_comments.csv",
    index=False,
)

removed_low_information_comments.to_parquet(
    PAPER_DIR / "removed_low_information_reddit_comments.parquet",
    index=False,
)

# Drop helper columns from final analysis dataset
master_processed_final = master_processed_final.drop(
    columns=[
        "_norm_clean_text",
        "_clean_word_count",
        "is_moderator_removal_boilerplate",
        "is_short_contact_noise",
    ],
    errors="ignore",
)

QUALITY_PATH = FILTER_DIR / "master_processed_final_quality_clean.parquet"
QUALITY_CSV = FILTER_DIR / "master_processed_final_quality_clean.csv"

master_processed_final.to_parquet(QUALITY_PATH, index=False)
master_processed_final.to_csv(QUALITY_CSV, index=False)

print("\nSaved quality-clean file:")
print(QUALITY_PATH)

print("\nmaster_processed_final:", master_processed_final.shape)

print("\nPlatform counts after low-info cleanup:")
print(master_processed_final["platform"].value_counts(dropna=False))

print("\nPost type counts after low-info cleanup:")
print(master_processed_final["post_type"].value_counts(dropna=False))


# ============================================================
# 2) CORRECTED DUPLICATE + NEAR-DUPLICATE AUDIT
# Uses clean_text only.
# Excludes self-pairs.
# Audit only, no automatic near-duplicate removal.
# ============================================================

audit_df = master_processed_final.copy()

def normalise_text_for_dup(x):
    x = "" if pd.isna(x) else str(x).lower()
    x = re.sub(r"https?://\S+|www\.\S+", " ", x)
    x = re.sub(r"@\w+", " ", x)
    x = re.sub(r"#", " ", x)
    x = re.sub(r"[^a-z0-9\s]", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x

audit_df["_norm_text_dup"] = audit_df["clean_text"].fillna("").astype(str).apply(normalise_text_for_dup)
audit_df["_norm_text_len"] = audit_df["_norm_text_dup"].str.len()
audit_df["_norm_word_count"] = audit_df["_norm_text_dup"].apply(word_count)

# ------------------------------------------------------------
# A) Platform + ID duplicate audit
# ------------------------------------------------------------

id_dup_rows = int(audit_df.duplicated(subset=["platform", "id"], keep=False).sum())

id_dup_groups = (
    audit_df.groupby(["platform", "id"], dropna=False)
    .size()
    .reset_index(name="rows")
)

id_dup_groups = id_dup_groups[id_dup_groups["rows"] > 1].copy()

# ------------------------------------------------------------
# B) Exact normalized clean_text duplicate audit
# ------------------------------------------------------------

text_base = audit_df[audit_df["_norm_text_dup"].ne("")].copy()

exact_groups = (
    text_base
    .groupby(["post_type", "_norm_text_dup"], dropna=False)
    .agg(
        rows=("id", "count"),
        unique_platforms=("platform", pd.Series.nunique),
        unique_authors=("username", pd.Series.nunique),
        unique_threads=("thread_id", pd.Series.nunique),
        example_text=("clean_text", "first"),
    )
    .reset_index()
)

exact_duplicate_groups = exact_groups[exact_groups["rows"] > 1].copy()

exact_duplicate_rows = (
    int(exact_duplicate_groups["rows"].sum())
    if len(exact_duplicate_groups)
    else 0
)

exact_duplicate_groups = exact_duplicate_groups.sort_values(
    ["rows", "unique_authors", "unique_threads"],
    ascending=False,
)

dup_norms = set(exact_duplicate_groups["_norm_text_dup"])
audit_df["exact_normalized_text_duplicate"] = audit_df["_norm_text_dup"].isin(dup_norms)

# ------------------------------------------------------------
# C) Near-duplicate audit using TF-IDF char n-grams
# ------------------------------------------------------------

NEAR_DUP_THRESHOLD = 0.90
near_duplicate_candidates_all = []

eligible = audit_df[
    (audit_df["_norm_text_len"] >= 40)
    &
    (audit_df["_norm_word_count"] >= 6)
].copy()

print("\nRows eligible for corrected near-duplicate audit:", len(eligible))

for group_name, group_df in eligible.groupby("post_type", dropna=False):
    group_df = group_df.copy()

    if len(group_df) < 2:
        continue

    print(f"Running corrected near-duplicate audit for {group_name}: {len(group_df)} rows")

    group_df = group_df.reset_index().rename(columns={"index": "original_index"})

    vectorizer = TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(3, 5),
        min_df=2,
        max_features=50000,
        lowercase=False,
    )

    X = vectorizer.fit_transform(group_df["_norm_text_dup"])

    n_neighbors = min(8, len(group_df))

    nn = NearestNeighbors(
        n_neighbors=n_neighbors,
        metric="cosine",
        algorithm="brute",
        n_jobs=-1,
    )

    nn.fit(X)
    distances, indices = nn.kneighbors(X)

    rows = []

    for i in range(len(group_df)):
        source_original_index = int(group_df.loc[i, "original_index"])

        for k in range(1, n_neighbors):
            candidate_position = int(indices[i, k])
            candidate_original_index = int(group_df.loc[candidate_position, "original_index"])

            # Skip self-pairs explicitly
            if candidate_original_index == source_original_index:
                continue

            sim = 1 - float(distances[i, k])

            if sim >= NEAR_DUP_THRESHOLD:
                rows.append({
                    "original_index": source_original_index,
                    "nearest_original_index": candidate_original_index,
                    "near_duplicate_similarity": sim,
                })
                break

    if rows:
        cand = pd.DataFrame(rows)

        lookup_cols = [
            "platform",
            "post_type",
            "id",
            "date",
            "username",
            "clean_text",
            "thread_id",
            "subreddit",
        ]

        for c in lookup_cols:
            if c not in audit_df.columns:
                audit_df[c] = pd.NA

        lookup = audit_df[lookup_cols].copy()

        lookup_left = lookup.copy()
        lookup_left["original_index"] = lookup_left.index

        lookup_right = lookup.copy()
        lookup_right["nearest_original_index"] = lookup_right.index

        cand = cand.merge(lookup_left, on="original_index", how="left")

        cand = cand.merge(
            lookup_right,
            on="nearest_original_index",
            how="left",
            suffixes=("_row", "_nearest"),
        )

        cand["pair_key"] = cand.apply(
            lambda r: tuple(
                sorted([
                    int(r["original_index"]),
                    int(r["nearest_original_index"]),
                ])
            ),
            axis=1,
        )

        cand = cand.drop_duplicates(subset=["pair_key"]).copy()

        cand["same_author_near_duplicate"] = (
            cand["username_row"].fillna("").astype(str).str.lower().str.strip()
            ==
            cand["username_nearest"].fillna("").astype(str).str.lower().str.strip()
        )

        near_duplicate_candidates_all.append(cand)

if near_duplicate_candidates_all:
    near_duplicate_candidates = pd.concat(
        near_duplicate_candidates_all,
        ignore_index=True,
    ).sort_values("near_duplicate_similarity", ascending=False)
else:
    near_duplicate_candidates = pd.DataFrame()

# Sanity check: no self-pairs
if len(near_duplicate_candidates):
    self_pair_count = int(
        (
            near_duplicate_candidates["original_index"]
            ==
            near_duplicate_candidates["nearest_original_index"]
        ).sum()
    )
else:
    self_pair_count = 0

assert self_pair_count == 0, f"Self-pairs still present: {self_pair_count}"

# ------------------------------------------------------------
# D) Summaries
# ------------------------------------------------------------

same_author_near_dup_pairs = (
    int(near_duplicate_candidates["same_author_near_duplicate"].sum())
    if len(near_duplicate_candidates)
    else 0
)

duplicate_audit_summary = pd.DataFrame([
    {
        "metric": "rows_before_low_information_cleanup",
        "value": int(len(df)),
        "note": "Rows after refined domain/listing/contact filtering.",
    },
    {
        "metric": "low_information_comments_removed",
        "value": int(low_info_count),
        "note": "Short contact-only, acknowledgement-only, or moderator-removal Reddit comments removed.",
    },
    {
        "metric": "final_rows_audited",
        "value": int(len(audit_df)),
        "note": "Rows after low-information cleanup.",
    },
    {
        "metric": "platform_id_duplicate_groups",
        "value": int(len(id_dup_groups)),
        "note": "Duplicate groups by platform + id.",
    },
    {
        "metric": "platform_id_duplicate_rows",
        "value": int(id_dup_rows),
        "note": "Rows involved in platform + id duplicates.",
    },
    {
        "metric": "normalized_text_duplicate_groups",
        "value": int(len(exact_duplicate_groups)),
        "note": "Groups with identical normalized clean_text.",
    },
    {
        "metric": "normalized_text_duplicate_rows",
        "value": int(exact_duplicate_rows),
        "note": "Rows involved in normalized clean_text duplicate groups.",
    },
    {
        "metric": "near_duplicate_candidate_pairs",
        "value": int(len(near_duplicate_candidates)),
        "note": "TF-IDF char n-gram cosine similarity >= 0.90; self-pairs excluded.",
    },
    {
        "metric": "same_author_near_duplicate_pairs",
        "value": int(same_author_near_dup_pairs),
        "note": "Near-duplicate candidate pairs where both rows have the same username.",
    },
    {
        "metric": "near_duplicate_policy",
        "value": "audit_only",
        "note": "Near duplicates are reported/flagged, not automatically removed in this step.",
    },
])

dup_by_platform = (
    audit_df.groupby(["platform", "post_type"], dropna=False)
    .agg(
        rows=("id", "count"),
        exact_norm_text_duplicate_rows=("exact_normalized_text_duplicate", "sum"),
    )
    .reset_index()
)

if len(near_duplicate_candidates):
    near_duplicate_summary_by_type = (
        near_duplicate_candidates
        .groupby(["platform_row", "post_type_row"], dropna=False)
        .agg(
            near_duplicate_candidate_pairs=("pair_key", "count"),
            same_author_near_duplicate_pairs=("same_author_near_duplicate", "sum"),
        )
        .reset_index()
        .sort_values("near_duplicate_candidate_pairs", ascending=False)
    )
else:
    near_duplicate_summary_by_type = pd.DataFrame(
        columns=[
            "platform_row",
            "post_type_row",
            "near_duplicate_candidate_pairs",
            "same_author_near_duplicate_pairs",
        ]
    )

# ------------------------------------------------------------
# E) Save audit outputs
# ------------------------------------------------------------

duplicate_audit_summary.to_csv(
    PAPER_DIR / "duplicate_audit_summary.csv",
    index=False,
)

id_dup_groups.to_csv(
    PAPER_DIR / "id_duplicate_groups.csv",
    index=False,
)

exact_duplicate_groups.to_csv(
    PAPER_DIR / "exact_normalized_text_duplicate_groups.csv",
    index=False,
)

dup_by_platform.to_csv(
    PAPER_DIR / "duplicate_audit_by_platform_post_type.csv",
    index=False,
)

near_duplicate_candidates.to_csv(
    PAPER_DIR / "near_duplicate_candidates.csv",
    index=False,
)

near_duplicate_summary_by_type.to_csv(
    PAPER_DIR / "near_duplicate_summary_by_post_type.csv",
    index=False,
)

audit_df.drop(
    columns=["_norm_text_dup", "_norm_text_len", "_norm_word_count"],
    errors="ignore",
).to_parquet(
    PAPER_DIR / "master_processed_with_duplicate_flags.parquet",
    index=False,
)

audit_df.drop(
    columns=["_norm_text_dup", "_norm_text_len", "_norm_word_count"],
    errors="ignore",
).to_csv(
    PAPER_DIR / "master_processed_with_duplicate_flags.csv",
    index=False,
)

# Save current quality-clean final master too
master_processed_final.to_parquet(
    FINAL_DIR / "master_processed_quality_clean.parquet",
    index=False,
)

master_processed_final.to_csv(
    FINAL_DIR / "master_processed_quality_clean.csv",
    index=False,
)

print("\nSaved corrected duplicate audit outputs to:")
print(PAPER_DIR)

print("\nSaved quality-clean final master to:")
print(FINAL_DIR / "master_processed_quality_clean.parquet")

print("\nDuplicate audit summary:")
display(duplicate_audit_summary)

print("\nDuplicate audit by platform/post_type:")
display(dup_by_platform)

print("\nNear-duplicate summary by post_type:")
display(near_duplicate_summary_by_type)

print("\nTop exact normalized duplicate groups:")
display(exact_duplicate_groups.head(20))

print("\nTop near-duplicate candidates:")
display(near_duplicate_candidates.head(20))

In [ ]:
# ============================================================
# AUTHOR-AWARE DUPLICATE CLEANING
# Input: master_processed_final_quality_clean.parquet
# Removes:
#   1) remaining contact/promotional low-info variants
#   2) same-author exact normalized-text repeats
#   3) high-confidence same-author near-duplicate repeats using saved near_duplicate_candidates.csv
# Keeps:
#   - different-author duplicates flagged only
#   - archive/audit outputs
# Output:
#   master_processed_analysis_clean.parquet/csv
# ============================================================

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

from pathlib import Path
import pandas as pd
import numpy as np
import re

BASE = Path("/content/drive/MyDrive/Dubai_Real_Estate_Data")

FILTER_DIR = BASE / "final_pipeline_outputs" / "filter_checkpoints"
PAPER_DIR = BASE / "final_pipeline_outputs" / "paper_ready_tables"
FINAL_DIR = BASE / "final_pipeline_outputs"

PAPER_DIR.mkdir(parents=True, exist_ok=True)
FINAL_DIR.mkdir(parents=True, exist_ok=True)

QUALITY_PATH = FILTER_DIR / "master_processed_final_quality_clean.parquet"
NEAR_CANDIDATES_PATH = PAPER_DIR / "near_duplicate_candidates.csv"

assert QUALITY_PATH.exists(), f"Missing quality-clean file: {QUALITY_PATH}"

df = pd.read_parquet(QUALITY_PATH).reset_index(drop=True)

print("Loaded quality-clean dataset:", df.shape)
print(df["platform"].value_counts(dropna=False))
print(df["post_type"].value_counts(dropna=False))

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def norm_text_for_dup(x):
    x = "" if pd.isna(x) else str(x).lower()
    x = re.sub(r"https?://\S+|www\.\S+", " ", x)
    x = re.sub(r"@\w+", " ", x)
    x = re.sub(r"#", " ", x)
    x = re.sub(r"[^a-z0-9\s]", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x

def norm_author(x):
    x = "" if pd.isna(x) else str(x).lower().strip()
    x = re.sub(r"\s+", " ", x)
    return x

def word_count(x):
    return len(re.findall(r"\b\w+\b", "" if pd.isna(x) else str(x)))

df["_row_index_quality"] = df.index
df["_norm_text_dup"] = df["clean_text"].fillna("").astype(str).apply(norm_text_for_dup)
df["_norm_author"] = df["username"].fillna("").astype(str).apply(norm_author)
df["_word_count"] = df["clean_text"].fillna("").astype(str).apply(word_count)
df["date"] = pd.to_datetime(df["date"], errors="coerce", utc=True)

# ============================================================
# 1) EXTRA LOW-INFORMATION / CONTACT VARIANT CLEANUP
# Catches remaining variants visible in duplicate audit:
# "Dm'd you", "Dmed you", "DM pls", "Just sent you in DM...",
# "profile bio", "hey man let's talk", etc.
# ============================================================

EXTRA_LOW_INFO_EXACT = {
    "sent",
    "dm d you",
    "dmed you",
    "dm pls",
    "dm you",
    "please share",
    "just sent you a message",
    "hey man let s talk",
}

EXTRA_CONTACT_REGEX = re.compile(
    r"(?:"
    r"\bdm\s*d you\b|"
    r"\bdmed you\b|"
    r"\bdm pls\b|"
    r"\bdm you\b|"
    r"\bjust sent you\b|"
    r"\bsent you in dm\b|"
    r"\bsent you a dm\b|"
    r"\bsent you a message\b|"
    r"\bprofile bio\b|"
    r"\bavailable in my bio\b|"
    r"\bmessage me\b|"
    r"\bping me\b|"
    r"\blet s talk\b|"
    r"\blet us talk\b"
    r")",
    flags=re.IGNORECASE,
)

df["is_extra_low_info_contact_variant"] = (
    df["post_type"].eq("reddit_comment")
    &
    (
        df["_norm_text_dup"].isin(EXTRA_LOW_INFO_EXACT)
        |
        (
            df["_word_count"].le(25)
            &
            df["clean_text"].fillna("").astype(str).str.contains(EXTRA_CONTACT_REGEX, na=False)
        )
    )
)

# ============================================================
# 2) SAME-AUTHOR EXACT NORMALIZED-TEXT DUPLICATES
# Keep first chronological occurrence.
# ============================================================

df = df.sort_values(
    ["platform", "_norm_author", "_norm_text_dup", "date"],
    na_position="last"
).copy()

df["same_author_exact_text_group_size"] = (
    df.groupby(
        ["platform", "_norm_author", "_norm_text_dup"],
        dropna=False
    )["id"].transform("count")
)

df["is_same_author_exact_text_repeat"] = (
    df["_norm_author"].ne("")
    &
    df["_norm_text_dup"].ne("")
    &
    (df["same_author_exact_text_group_size"] > 1)
    &
    df.duplicated(
        subset=["platform", "_norm_author", "_norm_text_dup"],
        keep="first"
    )
)

# ============================================================
# 3) DIFFERENT-AUTHOR EXACT TEXT DUPLICATES
# Flag only, do not remove.
# ============================================================

df["same_text_any_author_group_size"] = (
    df.groupby(["platform", "_norm_text_dup"], dropna=False)["id"]
    .transform("count")
)

df["same_text_unique_author_count"] = (
    df.groupby(["platform", "_norm_text_dup"], dropna=False)["_norm_author"]
    .transform("nunique")
)

df["is_exact_text_duplicate_different_authors_flag"] = (
    df["_norm_text_dup"].ne("")
    &
    (df["same_text_any_author_group_size"] > 1)
    &
    (df["same_text_unique_author_count"] > 1)
)

# ============================================================
# 4) HIGH-CONFIDENCE SAME-AUTHOR NEAR-DUPLICATE REMOVAL
# Uses saved near_duplicate_candidates.csv.
# Threshold: >= 0.98
# Within each connected duplicate cluster, keep earliest row.
# ============================================================

df["row_key"] = df["platform"].astype(str) + "||" + df["id"].astype(str)
df["is_same_author_highconf_near_duplicate_repeat"] = False

near_removed_keys = set()

if NEAR_CANDIDATES_PATH.exists():
    near = pd.read_csv(NEAR_CANDIDATES_PATH)

    if len(near):
        # Normalize boolean column
        if "same_author_near_duplicate" in near.columns:
            near["same_author_near_duplicate"] = (
                near["same_author_near_duplicate"]
                .astype(str)
                .str.lower()
                .isin(["true", "1", "yes"])
            )
        else:
            near["same_author_near_duplicate"] = False

        near_high = near[
            (near["same_author_near_duplicate"])
            &
            (pd.to_numeric(near["near_duplicate_similarity"], errors="coerce") >= 0.98)
        ].copy()

        print("\nHigh-confidence same-author near-duplicate candidate pairs:", len(near_high))

        # Union-find over row keys
        parent = {}

        def find(x):
            parent.setdefault(x, x)
            if parent[x] != x:
                parent[x] = find(parent[x])
            return parent[x]

        def union(a, b):
            ra, rb = find(a), find(b)
            if ra != rb:
                parent[rb] = ra

        for _, r in near_high.iterrows():
            key_a = str(r["platform_row"]) + "||" + str(r["id_row"])
            key_b = str(r["platform_nearest"]) + "||" + str(r["id_nearest"])

            if key_a in set(df["row_key"]) and key_b in set(df["row_key"]):
                union(key_a, key_b)

        # Build components
        components = {}
        for key in parent.keys():
            root = find(key)
            components.setdefault(root, set()).add(key)

        # For each component, keep earliest date, remove rest
        key_to_date = df.set_index("row_key")["date"].to_dict()

        for _, keys in components.items():
            keys = [k for k in keys if k in key_to_date]

            if len(keys) <= 1:
                continue

            keys_sorted = sorted(
                keys,
                key=lambda k: (
                    pd.Timestamp.max.tz_localize("UTC")
                    if pd.isna(key_to_date[k])
                    else key_to_date[k]
                )
            )

            keep_key = keys_sorted[0]
            remove_keys = keys_sorted[1:]

            near_removed_keys.update(remove_keys)

        df["is_same_author_highconf_near_duplicate_repeat"] = df["row_key"].isin(near_removed_keys)

else:
    print("\nNo near_duplicate_candidates.csv found. Skipping near-duplicate removal.")

# ============================================================
# 5) FINAL AUTHOR-AWARE REMOVAL POLICY
# ============================================================

df["removed_by_author_aware_policy"] = (
    df["is_extra_low_info_contact_variant"]
    |
    df["is_same_author_exact_text_repeat"]
    |
    df["is_same_author_highconf_near_duplicate_repeat"]
)

master_processed_analysis_clean = df[
    ~df["removed_by_author_aware_policy"]
].copy()

removed_author_aware = df[
    df["removed_by_author_aware_policy"]
].copy()

# ------------------------------------------------------------
# Summary tables
# ------------------------------------------------------------

author_duplicate_summary = pd.DataFrame([
    {
        "metric": "input_quality_clean_rows",
        "value": int(len(df)),
        "note": "Rows loaded from master_processed_final_quality_clean.parquet.",
    },
    {
        "metric": "extra_low_info_contact_variants_removed",
        "value": int(df["is_extra_low_info_contact_variant"].sum()),
        "note": "Remaining contact/message-only variants removed.",
    },
    {
        "metric": "same_author_exact_text_repeats_removed",
        "value": int(df["is_same_author_exact_text_repeat"].sum()),
        "note": "Same platform + same author + identical normalized text; first chronological occurrence retained.",
    },
    {
        "metric": "same_author_highconf_near_duplicate_repeats_removed",
        "value": int(df["is_same_author_highconf_near_duplicate_repeat"].sum()),
        "note": "Same-author near-duplicate clusters at similarity >= 0.98; earliest occurrence retained.",
    },
    {
        "metric": "different_author_exact_text_duplicates_flagged_only",
        "value": int(df["is_exact_text_duplicate_different_authors_flag"].sum()),
        "note": "Same text across different authors; flagged but not removed.",
    },
    {
        "metric": "total_rows_removed_author_aware_policy",
        "value": int(df["removed_by_author_aware_policy"].sum()),
        "note": "Rows removed by final author-aware policy.",
    },
    {
        "metric": "final_analysis_clean_rows",
        "value": int(len(master_processed_analysis_clean)),
        "note": "Rows in final analysis-clean dataset.",
    },
])

same_author_exact_groups = (
    df[df["same_author_exact_text_group_size"] > 1]
    .groupby(
        ["platform", "post_type", "_norm_author", "_norm_text_dup"],
        dropna=False
    )
    .agg(
        rows=("id", "count"),
        first_date=("date", "min"),
        last_date=("date", "max"),
        example_text=("clean_text", "first"),
    )
    .reset_index()
    .sort_values("rows", ascending=False)
)

# ------------------------------------------------------------
# Save outputs
# ------------------------------------------------------------

author_duplicate_summary.to_csv(
    PAPER_DIR / "author_aware_duplicate_cleaning_summary.csv",
    index=False,
)

same_author_exact_groups.to_csv(
    PAPER_DIR / "same_author_exact_text_duplicate_groups.csv",
    index=False,
)

removed_author_aware.to_csv(
    PAPER_DIR / "removed_by_author_aware_duplicate_policy.csv",
    index=False,
)

removed_author_aware.to_parquet(
    PAPER_DIR / "removed_by_author_aware_duplicate_policy.parquet",
    index=False,
)

df.drop(
    columns=["_norm_text_dup", "_norm_author", "_word_count", "row_key"],
    errors="ignore"
).to_parquet(
    PAPER_DIR / "master_processed_with_author_duplicate_flags.parquet",
    index=False,
)

df.drop(
    columns=["_norm_text_dup", "_norm_author", "_word_count", "row_key"],
    errors="ignore"
).to_csv(
    PAPER_DIR / "master_processed_with_author_duplicate_flags.csv",
    index=False,
)

master_processed_analysis_clean = master_processed_analysis_clean.drop(
    columns=["_norm_text_dup", "_norm_author", "_word_count", "row_key"],
    errors="ignore"
)

master_processed_analysis_clean.to_parquet(
    FINAL_DIR / "master_processed_analysis_clean.parquet",
    index=False,
)

master_processed_analysis_clean.to_csv(
    FINAL_DIR / "master_processed_analysis_clean.csv",
    index=False,
)

print("\nAUTHOR-AWARE DUPLICATE CLEANING SUMMARY:")
display(author_duplicate_summary)

print("\nFinal analysis-clean dataset:")
print(master_processed_analysis_clean.shape)

print("\nPlatform counts:")
print(master_processed_analysis_clean["platform"].value_counts(dropna=False))

print("\nPost type counts:")
print(master_processed_analysis_clean["post_type"].value_counts(dropna=False))

print("\nTop same-author exact duplicate groups:")
display(same_author_exact_groups.head(25))

print("\nExamples removed by author-aware policy:")
display(
    removed_author_aware[
        [
            "platform",
            "post_type",
            "date",
            "username",
            "clean_text",
            "is_extra_low_info_contact_variant",
            "is_same_author_exact_text_repeat",
            "is_same_author_highconf_near_duplicate_repeat",
        ]
    ].sample(min(30, len(removed_author_aware)), random_state=42)
)

print("\nSaved final analysis-clean dataset to:")
print(FINAL_DIR / "master_processed_analysis_clean.parquet")
print(FINAL_DIR / "master_processed_analysis_clean.csv")

In [ ]:
# ============================================================
# CREATE VALIDATION SAMPLE TEMPLATES
# Input: final analysis-clean dataset
# Output:
#   1. main_manual_validation_sample_template.csv
#   2. annotator1 / annotator2 copies
#   3. twitter_semantic_only_validation_template.csv
#   4. removed_rows_audit_sample.csv
# ============================================================

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

from pathlib import Path
import pandas as pd
import numpy as np

BASE = Path("/content/drive/MyDrive/Dubai_Real_Estate_Data")

FINAL_DIR = BASE / "final_pipeline_outputs"
PAPER_DIR = FINAL_DIR / "paper_ready_tables"
VALIDATION_DIR = FINAL_DIR / "validation_samples"

VALIDATION_DIR.mkdir(parents=True, exist_ok=True)

FINAL_PATH = FINAL_DIR / "master_processed_analysis_clean.parquet"
assert FINAL_PATH.exists(), f"Missing final dataset: {FINAL_PATH}"

df = pd.read_parquet(FINAL_PATH)

print("Loaded final analysis-clean dataset:", df.shape)
print(df["platform"].value_counts(dropna=False))
print(df["post_type"].value_counts(dropna=False))

df["date"] = pd.to_datetime(df["date"], errors="coerce", utc=True)
df["month"] = df["date"].dt.to_period("M").astype(str)

# ------------------------------------------------------------
# Helper: stratified sample
# ------------------------------------------------------------

def stratified_sample(data, group_cols, n_total, random_state=42):
    data = data.copy()

    group_sizes = data.groupby(group_cols, dropna=False).size().reset_index(name="size")

    # proportional allocation with minimum 1 per group
    group_sizes["n"] = np.ceil(group_sizes["size"] / len(data) * n_total).astype(int)
    group_sizes["n"] = group_sizes["n"].clip(lower=1)

    # if allocation exceeds n_total, trim from largest allocations
    while group_sizes["n"].sum() > n_total:
        idx = group_sizes[group_sizes["n"] > 1]["n"].idxmax()
        group_sizes.loc[idx, "n"] -= 1

    sampled_parts = []

    for _, row in group_sizes.iterrows():
        mask = pd.Series(True, index=data.index)
        for col in group_cols:
            val = row[col]
            if pd.isna(val):
                mask &= data[col].isna()
            else:
                mask &= data[col].eq(val)

        group_df = data[mask]
        n = min(int(row["n"]), len(group_df))

        sampled_parts.append(
            group_df.sample(n=n, random_state=random_state)
        )

    return pd.concat(sampled_parts, ignore_index=True).sample(frac=1, random_state=random_state).reset_index(drop=True)

# ------------------------------------------------------------
# 1) Main validation sample
# ------------------------------------------------------------

MAIN_SAMPLE_N = 400

main_sample = stratified_sample(
    df,
    group_cols=["platform", "post_type", "month"],
    n_total=MAIN_SAMPLE_N,
    random_state=42,
)

main_sample = main_sample.reset_index(drop=True)
main_sample["validation_id"] = ["VAL_MAIN_%04d" % (i + 1) for i in range(len(main_sample))]

# Columns for annotators
main_sample["annotator_label_domain_relevant"] = ""
main_sample["annotator_label_genuine_discourse"] = ""
main_sample["annotator_label_listing_or_promotion"] = ""
main_sample["annotator_label_low_information"] = ""
main_sample["annotator_notes"] = ""

main_cols = [
    "validation_id",
    "platform",
    "post_type",
    "date",
    "month",
    "source_label",
    "subreddit",
    "username",
    "clean_text",
    "context_text",
    "domain_context_text",
    "url",
    "domain_keywords_matched",
    "strong_listing_cues_matched",
    "weak_listing_cues_matched",
    "analysis_protection_cues_matched",
    "annotator_label_domain_relevant",
    "annotator_label_genuine_discourse",
    "annotator_label_listing_or_promotion",
    "annotator_label_low_information",
    "annotator_notes",
]

for c in main_cols:
    if c not in main_sample.columns:
        main_sample[c] = ""

main_sample = main_sample[main_cols]

main_sample.to_csv(
    VALIDATION_DIR / "main_manual_validation_sample_template.csv",
    index=False,
)

main_sample.to_csv(
    VALIDATION_DIR / "main_manual_validation_sample_annotator1.csv",
    index=False,
)

main_sample.to_csv(
    VALIDATION_DIR / "main_manual_validation_sample_annotator2.csv",
    index=False,
)

# ------------------------------------------------------------
# 2) Twitter semantic-only validation sample
# ------------------------------------------------------------

if "source_label" in df.columns:
    semantic_only = df[
        df["platform"].eq("twitter")
        &
        df["source_label"].astype(str).str.lower().eq("semantic")
    ].copy()
else:
    semantic_only = pd.DataFrame()

SEMANTIC_SAMPLE_N = 150

if len(semantic_only):
    semantic_sample = semantic_only.sample(
        n=min(SEMANTIC_SAMPLE_N, len(semantic_only)),
        random_state=42,
    ).reset_index(drop=True)

    semantic_sample["validation_id"] = [
        "VAL_SEM_%04d" % (i + 1) for i in range(len(semantic_sample))
    ]

    semantic_sample["annotator_label_semantic_relevant"] = ""
    semantic_sample["annotator_label_real_estate_discourse"] = ""
    semantic_sample["annotator_label_noise_or_offtopic"] = ""
    semantic_sample["annotator_notes"] = ""

    semantic_cols = [
        "validation_id",
        "platform",
        "post_type",
        "date",
        "source_label",
        "username",
        "clean_text",
        "url",
        "query_used",
        "minilm_positive_max_similarity",
        "minilm_negative_max_similarity",
        "minilm_semantic_margin",
        "minilm_best_positive_reference",
        "minilm_best_negative_reference",
        "annotator_label_semantic_relevant",
        "annotator_label_real_estate_discourse",
        "annotator_label_noise_or_offtopic",
        "annotator_notes",
    ]

    for c in semantic_cols:
        if c not in semantic_sample.columns:
            semantic_sample[c] = ""

    semantic_sample = semantic_sample[semantic_cols]

    semantic_sample.to_csv(
        VALIDATION_DIR / "twitter_semantic_only_validation_template.csv",
        index=False,
    )

    semantic_sample.to_csv(
        VALIDATION_DIR / "twitter_semantic_only_validation_annotator1.csv",
        index=False,
    )

    semantic_sample.to_csv(
        VALIDATION_DIR / "twitter_semantic_only_validation_annotator2.csv",
        index=False,
    )

    print("\nSemantic-only Twitter sample:", semantic_sample.shape)

else:
    print("\nNo semantic-only Twitter rows found in final dataset.")

# ------------------------------------------------------------
# 3) Removed-row audit sample
# ------------------------------------------------------------

removed_files = [
    PAPER_DIR / "removed_low_information_reddit_comments.csv",
    PAPER_DIR / "removed_by_author_aware_duplicate_policy.csv",
]

removed_parts = []

for p in removed_files:
    if p.exists():
        temp = pd.read_csv(p, low_memory=False)
        temp["removed_source_file"] = p.name
        removed_parts.append(temp)

if removed_parts:
    removed_all = pd.concat(removed_parts, ignore_index=True, sort=False)

    removed_sample = removed_all.sample(
        n=min(250, len(removed_all)),
        random_state=42,
    ).reset_index(drop=True)

    removed_sample["validation_id"] = [
        "VAL_REMOVED_%04d" % (i + 1) for i in range(len(removed_sample))
    ]

    removed_sample["annotator_agree_removed"] = ""
    removed_sample["annotator_reason"] = ""
    removed_sample["annotator_notes"] = ""

    removed_cols = [
        "validation_id",
        "removed_source_file",
        "platform",
        "post_type",
        "date",
        "username",
        "clean_text",
        "context_text",
        "is_low_information_comment",
        "is_extra_low_info_contact_variant",
        "is_same_author_exact_text_repeat",
        "is_same_author_highconf_near_duplicate_repeat",
        "annotator_agree_removed",
        "annotator_reason",
        "annotator_notes",
    ]

    for c in removed_cols:
        if c not in removed_sample.columns:
            removed_sample[c] = ""

    removed_sample = removed_sample[removed_cols]

    removed_sample.to_csv(
        VALIDATION_DIR / "removed_rows_audit_sample_template.csv",
        index=False,
    )

    print("\nRemoved-row audit sample:", removed_sample.shape)

else:
    print("\nNo removed-row audit files found.")

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

validation_summary = pd.DataFrame([
    {
        "file": "main_manual_validation_sample_template.csv",
        "rows": len(main_sample),
        "purpose": "Validate final kept dataset for domain relevance, discourse quality, listing/promo leakage, and low-information leakage.",
    },
    {
        "file": "twitter_semantic_only_validation_template.csv",
        "rows": int(len(semantic_sample)) if "semantic_sample" in locals() else 0,
        "purpose": "Validate MiniLM semantic-only Twitter retrieval quality.",
    },
    {
        "file": "removed_rows_audit_sample_template.csv",
        "rows": int(len(removed_sample)) if "removed_sample" in locals() else 0,
        "purpose": "Audit whether removed rows were correctly excluded.",
    },
])

validation_summary.to_csv(
    VALIDATION_DIR / "validation_sample_summary.csv",
    index=False,
)

print("\nSaved validation samples to:")
print(VALIDATION_DIR)

print("\nValidation summary:")
display(validation_summary)

print("\nFiles:")
for p in sorted(VALIDATION_DIR.glob("*.csv")):
    print("-", p.name)

In [ ]:
# ============================================================
# FINAL PAPER-READY TABLES
# Uses final analysis-clean dataset
# ============================================================

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

from pathlib import Path
import pandas as pd
import numpy as np

BASE = Path("/content/drive/MyDrive/Dubai_Real_Estate_Data")
FINAL_DIR = BASE / "final_pipeline_outputs"
PAPER_DIR = FINAL_DIR / "paper_ready_tables"
PAPER_DIR.mkdir(parents=True, exist_ok=True)

FINAL_PATH = FINAL_DIR / "master_processed_analysis_clean.parquet"
assert FINAL_PATH.exists(), f"Missing: {FINAL_PATH}"

df = pd.read_parquet(FINAL_PATH)

df["date"] = pd.to_datetime(df["date"], errors="coerce", utc=True)
df["month"] = df["date"].dt.to_period("M").astype(str)
df["yearweek"] = df["date"].dt.strftime("%G-W%V")

print("Loaded final dataset:", df.shape)

# ------------------------------------------------------------
# Table 1: Final dataset composition
# ------------------------------------------------------------

table_dataset_composition = (
    df.groupby(["platform", "post_type"], dropna=False)
    .size()
    .reset_index(name="rows")
    .sort_values(["platform", "post_type"])
)

table_dataset_composition.to_csv(
    PAPER_DIR / "table_dataset_composition.csv",
    index=False
)

# ------------------------------------------------------------
# Table 2: Monthly distribution
# ------------------------------------------------------------

table_monthly_distribution = (
    df.groupby(["month", "platform", "post_type"], dropna=False)
    .size()
    .reset_index(name="rows")
    .sort_values(["month", "platform", "post_type"])
)

table_monthly_distribution.to_csv(
    PAPER_DIR / "table_monthly_distribution.csv",
    index=False
)

# ------------------------------------------------------------
# Table 3: Twitter provenance
# ------------------------------------------------------------

if "source_label" in df.columns:
    table_twitter_provenance = (
        df[df["platform"].eq("twitter")]
        .groupby(["source_label"], dropna=False)
        .size()
        .reset_index(name="rows")
        .sort_values("rows", ascending=False)
    )
else:
    table_twitter_provenance = pd.DataFrame(columns=["source_label", "rows"])

table_twitter_provenance.to_csv(
    PAPER_DIR / "table_twitter_provenance.csv",
    index=False
)

# ------------------------------------------------------------
# Table 4: Reddit subreddit composition
# ------------------------------------------------------------

table_reddit_subreddit = (
    df[df["platform"].eq("reddit")]
    .groupby(["subreddit", "post_type"], dropna=False)
    .size()
    .reset_index(name="rows")
    .sort_values(["subreddit", "post_type"])
)

table_reddit_subreddit.to_csv(
    PAPER_DIR / "table_reddit_subreddit_composition.csv",
    index=False
)

# ------------------------------------------------------------
# Table 5: Cleaning / audit summaries
# ------------------------------------------------------------

summary_files = [
    PAPER_DIR / "duplicate_audit_summary.csv",
    PAPER_DIR / "author_aware_duplicate_cleaning_summary.csv",
    FINAL_DIR / "validation_samples" / "validation_sample_summary.csv",
]

summary_parts = []

for path in summary_files:
    if path.exists():
        temp = pd.read_csv(path)
        temp["source_file"] = path.name
        summary_parts.append(temp)

if summary_parts:
    table_pipeline_audit_summary = pd.concat(summary_parts, ignore_index=True, sort=False)
else:
    table_pipeline_audit_summary = pd.DataFrame()

table_pipeline_audit_summary.to_csv(
    PAPER_DIR / "table_pipeline_audit_summary.csv",
    index=False
)

# ------------------------------------------------------------
# Print outputs
# ------------------------------------------------------------

print("\nTable 1: Dataset composition")
display(table_dataset_composition)

print("\nTable 2: Monthly distribution")
display(table_monthly_distribution)

print("\nTable 3: Twitter provenance")
display(table_twitter_provenance)

print("\nTable 4: Reddit subreddit composition")
display(table_reddit_subreddit)

print("\nSaved paper-ready tables to:")
print(PAPER_DIR)

print("\nFiles:")
for p in sorted(PAPER_DIR.glob("table_*.csv")):
    print("-", p.name)